dataset without battery

In [ ]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario — NO BESS
# - runs in chunks
# - different seed per chunk
# - different output folder per chunk
# - PE computed from static edges each chunk (node_pe_k > 0, no CSV)
# ============================================================
import os
import math
import time
from pathlib import Path

# ---------------- user controls ----------------
INCLUDE_BESS = False

TOTAL_SCENARIOS = 2000
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20320230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

# Only used if INCLUDE_BESS is True
BESS_TOTAL_MVA_MEAN = 0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_CANDIDATE_NODES_150 = []  # fill if you set INCLUDE_BESS = True

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

CHUNK_ROOT = Path(r"H:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    os.getcwd(),
]

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

# ---------------- preflight: chunk parent template (optional) ----------------
# Probes first chunk folder name pattern — fails early if chunk_parent layout wrong
root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS}")

for chunk_idx in range(n_chunks):
    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
    )

    if INCLUDE_BESS:
        if not BESS_CANDIDATE_NODES_150:
            raise ValueError("INCLUDE_BESS=True requires a non-empty BESS_CANDIDATE_NODES_150 list.")
        gen_kw.update(
            bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
            bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
            bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
            bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
            bess_q_frac_max=float(BESS_Q_FRAC_MAX),
            bess_candidate_nodes_override=BESS_CANDIDATE_NODES_150,
        )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    print("Saved:")
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    print(" -", ns["NODE_CSV"])

print("\nAll chunks finished.")

datset with battery

In [1]:
# ============================================================
# CHUNKED RUN: scenarios x samples/scenario -- WITH BESS
# - runs in chunks
# - different seed per chunk
# - different output folder per chunk
# - auto-discovers eligible 3-phase MV BESS buses
# - writes both explicit-BESS mvagg and compatibility mvagg
#   where BESS P/Q is folded into load P/Q
# ============================================================
import os
import math
import time
import re
import shutil
from pathlib import Path

import pandas as pd

# ---------------- user controls ----------------
INCLUDE_BESS = True

TOTAL_SCENARIOS = 200
N_SAMPLES_PER_SCENARIO = 40
SCENARIOS_PER_CHUNK = 50
BASE_SEED = 20420230

SIGMA_LOAD = 0.5
SIGMA_PV = 0.5

P_LOAD_MEAN_KW = 13731.9
Q_LOAD_MEAN_KVAR = 2610.15
P_LOAD_SCALE_RANGE = (0.3, 1.8)
Q_LOAD_SCALE_RANGE = (0.3, 1.8)
P_PV_SCALE_RANGE = (0.3, 1.8)

BESS_TOTAL_MVA_MEAN = 4.0
BESS_TOTAL_MVA_SIGMA = 0.1
BESS_NUM_NODES_MIN = 1
BESS_NUM_NODES_MAX = 3
BESS_Q_FRAC_MAX = 0.44
BESS_SCATTERED_BUS_TARGET = 72
BESS_MIN_ELECTRICAL_DISTANCE_OHM = 0.5

# Disk-saving options (important on K: / Google Drive)
SAVE_EXPLICIT_BESS_MVAGG = False   # training only needs 3-feature mvagg.csv
DELETE_RAW_NODE_CSV_AFTER_MVAGG = True
MIN_FREE_GB_BEFORE_CHUNK = 2.0

VMIN_SAFE_PU = 0.55
VMAX_SAFE_PU = 1.45

NODE_PE_K = 8
NODE_PE_SEED = 42
NODE_PE_ZERO_EIG_TOL = 1e-8

RETURN_NODE_DF = False

CANDIDATES = [
    r"C:\Users\alita\OneDrive\Desktop\GNN2",
    "/content/GNN-Sandia",
    os.getcwd(),
]

# This folder is separate from the no-BESS dataset.
CHUNK_ROOT = Path(r"K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40")
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
print("CHUNK_ROOT:", CHUNK_ROOT)

script_path = None
for root in CANDIDATES:
    p = os.path.join(root, "run_original_style_dataset_8500_unbalanced.py")
    if os.path.isfile(p):
        script_path = os.path.abspath(p)
        break

if script_path is None:
    raise FileNotFoundError("run_original_style_dataset_8500_unbalanced.py not found in candidates.")

os.chdir(os.path.dirname(script_path))
print("CWD:", os.getcwd())
print("SCRIPT:", script_path)

ns = {"__name__": "run_original_style_dataset_8500_unbalanced", "__file__": script_path}
exec(open(script_path, encoding="utf-8").read(), ns)

FEEDER_DIR = Path(script_path).resolve().parent / "8500 nodes with solar unbalanced"
TX_DSS = FEEDER_DIR / "LoadXfmrs.dss"
if not TX_DSS.exists():
    raise FileNotFoundError(f"Missing file: {TX_DSS}")


def tok(s: str) -> str:
    return str(s).strip().lower()


def bus_base(node_or_bus: str) -> str:
    return tok(node_or_bus).split(".")[0]


def build_tx_map() -> pd.DataFrame:
    wdg_bus_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
    map_rows = []

    for line in TX_DSS.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s.lower().startswith("new transformer."):
            continue

        w = {int(k): tok(v) for k, v in wdg_bus_re.findall(s)}
        if not (1 in w and 2 in w and 3 in w):
            continue

        x2 = bus_base(w[2])
        x3 = bus_base(w[3])
        if not (x2.startswith("x") and x3.startswith("x")):
            continue

        mv_node = w[1]
        sx_set = {"s" + x2, "s" + x3}
        for sx_bus in sx_set:
            map_rows.append((mv_node, sx_bus))

    out = pd.DataFrame(map_rows, columns=["mv_node", "sx_bus"]).drop_duplicates()
    dup = out.duplicated(["mv_node", "sx_bus"]).sum()
    assert dup == 0, f"Unexpected duplicate mappings: {dup}"
    return out


def build_scattered_bess_candidate_csv(out_dir: Path) -> tuple[list[str], Path]:
    ns["_compile_8500_unbalanced_daily_setup"]()
    ns["_detach_daily_loadshape_from_loads"]()
    node_names_all, _, _, _ = ns["inj"].get_all_bus_phase_nodes()
    node_names_graph = ns["_filter_out_x_sx_nodes"](node_names_all)
    node_set_all = {str(n).strip().lower() for n in node_names_all}

    # Preferred path: transformer-adjacent MV load buses.
    auto_nodes = ns["_collect_bess_candidate_nodes_from_mv_load_transformers"](node_names_all)
    bus_to_nodes = ns["_candidate_three_phase_buses_from_nodes"](auto_nodes)
    method = "transformer-adjacent MV load buses"

    # Fallback: any valid 3-phase MV bus in the retained feeder graph.
    if not bus_to_nodes:
        fallback_nodes = []
        for node in node_names_all:
            s = str(node).strip().lower()
            if "." not in s:
                continue
            bus, phs = s.rsplit(".", 1)
            if bus.startswith(("x", "sx", "sourcebus", "hvmv_sub_hsb", "_hvmv_sub")):
                continue
            try:
                ph = int(phs)
            except Exception:
                continue
            if ph not in (1, 2, 3):
                continue
            try:
                ns["dss"].Circuit.SetActiveBus(bus)
                kvb = float(ns["dss"].Bus.kVBase())
            except Exception:
                kvb = float("nan")
            if pd.notna(kvb) and (kvb > 1.0) and (kvb <= 40.0):
                fallback_nodes.append(f"{bus}.{ph}")
        bus_to_nodes = ns["_candidate_three_phase_buses_from_nodes"](fallback_nodes)
        method = "fallback all 3-phase MV buses"

    if not bus_to_nodes:
        raise RuntimeError("No eligible 3-phase MV BESS candidate buses were found, even after MV fallback.")

    # Build electrical-distance metadata so the saved CSV is spread over the feeder.
    edge_tmp = out_dir / "_tmp_candidate_edges.csv"
    ns["inj"].extract_static_phase_edges_to_csv(
        node_names_master=node_names_graph,
        edge_csv_path=str(edge_tmp),
        excluded_buses=(),
    )
    node_to_dist = ns["lt_dist"]._compute_electrical_distance_from_source(node_names_graph, str(edge_tmp))

    rows = []
    dropped_regxfmr = []
    dropped_near_source = []
    for bus, nodes in sorted(bus_to_nodes.items()):
        dist_vals = [float(node_to_dist.get(node, float("nan"))) for node in nodes]
        finite = [x for x in dist_vals if pd.notna(x)]
        dist = float(sum(finite) / len(finite)) if finite else float("nan")

        if str(bus).lower().startswith("regxfmr_"):
            dropped_regxfmr.append((bus, dist))
            continue
        if pd.notna(dist) and float(dist) < float(BESS_MIN_ELECTRICAL_DISTANCE_OHM):
            dropped_near_source.append((bus, dist))
            continue

        rows.append(
            {
                "bus": bus,
                "node_1": nodes[0],
                "node_2": nodes[1],
                "node_3": nodes[2],
                "electrical_distance_ohm": dist,
            }
        )

    cand_df = pd.DataFrame(rows).sort_values(["electrical_distance_ohm", "bus"], kind="stable").reset_index(drop=True)
    if cand_df.empty:
        raise RuntimeError(
            "Candidate filtering removed all buses. "
            "Reduce BESS_MIN_ELECTRICAL_DISTANCE_OHM or relax exclusions."
        )
    n_total = len(cand_df)
    target = max(int(BESS_NUM_NODES_MAX), min(int(BESS_SCATTERED_BUS_TARGET), n_total))

    if n_total > target:
        picks = sorted({int(round(i * (n_total - 1) / (target - 1))) for i in range(target)})
        while len(picks) < target:
            for idx in range(n_total):
                if idx not in picks:
                    picks.append(idx)
                if len(picks) == target:
                    break
        cand_df = cand_df.iloc[sorted(picks)].reset_index(drop=True)

    cand_df.insert(0, "selection_rank", range(1, len(cand_df) + 1))
    cand_df.insert(1, "selection_method", method)

    csv_path = out_dir / "bess_candidate_buses_scattered_3ph_mv.csv"
    cand_df.to_csv(csv_path, index=False)

    candidate_nodes = []
    for _, row in cand_df.iterrows():
        candidate_nodes.extend([str(row["node_1"]), str(row["node_2"]), str(row["node_3"])])

    # ---------------- sanity checks ----------------
    req_cols = {
        "selection_rank", "selection_method", "bus", "node_1", "node_2", "node_3", "electrical_distance_ohm"
    }
    miss = req_cols - set(cand_df.columns)
    if miss:
        raise RuntimeError(f"Candidate CSV missing required columns: {sorted(miss)}")

    bus_unique = int(cand_df["bus"].nunique())
    if bus_unique != len(cand_df):
        raise RuntimeError("Candidate CSV contains duplicate buses.")

    node_cols = ["node_1", "node_2", "node_3"]
    phase_ok = True
    missing_nodes = []
    bad_phase_rows = []
    for _, row in cand_df.iterrows():
        nodes = [str(row[c]).strip().lower() for c in node_cols]
        buses = {n.rsplit(".", 1)[0] for n in nodes}
        phases = sorted(int(n.rsplit(".", 1)[1]) for n in nodes)
        if len(buses) != 1 or phases != [1, 2, 3]:
            phase_ok = False
            bad_phase_rows.append((row["bus"], nodes))
        for n in nodes:
            if n not in node_set_all:
                missing_nodes.append(n)

    if not phase_ok:
        raise RuntimeError(f"Candidate CSV has non-3phase rows, examples: {bad_phase_rows[:3]}")
    if missing_nodes:
        raise RuntimeError(f"Candidate CSV contains nodes missing from circuit, examples: {missing_nodes[:6]}")

    dist_series = cand_df["electrical_distance_ohm"].astype(float)
    dist_finite = dist_series[pd.notna(dist_series)]
    dist_min = float(dist_finite.min()) if len(dist_finite) else float("nan")
    dist_max = float(dist_finite.max()) if len(dist_finite) else float("nan")
    dist_q = dist_finite.quantile([0.1, 0.5, 0.9]).to_dict() if len(dist_finite) else {}
    gaps = dist_finite.sort_values().diff().dropna()
    gap_min = float(gaps.min()) if len(gaps) else float("nan")
    gap_med = float(gaps.median()) if len(gaps) else float("nan")

    print(
        f"Saved {len(cand_df)} scattered candidate 3-phase MV buses "
        f"({len(candidate_nodes)} phase nodes) using {method}."
    )
    print("Candidate CSV:", csv_path)
    print("First few candidate buses:", cand_df["bus"].head(10).tolist())
    print(
        f"Filtered out regxfmr_* buses: {len(dropped_regxfmr)} | "
        f"near-source buses (< {float(BESS_MIN_ELECTRICAL_DISTANCE_OHM):.3f} ohm): {len(dropped_near_source)}"
    )
    if dropped_regxfmr:
        print(" - sample regxfmr drops:", dropped_regxfmr[:5])
    if dropped_near_source:
        print(" - sample near-source drops:", dropped_near_source[:5])
    print("Sanity checks:")
    print(f" - unique buses: {bus_unique}")
    print(f" - phase rows valid [.1,.2,.3 on same bus]: {phase_ok}")
    print(f" - nodes present in circuit: {len(missing_nodes) == 0}")
    print(f" - electrical distance finite rows: {int(len(dist_finite))}/{len(cand_df)}")
    print(f" - distance range ohm: [{dist_min:.3f}, {dist_max:.3f}]")
    if dist_q:
        print(
            " - distance quantiles ohm: "
            f"q10={float(dist_q.get(0.1, float('nan'))):.3f}, "
            f"q50={float(dist_q.get(0.5, float('nan'))):.3f}, "
            f"q90={float(dist_q.get(0.9, float('nan'))):.3f}"
        )
    print(f" - spacing gaps ohm: min={gap_min:.3f}, median={gap_med:.3f}")
    return candidate_nodes, csv_path


def build_mvagg_outputs_for_chunk(chunk_dir: Path, map_long: pd.DataFrame) -> dict:
    in_csv = chunk_dir / "gnn_node_features_and_targets.csv"
    idx_csv = chunk_dir / "gnn_node_index_master.csv"
    out_explicit = chunk_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv"
    out_compat = chunk_dir / "gnn_node_features_and_targets_mvagg.csv"

    if not in_csv.exists() or not idx_csv.exists():
        raise FileNotFoundError(f"Missing inputs for mvagg build under: {chunk_dir}")

    df = pd.read_csv(in_csv)
    idx = pd.read_csv(idx_csv, usecols=["node"])
    allowed_nodes = set(idx["node"].astype(str).str.strip().str.lower())

    df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
    df["bus"] = df["node_lc"].str.split(".").str[0]

    sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].copy()
    sx = sx.rename(columns={"bus": "sx_bus"})

    sx_join = sx.merge(map_long, on="sx_bus", how="inner")
    mv_agg = (
        sx_join.groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
        .sum()
        .rename(columns={"p_load_kw": "p_load_kw_agg", "q_load_kvar": "q_load_kvar_agg"})
    )

    df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
    df2 = df2[df2["node_lc"].isin(allowed_nodes)].copy()

    # Replace raw LV-side loads with MV-aggregated equivalent load.
    df2["p_load_kw"] = 0.0
    df2["q_load_kvar"] = 0.0
    df2 = df2.merge(
        mv_agg,
        left_on=["sample_id", "node_lc"],
        right_on=["sample_id", "mv_node"],
        how="left",
    )

    hit = df2["p_load_kw_agg"].notna()
    df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_load_kw_agg"]
    df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_load_kvar_agg"]

    for c in ["node_lc", "bus", "mv_node", "p_load_kw_agg", "q_load_kvar_agg"]:
        if c in df2.columns:
            df2.drop(columns=c, inplace=True)

    # Save only what we need for training augmentation.
    if SAVE_EXPLICIT_BESS_MVAGG:
        df2.to_csv(out_explicit, index=False)

    compat = df2.copy()
    compat["p_load_kw"] = compat["p_load_kw"].astype(float) + compat["p_bess_kw"].fillna(0.0).astype(float)
    compat["q_load_kvar"] = compat["q_load_kvar"].astype(float) + compat["q_bess_kvar"].fillna(0.0).astype(float)
    compat.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")
    compat.to_csv(out_compat, index=False)

    if DELETE_RAW_NODE_CSV_AFTER_MVAGG and in_csv.exists():
        in_csv.unlink()
        print("   deleted raw:", in_csv.name)

    n_samples = int(df2["sample_id"].nunique()) if len(df2) else 0
    rows_per_sample = df2.groupby("sample_id").size() if len(df2) else pd.Series(dtype=int)
    rmin = int(rows_per_sample.min()) if len(rows_per_sample) else 0
    rmax = int(rows_per_sample.max()) if len(rows_per_sample) else 0
    print("Post-processed:")
    if SAVE_EXPLICIT_BESS_MVAGG:
        print(" -", out_explicit)
    print(" -", out_compat)
    print(f"   rows={len(df2)} samples={n_samples} rows_per_sample=[{rmin}, {rmax}]")
    return {
        "rows": int(len(df2)),
        "samples": n_samples,
        "rows_per_sample_min": rmin,
        "rows_per_sample_max": rmax,
    }


BESS_CANDIDATE_NODES_AUTO, BESS_CANDIDATE_CSV = build_scattered_bess_candidate_csv(CHUNK_ROOT)
TX_MAP_LONG = build_tx_map()

root = CHUNK_ROOT
if not root.exists():
    raise FileNotFoundError(f"CHUNK_ROOT not found: {root}")

n_chunks = math.ceil(TOTAL_SCENARIOS / SCENARIOS_PER_CHUNK)
print(f"\nTotal chunks: {n_chunks} | INCLUDE_BESS={INCLUDE_BESS}")

free_gb = shutil.disk_usage(CHUNK_ROOT).free / (1024 ** 3)
print(f"Free space on drive: {free_gb:.2f} GB")
if free_gb < MIN_FREE_GB_BEFORE_CHUNK:
    raise OSError(
        f"Not enough free space on {CHUNK_ROOT.drive}: {free_gb:.2f} GB free, "
        f"need at least {MIN_FREE_GB_BEFORE_CHUNK:.2f} GB"
    )

for chunk_idx in range(n_chunks):
    free_gb = shutil.disk_usage(CHUNK_ROOT).free / (1024 ** 3)
    if free_gb < MIN_FREE_GB_BEFORE_CHUNK:
        raise OSError(
            f"Stopping before chunk {chunk_idx + 1}: only {free_gb:.2f} GB free on "
            f"{CHUNK_ROOT.drive} (need {MIN_FREE_GB_BEFORE_CHUNK:.2f} GB)"
        )

    s0 = chunk_idx * SCENARIOS_PER_CHUNK
    n_this = min(SCENARIOS_PER_CHUNK, TOTAL_SCENARIOS - s0)
    chunk_seed = int(BASE_SEED + 100003 * (chunk_idx + 1))
    out_dir = CHUNK_ROOT / f"run_{chunk_idx+1:03d}_scen_{s0:04d}_{s0+n_this-1:04d}_seed_{chunk_seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ns["OUT_DIR"] = out_dir
    ns["EDGE_CSV"] = out_dir / "gnn_edges_phase_static.csv"
    ns["NODE_CSV"] = out_dir / "gnn_node_features_and_targets.csv"
    ns["SAMPLE_CSV"] = out_dir / "gnn_sample_meta.csv"
    ns["NODE_INDEX_CSV"] = out_dir / "gnn_node_index_master.csv"

    print(f"\n=== Chunk {chunk_idx+1}/{n_chunks} ===")
    print(f"Scenarios in chunk: {n_this}")
    print(f"Seed: {chunk_seed}")
    print(f"OUT_DIR: {out_dir}")

    gen_kw = dict(
        n_scenarios=int(n_this),
        k_snapshots_per_scenario_total=int(N_SAMPLES_PER_SCENARIO),
        bins_by_profile={"load": 3, "pv": 3, "net": 3},
        include_anchors=True,
        master_seed=int(chunk_seed),
        sigma_load=float(SIGMA_LOAD),
        sigma_pv=float(SIGMA_PV),
        p_load_mean_kw=float(P_LOAD_MEAN_KW),
        q_load_mean_kvar=float(Q_LOAD_MEAN_KVAR),
        p_load_scale_range=tuple(P_LOAD_SCALE_RANGE),
        q_load_scale_range=tuple(Q_LOAD_SCALE_RANGE),
        p_pv_scale_range=tuple(P_PV_SCALE_RANGE),
        vmin_safe_pu=float(VMIN_SAFE_PU),
        vmax_safe_pu=float(VMAX_SAFE_PU),
        include_source_in_safe_band=True,
        return_node_df=bool(RETURN_NODE_DF),
        node_pe_k=int(NODE_PE_K),
        node_pe_seed=int(NODE_PE_SEED),
        node_pe_zero_eig_tol=float(NODE_PE_ZERO_EIG_TOL),
        include_bess=bool(INCLUDE_BESS),
        bess_total_mva_mean=float(BESS_TOTAL_MVA_MEAN),
        bess_total_mva_sigma=float(BESS_TOTAL_MVA_SIGMA),
        bess_num_nodes_min=int(BESS_NUM_NODES_MIN),
        bess_num_nodes_max=int(BESS_NUM_NODES_MAX),
        bess_q_frac_max=float(BESS_Q_FRAC_MAX),
        bess_candidate_nodes_override=BESS_CANDIDATE_NODES_AUTO,
    )

    t0 = time.time()
    df_sample, df_node = ns["generate_original_style_dataset_8500_unbalanced"](**gen_kw)
    mvagg_stats = build_mvagg_outputs_for_chunk(out_dir, TX_MAP_LONG)

    dt = time.time() - t0
    n_kept = int(df_sample["sample_id"].nunique()) if len(df_sample) else 0
    print(f"Chunk done in {dt/60:.1f} min | kept samples: {n_kept} | rows sample_meta: {len(df_sample)}")
    print(
        f"MVAGG summary | rows={mvagg_stats['rows']} | samples={mvagg_stats['samples']} "
        f"| rows/sample=[{mvagg_stats['rows_per_sample_min']}, {mvagg_stats['rows_per_sample_max']}]"
    )
    print("Saved:")
    print(" -", BESS_CANDIDATE_CSV)
    print(" -", ns["NODE_INDEX_CSV"])
    print(" -", ns["EDGE_CSV"])
    print(" -", ns["SAMPLE_CSV"])
    if not DELETE_RAW_NODE_CSV_AFTER_MVAGG:
        print(" -", ns["NODE_CSV"])
    print(" -", out_dir / "gnn_node_features_and_targets_mvagg.csv")
    if SAVE_EXPLICIT_BESS_MVAGG:
        print(" -", out_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv")

print("\nAll chunks finished.")

CHUNK_ROOT: K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40
CWD: c:\Users\alita\OneDrive\Desktop\GNN2
SCRIPT: C:\Users\alita\OneDrive\Desktop\GNN2\run_original_style_dataset_8500_unbalanced.py
[saved] phase-edge CSV -> K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40\_tmp_candidate_edges.csv | rows=7628 | cols=17 | bidirectional=True
Saved 72 scattered candidate 3-phase MV buses (216 phase nodes) using fallback all 3-phase MV buses.
Candidate CSV: K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40\bess_candidate_buses_scattered_3ph_mv.csv
First few candidate buses: ['m1209797', 'm3032980', 'm1209774', 'l3139366', 'e184626', 'm1149235', 'l3081380', 'm1142839', 'l2674047', 'm1142818']
Filtered out regxfmr_* buses: 4 | near-source buses (< 0.500 ohm): 15
 - sample regxfmr drops: [('regxfmr_190-7361', 6.553390629007642), ('regxfmr_190-8581', 4.481039542293969), ('regxfmr_190-8593', 6.972486484840

post processing

In [ ]:
import re
from pathlib import Path
import pandas as pd

# ---------------- paths ----------------
FEEDER_DIR = Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500 nodes with solar unbalanced")
CHUNK_PARENT = Path(r"K:\My Drive\datasets_gnn2\original_8500_unbalanced_chunked_with_bess_aug_2000_40")
TX_DSS = FEEDER_DIR / "LoadXfmrs.dss"

if not TX_DSS.exists():
    raise FileNotFoundError(f"Missing file: {TX_DSS}")
if not CHUNK_PARENT.exists():
    raise FileNotFoundError(f"Missing folder: {CHUNK_PARENT}")

def tok(s: str) -> str:
    return str(s).strip().lower()

def bus_base(node_or_bus: str) -> str:
    return tok(node_or_bus).split(".")[0]

# ---------------- 1) build global transformer map once ----------------
wdg_bus_re = re.compile(r"wdg=(\d+)\s+bus=([^\s]+)", re.I)
map_rows = []

for line in TX_DSS.read_text(encoding="utf-8", errors="ignore").splitlines():
    s = line.strip()
    if not s.lower().startswith("new transformer."):
        continue

    w = {int(k): tok(v) for k, v in wdg_bus_re.findall(s)}
    if not (1 in w and 2 in w and 3 in w):
        continue

    x2 = bus_base(w[2])
    x3 = bus_base(w[3])
    if not (x2.startswith("x") and x3.startswith("x")):
        continue

    mv_node = w[1]                   # node-level MV target (keep phase)
    sx_set = {"s" + x2, "s" + x3}    # dedupe prevents double counting
    for sx_bus in sx_set:
        map_rows.append((mv_node, sx_bus))

map_long = pd.DataFrame(map_rows, columns=["mv_node", "sx_bus"]).drop_duplicates()

dup = map_long.duplicated(["mv_node", "sx_bus"]).sum()
assert dup == 0, f"Unexpected duplicate mappings: {dup}"

print("Global mapping ready.")
print("Unique MV nodes mapped:", map_long["mv_node"].nunique())
print("Mapping rows (mv_node,sx_bus):", len(map_long))

# ---------------- 2) process all chunks ----------------
chunk_dirs = sorted([p for p in CHUNK_PARENT.iterdir() if p.is_dir() and p.name.startswith("run_")])
print(f"Found {len(chunk_dirs)} chunk folders.")

summary = []
failed = []
skipped = []

for i, chunk_dir in enumerate(chunk_dirs, 1):
    in_csv = chunk_dir / "gnn_node_features_and_targets.csv"
    idx_csv = chunk_dir / "gnn_node_index_master.csv"
    out_explicit = chunk_dir / "gnn_node_features_and_targets_mvagg_with_bess_cols.csv"
    out_compat = chunk_dir / "gnn_node_features_and_targets_mvagg.csv"

    try:
        if not in_csv.exists() or not idx_csv.exists():
            raise FileNotFoundError(f"Missing input(s): {in_csv.exists()=}, {idx_csv.exists()=}")

        if out_explicit.exists() and out_explicit.stat().st_size > 0 and out_compat.exists() and out_compat.stat().st_size > 0:
            skipped.append(chunk_dir.name)
            print(f"[{i}/{len(chunk_dirs)}] SKIP {chunk_dir.name} (already post-processed)")
            continue

        df = pd.read_csv(in_csv)
        need = {"sample_id", "node", "p_load_kw", "q_load_kvar", "p_pv_kw", "p_bess_kw", "q_bess_kvar"}
        miss = need - set(df.columns)
        if miss:
            raise ValueError(f"Missing required columns: {miss}")

        idx = pd.read_csv(idx_csv, usecols=["node"])
        allowed_nodes = set(idx["node"].astype(str).str.strip().str.lower())

        df["node_lc"] = df["node"].astype(str).str.strip().str.lower()
        df["bus"] = df["node_lc"].str.split(".").str[0]

        # aggregate SX -> exact MV node for load channels only
        sx = df[df["bus"].str.startswith("sx")][["sample_id", "bus", "p_load_kw", "q_load_kvar"]].copy()
        sx = sx.rename(columns={"bus": "sx_bus"})

        sx_join = sx.merge(map_long, on="sx_bus", how="inner")
        mv_agg = (
            sx_join.groupby(["sample_id", "mv_node"], as_index=False)[["p_load_kw", "q_load_kvar"]]
            .sum()
            .rename(columns={"p_load_kw": "p_load_kw_agg", "q_load_kvar": "q_load_kvar_agg"})
        )

        # remove X/SX and filter to index master
        df2 = df[~(df["bus"].str.startswith("x") | df["bus"].str.startswith("sx"))].copy()
        df2 = df2[df2["node_lc"].isin(allowed_nodes)].copy()

        # strict MV load assignment from aggregated SX load
        df2["p_load_kw"] = 0.0
        df2["q_load_kvar"] = 0.0

        df2 = df2.merge(
            mv_agg,
            left_on=["sample_id", "node_lc"],
            right_on=["sample_id", "mv_node"],
            how="left"
        )

        hit = df2["p_load_kw_agg"].notna()
        df2.loc[hit, "p_load_kw"] = df2.loc[hit, "p_load_kw_agg"]
        df2.loc[hit, "q_load_kvar"] = df2.loc[hit, "q_load_kvar_agg"]

        for c in ["node_lc", "bus", "mv_node", "p_load_kw_agg", "q_load_kvar_agg"]:
            if c in df2.columns:
                df2.drop(columns=c, inplace=True)

        # explicit-BESS mvagg output: keeps 5 dynamic features
        df2.to_csv(out_explicit, index=False)

        # compatibility mvagg output: folds BESS into load, drops BESS columns
        compat = df2.copy()
        compat["p_load_kw"] = compat["p_load_kw"].astype(float) + compat["p_bess_kw"].fillna(0.0).astype(float)
        compat["q_load_kvar"] = compat["q_load_kvar"].astype(float) + compat["q_bess_kvar"].fillna(0.0).astype(float)
        compat.drop(columns=["p_bess_kw", "q_bess_kvar"], inplace=True, errors="ignore")
        compat.to_csv(out_compat, index=False)

        # quick stats
        n_samples = int(df2["sample_id"].nunique()) if len(df2) else 0
        rows_per_sample = df2.groupby("sample_id").size() if len(df2) else pd.Series(dtype=int)
        rmin = int(rows_per_sample.min()) if len(rows_per_sample) else 0
        rmax = int(rows_per_sample.max()) if len(rows_per_sample) else 0

        summary.append({
            "chunk": chunk_dir.name,
            "samples": n_samples,
            "rows": int(len(df2)),
            "rows_per_sample_min": rmin,
            "rows_per_sample_max": rmax,
            "saved_explicit": str(out_explicit),
            "saved_compat": str(out_compat),
        })

        print(
            f"[{i}/{len(chunk_dirs)}] OK  {chunk_dir.name} "
            f"-> rows={len(df2)} samples={n_samples} "
            f"rps=[{rmin},{rmax}]"
        )
        print("   explicit:", out_explicit.name)
        print("   compat  :", out_compat.name)

    except Exception as e:
        failed.append((chunk_dir.name, str(e)))
        print(f"[{i}/{len(chunk_dirs)}] FAIL {chunk_dir.name}: {e}")

print("\nDone.")
print(f"Success: {len(summary)} | Skipped: {len(skipped)} | Failed: {len(failed)}")

if skipped:
    print("\nSkipped (already post-processed):")
    for name in skipped:
        print(f"- {name}")

if failed:
    print("\nFailures:")
    for name, err in failed:
        print(f"- {name}: {err}")

inspection

In [ ]:
import pandas as pd
from pathlib import Path

in_csv = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked/run_001_scen_0000_0049_seed_20360133/gnn_node_features_and_targets_mvagg.csv")
out_csv = in_csv.with_name("gnn_node_features_and_targets_mvagg_sample0.csv")

df = pd.read_csv(in_csv)

# first sample by appearance order in file
first_sid = df["sample_id"].iloc[0]
df_first = df[df["sample_id"] == first_sid].copy()

df_first.to_csv(out_csv, index=False)
print("first sample_id:", first_sid)
print("rows saved:", len(df_first))
print("saved:", out_csv)

plotting

In [10]:
%matplotlib inline
import os
import sys
from pathlib import Path

# ── find repo (Colab clone, GNN2_REPO_ROOT, Windows path, or cwd) ─────────────
def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "nonunique_notebook_bootstrap.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone to /content/GNN2 first. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )

REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Reload after `git pull` on Colab so you get p_pv_kw fix, wide warm-starts, discrete aux metrics
for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_warmstart_band_daily",
    "nonunique_opendss_daily",
    "nonunique_da_gps_daily_compare",
    "run_da_gps_daily_opendss_compare",
    "compare_gnn_inference_utils",
    "nonunique_notebook_bootstrap",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook
from nonunique_opendss_daily import DailySimConfig
from nonunique_da_gps_warmstart_band_daily import run_da_gps_warmstart_band_daily

# ── checkpoint source ─────────────────────────────────────────────────────────
# Default: shipped CCE baseline (h=96, L=2, reg CE, 4 meta-aux heads)
USE_FINETUNED_CHECKPOINT = False
# Local fine-tune run (also try Drive path on Colab). Toggle True to use it here,
# or run the appended fine-tune warmstart cell at the end of this notebook.
FINETUNE_RUN_DIR = Path(
    r"C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints"
    r"\da_gps_finetune_withder_l2_h96_regce_20260710_011052"
)
if not FINETUNE_RUN_DIR.is_dir():
    _drive_ft = Path(
        "/content/drive/MyDrive/datasets_gnn2/runs/"
        "da_gps_finetune_withder_l2_h96_regce_20260710_011052"
    )
    if _drive_ft.is_dir():
        FINETUNE_RUN_DIR = _drive_ft


DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="auto",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
)

if USE_FINETUNED_CHECKPOINT:
    finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
    ckpt = finetune_run / "da_gps_multitask_best.pt"
    if not ckpt.is_file():
        ckpt = finetune_run / "training_last.pt"
    if not ckpt.is_file():
        raise FileNotFoundError(f"No checkpoint in {finetune_run}")
    boot = boot.__class__(
        **{
            **boot.__dict__,
            "run_dir": finetune_run,
            "checkpoint": ckpt,
        }
    )
    print(f"[checkpoint] fine-tuned: {ckpt}")

# ── knobs (same on Colab and local) ───────────────────────────────────────────
INCLUDE_DER      = True
DER_MAX_KW       = 1000.0
DER_MAX_KVAR     = 500.0
# Comma-separated: total P/Q split equally across buses, then equally across 3 phases (GNN).
# OpenDSS: one 3ph Generator per bus with that bus's P/Q share. Q/P = DER_MAX_KVAR/DER_MAX_KW.
DER_BUS = "m1142818,r42246"

N_WARM_STARTS    = 5                 # more starts → wider cloud (slower)
WARM_START_MODE  = "wide"             # "uniform" | "corners" | "wide"
WARM_START_RANDOMIZE_STATIC_CAPS = False  # True = wilder cap bands
WARMSTART_SEED   = 42

STEP_MIN         = 5
DAILY_STRESS     = 0.0
SCENARIO_SCALE   = 1.0
REF_SAMPLE_INDEX = 0

PLOT_ALL_CACHE_NODES = True
PLOT_ALL_MAX_NODES   = 100              # 0 = all cache∩circuit nodes
PLOT_REG_CAP         = True
PLOT_META_AUX        = True
PLOT_WARMSTART_LINES = True
SHOW_INLINE          = False
VOLTAGE_PLOT_DPI     = 96
GNN_BATCH_STEPS      = None           # or 8; also env GNN_BATCH_STEPS

OUT_DIR = boot.out_dir                # or Path("/content/drive/MyDrive/.../warmstart_runs/my_tag")

# ── run ─────────────────────────────────────────────────────────────────────
cfg = DailySimConfig(
    step_min=STEP_MIN,
    include_der=INCLUDE_DER,
    der_nominal_kw=float(DER_MAX_KW if INCLUDE_DER else 0.0),
    der_nominal_kvar=float(DER_MAX_KVAR if INCLUDE_DER else 0.0),
    der_bus=str(DER_BUS),
    der_profile_csv=boot.der_profile,
    da_gps_run_dir=boot.run_dir,
    da_gps_cache_pt=boot.cache_pt,
    da_gps_checkpoint=boot.checkpoint,
)

print(
    f"env={'Colab' if boot.on_colab else 'local'}  device={boot.device}\n"
    f"checkpoint={boot.checkpoint}\n"
    f"DER={'ON' if INCLUDE_DER else 'OFF'}  "
    f"warm_starts={N_WARM_STARTS}  mode={WARM_START_MODE!r}  step_min={STEP_MIN}"
)

result = run_da_gps_warmstart_band_daily(
    cfg,
    n_warm_starts=N_WARM_STARTS,
    warm_start_mode=WARM_START_MODE,
    warm_start_randomize_static_caps=WARM_START_RANDOMIZE_STATIC_CAPS,
    seed=WARMSTART_SEED,
    load_profile_path=boot.load_profile,
    pv_profile_path=boot.irr_profile,
    ref_sample_index=REF_SAMPLE_INDEX,
    scenario_scale=SCENARIO_SCALE,
    daily_stress=DAILY_STRESS,
    plot_all_cache_nodes=PLOT_ALL_CACHE_NODES,
    plot_all_max_nodes=PLOT_ALL_MAX_NODES,
    out_dir=OUT_DIR,
    voltage_plot_dpi=VOLTAGE_PLOT_DPI,
    plot_reg_cap=PLOT_REG_CAP,
    plot_meta_aux=PLOT_META_AUX,
    plot_warmstart_lines=PLOT_WARMSTART_LINES,
    show=SHOW_INLINE,
    device=boot.device,
    gnn_batch_steps=GNN_BATCH_STEPS,
)

inside = result["da_gps_inside_band_frac"]
proximity = result["da_gps_cloud_proximity"]
set_dist = result["da_gps_set_distance"]
outside_dist = result["da_gps_mean_outside_distance"]
aggregated = result["da_gps_aggregated"]

_GROUP_UNITS = {
    "voltage": "pu",
    "regulator": "tap steps",
    "capacitor": "cap steps ON",
    "meta_aux": "pu/kW/kvar",
}

print("\nOutputs:", result["out_dir"])
print(f"Nodes: {len(result['collect_nodes'])}")

print("\n=== Aggregated warm-start band metrics (all devices) ===")
for group, stats in aggregated.items():
    n = int(stats.get("n_devices", 0))
    frac = float(stats.get("mean_inside_band_frac", float("nan")))
    prox = float(stats.get("mean_cloud_proximity", float("nan")))
    sdist = float(stats.get("mean_set_distance", float("nan")))
    odist = float(stats.get("mean_outside_distance", float("nan")))
    unit = _GROUP_UNITS.get(group, "")
    frac_s = f"{100.0 * frac:.1f}%" if frac == frac else "n/a"
    prox_s = f"{prox:.3f}" if prox == prox else "n/a"
    sdist_s = f"{sdist:.4g}" if sdist == sdist else "n/a"
    odist_s = f"{odist:.4g}" if odist == odist else "n/a"
    print(
        f"  [{group}] n={n}  inside={frac_s}  "
        f"cloud_proximity={prox_s}  set_distance={sdist_s}  "
        f"mean_outside_distance={odist_s} {unit}"
    )

print("\n=== Per-device breakdown ===")
for group, unit in (("regulator", "tap steps"), ("capacitor", "cap steps ON"), ("meta_aux", "pu/kW/kvar")):
    ib = inside.get(group) or {}
    pr = proximity.get(group) or {}
    sd = set_dist.get(group) or {}
    od = outside_dist.get(group) or {}
    if not ib:
        continue
    print(f"[{group}]")
    for name in sorted(ib):
        print(
            f"  {name}: inside={100.0 * ib[name]:.1f}%  "
            f"proximity={pr.get(name, float('nan')):.3f}  "
            f"set dist={sd.get(name, float('nan')):.4g}  "
            f"outside dist={od.get(name, float('nan')):.4g} {unit}"
        )

{
    "aggregated": aggregated,
    "inside_band_frac": inside,
    "cloud_proximity": proximity,
    "set_distance": set_dist,
    "mean_outside_distance": outside_dist,
}

[bootstrap] env=local  repo=C:\Users\alita\OneDrive\Desktop\GNN2
[bootstrap] device=cpu  cache=run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=C:\Users\alita\OneDrive\Desktop\GNN2\warmstart_band_runs\20260710_100231
env=local  device=cpu
checkpoint=C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE\training_last.pt
DER=ON  warm_starts=5  mode='wide'  step_min=5
DA-GPS warm-start band daily (OpenDSS snapshot + N random controller inits/step)
  step_min=5 min, npts=288, n_warm_starts=5
  warm_start_mode='wide'  randomize_static_caps=False
  load/PV profiles: C:\Users\alita\OneDrive\Desktop\GNN2\a representativ days\load_day_004.csv / C:\Users\alita\OneDrive\Desktop\GNN2\a representativ days\irr_day_004.csv
  cache .pt: C:\Users\alita\OneDrive\Des

{'aggregated': {'voltage': {'n_devices': 3817,
   'mean_inside_band_frac': 0.8803261359997672,
   'mean_cloud_proximity': 0.9434374659274676,
   'mean_set_distance': 0.0004895516580939518,
   'mean_outside_distance': 0.0026366643985013555},
  'regulator': {'n_devices': 12,
   'mean_inside_band_frac': 0.9010416666666666,
   'mean_cloud_proximity': 0.9314941051226464,
   'mean_set_distance': 0.1287615740740741,
   'mean_outside_distance': 1.103688320263023},
  'capacitor': {'n_devices': 10,
   'mean_inside_band_frac': 0.9322916666666666,
   'mean_cloud_proximity': 0.9572001704959832,
   'mean_set_distance': 0.06770833333333333,
   'mean_outside_distance': 0.6},
  'meta_aux': {'n_devices': 4,
   'mean_inside_band_frac': 0.21354166666666666,
   'mean_cloud_proximity': 0.25806113907088746,
   'mean_set_distance': 63.70029949329107,
   'mean_outside_distance': 65.6104055692623}},
 'inside_band_frac': {'voltage': {'_hvmv_sub_lsb.1': 0.75,
   '_hvmv_sub_lsb.2': 0.6979166666666666,
   '_hvmv_su

compare with this

In [ ]:
# --- Lighter DA-GPS training (long-running; run on Colab GPU or local CUDA) ---
import os
import sys
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
WIN_CHUNK_DEFAULT = Path(r"D:\datasets\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]

def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _smoke_chunk_subdir_glob(chunk_parent: Path, count: int) -> str:
    """First `count` run_* names, comma-separated for --chunk_subdir_glob."""
    names = [p.name for p in _sorted_run_chunk_dirs(chunk_parent)]
    if len(names) < count:
        raise ValueError(
            f"SMOKE_CHUNK_COUNT={count} but only {len(names)} run_* under {chunk_parent}"
        )
    return ",".join(names[:count])


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- smoke / full-run toggles (cells 6 baseline & 17 physics; cell 18 compares) ---
SMOKE_TEST = True
SMOKE_CHUNK_COUNT = 3  # first N run_* alphabetically; 3-5 = fast multi-chunk (not all 40)
SMOKE_EPOCHS = 15      # low for speed; more epochs = stabler MAE, ~linear time
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
_DA_CACHE_NAME = "da_gps_chunked_mvagg_smoke_gine" if SMOKE_TEST else "da_gps_chunked_mvagg_full_gine"

# --- paths: auto-detect Colab + Drive; override below if needed ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = MYDRIVE_DATA / "runs"  # persisted on Drive; /content/GNN-Sandia is ephemeral
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"

NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT

# Optional overrides (use POSIX paths on Colab, not D:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"

# --- lighter architecture (sweeps: LIGHTER_LAYERS=1, LIGHTER_HIDDEN=48) ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 4  # current shipped model used 4 at h=96

PHYSICS_WEIGHT = 0.0  # baseline; use 0.01 for physics-informed


# Explicit PF balance nodes when physics on (1177 hetero MV load nodes (node-name keyed; chunk-safe); skips auto mask refinement)
PF_BALANCE_NODE_LIST_CSV = REPO / "colab_pf_data/pf_balance_nodes_explicit.csv"

META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = 200
    PATIENCE = 30
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

# --- PF topology root (repo colab_pf_data/ after git pull, or Drive dailyagg fallback) ---
from gnn2_pf_data_paths import PF_CAP_NODES_REL, PF_REG_CATALOG_REL, resolve_pf_catalog_paths

PF_DATA_ROOT = None
if PHYSICS_WEIGHT > 0:
    _reg_cat, _cap_map, PF_DATA_ROOT = resolve_pf_catalog_paths(
        repo=REPO,
        preferred_root=None,
        chunk_parent=chunk_parent,
    )

print("=== Preflight ===")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"CHUNK_PARENT:   {chunk_parent}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")

if PHYSICS_WEIGHT > 0:
    assert PF_DATA_ROOT is not None
    _pf_checks = [
        ("reg_catalog", PF_DATA_ROOT / PF_REG_CATALOG_REL),
        ("cap_nodes", PF_DATA_ROOT / PF_CAP_NODES_REL),
        ("electrical_distance", PF_DATA_ROOT / "electrical_distance_from_substation.csv"),
        (
            "hetero_mv_nodes",
            PF_DATA_ROOT / "Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer.csv",
        ),
        ("bus_kv_cache", PF_DATA_ROOT / "bus_kv_base_by_node.csv"),
    ]
    print(f"PF_DATA_ROOT:   {PF_DATA_ROOT}")
    for label, p in _pf_checks:
        status = "OK" if p.is_file() else "MISSING"
        print(f"  PF {label}: {status}  {p}")
        if not p.is_file():
            raise FileNotFoundError(f"Physics preflight missing {label}: {p}")
else:
    print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")

print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_pf_suffix = "_pf" if PHYSICS_WEIGHT > 0 else ""
out_dir = runs_parent / (
    f"da_gps_chunked_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_mvagg_gine_metaaux_regce{_pf_suffix}{'_smoke' if SMOKE_TEST else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "voltage" if PHYSICS_WEIGHT > 0 else "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", "0.1",
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "96",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", "0",
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", "0.1",
    "--lambda_reg", "0.1",
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience", str(PATIENCE),
    "--seed", str(SMOKE_SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--log_every", "10",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
]

if PHYSICS_WEIGHT > 0:
    _pf_flags = [
        "--pf_data_root", str(PF_DATA_ROOT),
        "--loss_power_balance_weight", str(PHYSICS_WEIGHT),
        "--pf_sparse_y", "1",
        "--pf_huber_delta_kw", "10",
        "--pf_detach_controls",
    ]
    _bal = Path(PF_BALANCE_NODE_LIST_CSV)
    if not _bal.is_absolute():
        _bal = (REPO / _bal).resolve()
    if not _bal.is_file():
        raise FileNotFoundError(f"PF_BALANCE_NODE_LIST_CSV not found: {_bal}")
    _pf_flags.extend(["--pf_balance_node_list_csv", str(_bal)])
    cmd.extend(_pf_flags)

_mode = "physics-informed" if PHYSICS_WEIGHT > 0 else "baseline"
print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} ({_mode})")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_run_name = out_dir.name
print("\n=== Section 8 prerequisite ===")
print(f"Baseline run folder:  {out_dir.resolve()}")
print(f"Section 8 expects:    {MYDRIVE_DATA / 'runs' / _run_name}")
print("Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)")
if _on_colab() and not str(out_dir.resolve()).startswith(str(MYDRIVE_DATA.resolve())):
    print(
        "WARNING: run dir is NOT under MyDrive/datasets_gnn2 — "
        "copy to Drive before disconnecting the VM or Section 8 will fail."
    )
else:
    print("Checkpoints are on Drive; safe to run Section 8 after this session ends.")
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SMOKE_SEED if SMOKE_TEST else 42,
)



## Fine-tune shipped DA-GPS checkpoint on no-BESS + with-DER (compatibility mvagg)

This cell **continues training** from your current inference checkpoint:

`da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE` (**h=96, L=2, heads=2**, reg CE, 4 meta-aux heads).

It uses the **compatibility** `gnn_node_features_and_targets_mvagg.csv` from the with-BESS generator (BESS P/Q folded into `p_load_kw` / `q_load_kvar`), so the feature schema stays **`p_load_kw,q_load_kvar,p_pv_kw` + PE** — same as inference.

### Google Drive layout (upload once)

```
MyDrive/datasets_gnn2/
├── checkpoints/
│   └── da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/   ← upload whole folder from PC
│       ├── training_last.pt
│       ├── x_mean.pt  x_std.pt  y_mean.pt  y_std.pt
│       ├── pv_mean.pt  pv_std.pt
│       ├── reg_class_values.pt  reg_class_tables.json
│       └── (optional) da_gps_multitask_best.pt
├── original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/
│   └── run_*/gnn_node_features_and_targets_mvagg.csv  ...
├── original_8500_unbalanced_chunked_with_bess_aug_2000_40/   ← from cell 3 above
│   └── run_*/gnn_node_features_and_targets_mvagg.csv  ...
├── cache/          ← tensor caches (created automatically)
└── runs/           ← new fine-tune output folders
```

**PC source for the checkpoint folder:**

`GNN2/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/`

Zip that folder → upload to Drive → unzip under `MyDrive/datasets_gnn2/checkpoints/`.

Colab also needs the **GNN2 code repo** cloned to `/content/GNN2` (or set `GNN2_REPO_ROOT`).

Run **cell 3** (with-BESS dataset generation) first, or copy finished chunks to the Drive path above.

In [ ]:
# --- Fine-tune shipped DA-GPS checkpoint on blended no-BESS + with-DER mvagg chunks ---
import os
import sys
import subprocess
import datetime
import warnings
import fnmatch
import re
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"

# ---------------- user toggles ----------------
# Full blend: 40 no-BESS + 4 with-DER chunks (44 total), 60 epochs
BLEND_NO_BESS = True            # recommended: keep old behavior + learn DER-augmented load
WITHDER_ONLY = False            # True = only with-DER chunks (faster, more forget risk)
FINETUNE_SMOKE = False          # True = 3 no-BESS + 2 with-DER chunks, 15 epochs
EPOCHS = None                   # None -> 60 (or set explicitly, e.g. 60)

NOBESS_CHUNK_PARENT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
WITHDER_CHUNK_PARENT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_with_bess_aug_2000_40"
BLENDED_CHUNK_PARENT = MYDRIVE_DATA / "chunk_parents/blended_nobess_withder"

INIT_RUN_DIR = MYDRIVE_DATA / "checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
INIT_CHECKPOINT = INIT_RUN_DIR / "training_last.pt"

# Must match shipped inference checkpoint (folder name "l4" is legacy)
MODEL_HIDDEN = 96
MODEL_LAYERS = 2
MODEL_HEADS = 2
MODEL_NODE_EMB_DIM = 4
N_SYSTEM_TOKENS = 10
META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"

FINETUNE_EPOCHS = int(EPOCHS) if EPOCHS is not None else (15 if FINETUNE_SMOKE else 60)
FINETUNE_PATIENCE = 5 if FINETUNE_SMOKE else 15
FINETUNE_LR = 1e-4
FINETUNE_SEED = 20420231
EVAL_EVERY = 10                 # train_pool_eval + subset pools every N epochs (epoch 1 + final always)
REFRESH_BLEND_SYMLINKS = False    # True = rebuild blended chunk_parent symlinks

SMOKE_NOBESS_CHUNKS = 3
SMOKE_WITHDER_CHUNKS = 2


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone to /content/GNN2 or /content/GNN-Sandia. "
        "Set GNN2_REPO_ROOT or run from repo folder."
    )


def _sorted_run_dirs(parent: Path) -> list[Path]:
    return sorted(
        (p for p in parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(f"No run_*/gnn_node_index_master.csv under {chunk_parent}")
    return hits[0]


def _chunk_glob_exact(names: list[str]) -> str:
    return ",".join(names)


def _ensure_blended_chunk_parent(
    blend_root: Path,
    *,
    nobess_parent: Path,
    withder_parent: Path,
    nobess_names: list[str] | None,
    withder_names: list[str] | None,
    refresh: bool = False,
) -> Path:
    blend_root.mkdir(parents=True, exist_ok=True)
    pairs: list[tuple[Path, Path]] = []
    for nm in nobess_names or []:
        pairs.append((nobess_parent / nm, blend_root / nm))
    for nm in withder_names or []:
        pairs.append((withder_parent / nm, blend_root / nm))

    for src, dst in pairs:
        if not src.is_dir():
            raise FileNotFoundError(f"Missing chunk folder: {src}")
        if dst.exists() or dst.is_symlink():
            if refresh:
                dst.unlink(missing_ok=True)
            else:
                continue
        dst.symlink_to(src.resolve(), target_is_directory=True)
    return blend_root


def _preflight_chunk(p: Path) -> None:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
        "gnn_node_index_master.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p}")


def _preflight_init_run(init_dir: Path, ckpt: Path) -> None:
    if not ckpt.is_file():
        raise FileNotFoundError(f"INIT_CHECKPOINT not found: {ckpt}")
    required = [
        "x_mean.pt",
        "x_std.pt",
        "y_mean.pt",
        "y_std.pt",
        "reg_class_values.pt",
        "reg_class_tables.json",
        "pv_mean.pt",
        "pv_std.pt",
    ]
    missing = [f for f in required if not (init_dir / f).is_file()]
    if missing:
        raise FileNotFoundError(
            f"INIT_RUN_DIR missing files {missing}. Upload the full checkpoint folder from PC:\n"
            "  gnn2_architecture_search/attention checkpoints/"
            "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/"
        )


if _on_colab() and not _drive_mounted():
    from google.colab import drive

    drive.mount("/content/drive")

REPO = _resolve_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

if not NOBESS_CHUNK_PARENT.is_dir():
    raise FileNotFoundError(f"NOBESS_CHUNK_PARENT not found: {NOBESS_CHUNK_PARENT}")
if not WITHDER_CHUNK_PARENT.is_dir():
    raise FileNotFoundError(f"WITHDER_CHUNK_PARENT not found: {WITHDER_CHUNK_PARENT}")

nobess_runs = _sorted_run_dirs(NOBESS_CHUNK_PARENT)
withder_runs = _sorted_run_dirs(WITHDER_CHUNK_PARENT)
if not withder_runs:
    raise RuntimeError(f"No run_* folders under {WITHDER_CHUNK_PARENT} — run cell 3 first.")

if WITHDER_ONLY:
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    selected_nobess: list[Path] = []
    chunk_parent = WITHDER_CHUNK_PARENT
    chunk_glob = _chunk_glob_exact([p.name for p in selected_withder])
elif BLEND_NO_BESS:
    selected_nobess = nobess_runs[:SMOKE_NOBESS_CHUNKS] if FINETUNE_SMOKE else nobess_runs
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    chunk_parent = _ensure_blended_chunk_parent(
        BLENDED_CHUNK_PARENT,
        nobess_parent=NOBESS_CHUNK_PARENT,
        withder_parent=WITHDER_CHUNK_PARENT,
        nobess_names=[p.name for p in selected_nobess],
        withder_names=[p.name for p in selected_withder],
        refresh=REFRESH_BLEND_SYMLINKS,
    )
    chunk_glob = _chunk_glob_exact([p.name for p in selected_nobess] + [p.name for p in selected_withder])
else:
    selected_withder = withder_runs[:SMOKE_WITHDER_CHUNKS] if FINETUNE_SMOKE else withder_runs
    selected_nobess = []
    chunk_parent = WITHDER_CHUNK_PARENT
    chunk_glob = _chunk_glob_exact([p.name for p in selected_withder])

_preflight_init_run(INIT_RUN_DIR, INIT_CHECKPOINT)
node_pe = _find_node_pe_csv(chunk_parent)

for ch in _sorted_run_dirs(chunk_parent):
    if ch.name in {s.strip() for s in chunk_glob.split(",")}:
        _preflight_chunk(ch)

# Reuses existing .pt caches from prior runs when filenames match (incl. __dropunseen__maux suffix)
cache_root = MYDRIVE_DATA / "cache/da_gps_finetune_nobess_withder_gine"
gnn_cache_root = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
runs_parent = MYDRIVE_DATA / "runs"
cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = runs_parent / (
    f"da_gps_finetune_withder_l{MODEL_LAYERS}_h{MODEL_HIDDEN}_regce"
    f"{'_smoke' if FINETUNE_SMOKE else ''}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

print("=== Fine-tune preflight ===")
print(f"REPO:              {REPO}")
print(f"INIT_RUN_DIR:      {INIT_RUN_DIR}")
print(f"INIT_CHECKPOINT:   {INIT_CHECKPOINT}")
print(f"CHUNK_PARENT:      {chunk_parent}")
print(f"CHUNK_GLOB:        {chunk_glob}")
print(f"no-BESS chunks:    {len(selected_nobess)}")
print(f"with-DER chunks:   {len(selected_withder)}")
print(f"NODE_PE_CSV:       {node_pe}")
print(f"OUT_DIR:           {out_dir}")
holdout_cmd_extra: list[str] = []
subset_cmd_extra: list[str] = []
if WITHDER_ONLY:
    holdout_chunks = nobess_runs[:SMOKE_NOBESS_CHUNKS] if FINETUNE_SMOKE else nobess_runs
    holdout_cmd_extra = [
        "--eval_holdout_chunk_parent",
        str(NOBESS_CHUNK_PARENT),
        "--eval_holdout_chunk_glob",
        _chunk_glob_exact([p.name for p in holdout_chunks]),
        "--eval_holdout_label",
        "nobess_holdout",
        "--eval_holdout_seed",
        str(FINETUNE_SEED),
    ]
    holdout_label = f"{len(holdout_chunks)} no-BESS chunk(s) -> nobess_holdout"
elif BLEND_NO_BESS:
    holdout_label = "skipped (no-BESS already in training blend)"
    subset_cmd_extra = [
        "--eval_subset_nobess_chunk_parent",
        str(NOBESS_CHUNK_PARENT),
        "--eval_subset_nobess_chunk_glob",
        _chunk_glob_exact([p.name for p in selected_nobess]),
        "--eval_subset_nobess_label",
        "nobess_40",
        "--eval_subset_withder_chunk_parent",
        str(WITHDER_CHUNK_PARENT),
        "--eval_subset_withder_chunk_glob",
        _chunk_glob_exact([p.name for p in selected_withder]),
        "--eval_subset_withder_label",
        "withder_4",
    ]
    subset_label = (
        f"train-pool subsets: {len(selected_nobess)} nobess_40 + "
        f"{len(selected_withder)} withder_4"
    )
else:
    holdout_label = "none (with-DER only, no holdout parent)"

print(f"EPOCHS/LR:         {FINETUNE_EPOCHS} / {FINETUNE_LR}")
print(f"EVAL_EVERY:        {EVAL_EVERY} (init eval still runs with --eval_before_train)")
print(f"HOLDOUT:           {holdout_label}")
if BLEND_NO_BESS and subset_cmd_extra:
    print(f"SUBSET_EVAL:       {subset_label}")
elif WITHDER_ONLY:
    print(f"HOLDOUT_PARENT:    {NOBESS_CHUNK_PARENT}")
print("=======================\n")

cmd = [
    sys.executable,
    "-u",
    "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent",
    str(chunk_parent),
    "--chunk_subdir_glob",
    chunk_glob,
    "--nodes_csv",
    "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv",
    "gnn_edges_phase_static.csv",
    "--meta_csv",
    "gnn_sample_meta.csv",
    "--node_feature_cols",
    "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv",
    str(node_pe),
    "--node_pe_cols",
    "auto",
    "--n_system_tokens",
    str(N_SYSTEM_TOKENS),
    "--aux_meta_cols",
    META_AUX_COLS,
    "--lambda_pv",
    "0.1",
    "--out_dir",
    str(out_dir),
    "--cache_dir",
    str(cache_root),
    "--bootstrap_gnn_cache_dir",
    str(gnn_cache_root),
    "--init_checkpoint",
    str(INIT_CHECKPOINT),
    "--init_run_dir",
    str(INIT_RUN_DIR),
    "--drop_samples_unseen_reg_taps",
    "--eval_before_train",
    "--epochs",
    str(FINETUNE_EPOCHS),
    "--batch_size",
    "96",
    "--hidden",
    str(MODEL_HIDDEN),
    "--layers",
    str(MODEL_LAYERS),
    "--heads",
    str(MODEL_HEADS),
    "--node_emb_dim",
    str(MODEL_NODE_EMB_DIM),
    "--edge_emb_dim",
    "0",
    "--lr",
    str(FINETUNE_LR),
    "--weight_decay",
    "1e-5",
    "--lambda_cap",
    "0.1",
    "--lambda_reg",
    "0.1",
    "--reg_loss",
    "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience",
    str(FINETUNE_PATIENCE),
    "--seed",
    str(FINETUNE_SEED),
    "--train_frac",
    "0.80",
    "--val_frac",
    "0.10",
    "--sample_frac",
    "1.0",
    "--num_workers",
    "0" if os.name == "nt" else "4",
    "--log_every",
    "0",
    "--eval_every",
    str(EVAL_EVERY),
    "--checkpoint_every",
    "10",
    "--early_stop_on",
    "total",
    "--dropout",
    "0.1",
] + holdout_cmd_extra + subset_cmd_extra



_TRAIN_NOISE_RE = re.compile(
    r"FutureWarning:\s*`torch\.cuda\.amp\.|"
    r"^\s+with \(torch\.cuda\.amp\.autocast|"
    r"^\s+scaler = _GradScaler\(\)"
)


def _should_print_train_line(line: str, *, in_eval_block: bool) -> tuple[bool, bool]:
    """Quiet filter for DA-GPS fine-tune subprocess logs."""
    s = line.rstrip("\n")
    stripped = s.strip()
    if not stripped:
        return False, False

    if _TRAIN_NOISE_RE.search(s):
        return False, in_eval_block

    if "Traceback (most recent call last)" in s:
        return True, False
    if s.startswith('  File "'):
        return True, in_eval_block
    if any(
        tok in s
        for tok in (
            "Error:",
            "Exception:",
            "CalledProcessError",
            "RuntimeError:",
            "FileNotFoundError:",
            "KeyboardInterrupt",
        )
    ):
        return True, in_eval_block

    if "[da_gps chunk_parent] epoch" in s:
        return True, False
    if s.startswith("[da_gps chunk_parent]") and any(
        tok in s
        for tok in ("cap_BCE", "reg_CE", "reg_MAE", "reg_MSE", "meta_aux_MSE", " addons] epoch ")
    ):
        return True, False
    if s.startswith("[da_gps addons counterfactual]"):
        return True, False

    if s.startswith("==="):
        if any(
            tok in s
            for tok in (
                "train_pool_eval",
                "nobess_40",
                "withder_4",
                "nobess_holdout",
                "Before / after",
                "Baseline before",
            )
        ):
            return True, True
        return False, False

    if in_eval_block and (
        s.startswith("Val ")
        or s.startswith("Test ")
        or s.startswith("split ")
        or re.match(r"^(val|test)\s+\S", s, re.I)
        or "[vs baseline]" in s
    ):
        return True, True

    if "[vs baseline]" in s:
        return True, False
    if "[init eval]" in s:
        return True, False

    if any(
        tok in s
        for tok in (
            "Wrote chunk tensor cache:",
            "Bootstrapped DA cache",
            "chunk cache: reusing",
            "Added meta-aux columns to chunk cache:",
        )
    ):
        return True, False

    if any(
        tok in s
        for tok in (
            "Loaded init_checkpoint=",
            "AMP (autocast + GradScaler): enabled",
            "[da_gps chunk_parent] early stop",
            "periodic checkpoint ->",
        )
    ):
        return True, False
    if s.startswith("Saved ") and s.rstrip().endswith(".pt"):
        return True, False

    return False, False


def _stream_filtered_train_output(proc_stdout) -> None:
    in_eval_block = False
    for raw in proc_stdout:
        keep, in_eval_block = _should_print_train_line(raw, in_eval_block=in_eval_block)
        if keep:
            print(raw, end="")

print("Running:\n ", " ".join(cmd), "\n", flush=True)
_train_env = os.environ.copy()
_train_env.setdefault("PYTHONWARNINGS", "ignore::FutureWarning")
warnings.filterwarnings("ignore", category=FutureWarning)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    env=_train_env,
) as proc:
    assert proc.stdout is not None
    _stream_filtered_train_output(proc.stdout)
    rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nFine-tune completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
print("\n=== Inference / warm-start ===")
print(f"RUN_DIR     = {out_dir.resolve()}")
print(f"CHECKPOINT  = {(out_dir / 'training_last.pt').resolve()}")
print("Point cell 9 bootstrap or daily-compare at the paths above after training.")


train on 4 chunks

=== Fine-tune preflight ===
REPO:              /content/GNN-Sandia
INIT_RUN_DIR:      /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
INIT_CHECKPOINT:   /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
CHUNK_PARENT:      /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40
CHUNK_GLOB:        run_001_scen_0000_0049_seed_20520233,run_002_scen_0050_0099_seed_20620236,run_003_scen_0100_0149_seed_20720239,run_004_scen_0150_0199_seed_20820242
no-BESS chunks:    0
with-DER chunks:   4
NODE_PE_CSV:       /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv
OUT_DIR:           /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739
EPOCHS/LR:         60 / 0.0001
HOLDOUT:           40 no-BESS chunk(s) -> nobess_holdout
HOLDOUT_PARENT:    /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
=======================

Running:
  /usr/bin/python3 -u train_da_gps_multitask_complex_voltage_gine.py --chunk_parent /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40 --chunk_subdir_glob run_001_scen_0000_0049_seed_20520233,run_002_scen_0050_0099_seed_20620236,run_003_scen_0100_0149_seed_20720239,run_004_scen_0150_0199_seed_20820242 --nodes_csv gnn_node_features_and_targets_mvagg.csv --edge_catalog_csv gnn_edges_phase_static.csv --meta_csv gnn_sample_meta.csv --node_feature_cols p_load_kw,q_load_kvar,p_pv_kw --exclude_bess_features --node_pe_csv /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv --node_pe_cols auto --n_system_tokens 10 --aux_meta_cols pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar --lambda_pv 0.1 --out_dir /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739 --cache_dir /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine --bootstrap_gnn_cache_dir /content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine --init_checkpoint /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt --init_run_dir /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE --drop_samples_unseen_reg_taps --eval_before_train --epochs 60 --batch_size 96 --hidden 96 --layers 2 --heads 2 --node_emb_dim 4 --edge_emb_dim 0 --lr 0.0001 --weight_decay 1e-5 --lambda_cap 0.1 --lambda_reg 0.1 --reg_loss ce --per_device_cap_head --per_device_reg_head --patience 15 --seed 20420231 --train_frac 0.80 --val_frac 0.10 --sample_frac 1.0 --num_workers 4 --log_every 10 --checkpoint_every 10 --early_stop_on total --dropout 0.1 --eval_holdout_chunk_parent /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40 --eval_holdout_chunk_glob run_001_scen_0000_0049_seed_20420233,run_002_scen_0050_0099_seed_20520236,run_003_scen_0100_0149_seed_20620239,run_004_scen_0150_0199_seed_20720242,run_005_scen_0200_0249_seed_20820245,run_006_scen_0250_0299_seed_20920248,run_007_scen_0300_0349_seed_21020251,run_008_scen_0350_0399_seed_21120254,run_009_scen_0400_0449_seed_21220257,run_010_scen_0450_0499_seed_21320260,run_011_scen_0500_0549_seed_21420263,run_012_scen_0550_0599_seed_21520266,run_013_scen_0600_0649_seed_21620269,run_014_scen_0650_0699_seed_21720272,run_015_scen_0700_0749_seed_21820275,run_016_scen_0750_0799_seed_21920278,run_017_scen_0800_0849_seed_22020281,run_018_scen_0850_0899_seed_22120284,run_019_scen_0900_0949_seed_22220287,run_020_scen_0950_0999_seed_22320290,run_021_scen_1000_1049_seed_22420293,run_022_scen_1050_1099_seed_22520296,run_023_scen_1100_1149_seed_22620299,run_024_scen_1150_1199_seed_22720302,run_025_scen_1200_1249_seed_22820305,run_026_scen_1250_1299_seed_22920308,run_027_scen_1300_1349_seed_23020311,run_028_scen_1350_1399_seed_23120314,run_029_scen_1400_1449_seed_23220317,run_030_scen_1450_1499_seed_23320320,run_031_scen_1500_1549_seed_23420323,run_032_scen_1550_1599_seed_23520326,run_033_scen_1600_1649_seed_23620329,run_034_scen_1650_1699_seed_23720332,run_035_scen_1700_1749_seed_23820335,run_036_scen_1750_1799_seed_23920338,run_037_scen_1800_1849_seed_24020341,run_038_scen_1850_1899_seed_24120344,run_039_scen_1900_1949_seed_24220347,run_040_scen_1950_1999_seed_24320350 --eval_holdout_label nobess_holdout --eval_holdout_seed 20420231 

init_checkpoint: /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
init_run_dir: /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
drop_samples_unseen_reg_taps: enabled (exclude samples with tap_pu outside init CE classes)
regulator tap training loss: ce (discrete tap classes + cross-entropy)
exclude_bess_features: using node_feature_cols= ['p_load_kw', 'q_load_kvar', 'p_pv_kw']
chunk_parent cache override via --cache_dir: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine
bootstrap GNN cache dir: /content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine
Meta aux (sample_meta): 4 column(s); chunk DA caches use suffix __mauxb7bd1d58 (per chunk name).
  global token index 22 (system slot 0): column 'pv_pv2_p_post_kw'
  global token index 23 (system slot 1): column 'pv_pv2_q_post_kvar'
  global token index 24 (system slot 2): column 'p_loss_total_post_kw'
  global token index 25 (system slot 3): column 'q_loss_total_post_kvar'
[chunk_parent] 4 chunks under /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40
  - run_001_scen_0000_0049_seed_20520233
  - run_002_scen_0050_0099_seed_20620236
  - run_003_scen_0100_0149_seed_20720239
  - run_004_scen_0150_0199_seed_20820242
reg_class_tables: loaded from init run /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/reg_class_tables.json
  reg_feeder_rega_tap_pu: n_classes=19
  reg_feeder_regb_tap_pu: n_classes=19
  reg_feeder_regc_tap_pu: n_classes=19
  reg_vreg2_a_tap_pu: n_classes=19
  reg_vreg2_b_tap_pu: n_classes=22
  reg_vreg2_c_tap_pu: n_classes=29
  reg_vreg3_a_tap_pu: n_classes=21
  reg_vreg3_b_tap_pu: n_classes=22
  reg_vreg3_c_tap_pu: n_classes=24
  reg_vreg4_a_tap_pu: n_classes=21
  reg_vreg4_b_tap_pu: n_classes=22
  reg_vreg4_c_tap_pu: n_classes=21
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_edges_phase_static.csv
  directed edges: 15256
[eval_holdout:nobess_holdout] 40 chunks under /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40
  - run_001_scen_0000_0049_seed_20420233
  - run_002_scen_0050_0099_seed_20520236
  - run_003_scen_0100_0149_seed_20620239
  - run_004_scen_0150_0199_seed_20720242
  - run_005_scen_0200_0249_seed_20820245
  - run_006_scen_0250_0299_seed_20920248
  - run_007_scen_0300_0349_seed_21020251
  - run_008_scen_0350_0399_seed_21120254
  - run_009_scen_0400_0449_seed_21220257
  - run_010_scen_0450_0499_seed_21320260
  - run_011_scen_0500_0549_seed_21420263
  - run_012_scen_0550_0599_seed_21520266
  - run_013_scen_0600_0649_seed_21620269
  - run_014_scen_0650_0699_seed_21720272
  - run_015_scen_0700_0749_seed_21820275
  - run_016_scen_0750_0799_seed_21920278
  - run_017_scen_0800_0849_seed_22020281
  - run_018_scen_0850_0899_seed_22120284
  - run_019_scen_0900_0949_seed_22220287
  - run_020_scen_0950_0999_seed_22320290
  - run_021_scen_1000_1049_seed_22420293
  - run_022_scen_1050_1099_seed_22520296
  - run_023_scen_1100_1149_seed_22620299
  - run_024_scen_1150_1199_seed_22720302
  - run_025_scen_1200_1249_seed_22820305
  - run_026_scen_1250_1299_seed_22920308
  - run_027_scen_1300_1349_seed_23020311
  - run_028_scen_1350_1399_seed_23120314
  - run_029_scen_1400_1449_seed_23220317
  - run_030_scen_1450_1499_seed_23320320
  - run_031_scen_1500_1549_seed_23420323
  - run_032_scen_1550_1599_seed_23520326
  - run_033_scen_1600_1649_seed_23620329
  - run_034_scen_1650_1699_seed_23720332
  - run_035_scen_1700_1749_seed_23820335
  - run_036_scen_1750_1799_seed_23920338
  - run_037_scen_1800_1849_seed_24020341
  - run_038_scen_1850_1899_seed_24120344
  - run_039_scen_1900_1949_seed_24220347
  - run_040_scen_1950_1999_seed_24320350
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_001_scen_0000_0049_seed_20420233__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_002_scen_0050_0099_seed_20520236/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_002_scen_0050_0099_seed_20520236__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_003_scen_0100_0149_seed_20620239/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_003_scen_0100_0149_seed_20620239__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_004_scen_0150_0199_seed_20720242/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_004_scen_0150_0199_seed_20720242__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_005_scen_0200_0249_seed_20820245/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_005_scen_0200_0249_seed_20820245__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_006_scen_0250_0299_seed_20920248/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_006_scen_0250_0299_seed_20920248__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_007_scen_0300_0349_seed_21020251/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_007_scen_0300_0349_seed_21020251__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_008_scen_0350_0399_seed_21120254/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_008_scen_0350_0399_seed_21120254__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_009_scen_0400_0449_seed_21220257/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_009_scen_0400_0449_seed_21220257__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_010_scen_0450_0499_seed_21320260/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_010_scen_0450_0499_seed_21320260__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_011_scen_0500_0549_seed_21420263/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_011_scen_0500_0549_seed_21420263__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_012_scen_0550_0599_seed_21520266/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_012_scen_0550_0599_seed_21520266__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_013_scen_0600_0649_seed_21620269/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_013_scen_0600_0649_seed_21620269__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_014_scen_0650_0699_seed_21720272/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_014_scen_0650_0699_seed_21720272__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_015_scen_0700_0749_seed_21820275/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_015_scen_0700_0749_seed_21820275__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_016_scen_0750_0799_seed_21920278/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_016_scen_0750_0799_seed_21920278__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_017_scen_0800_0849_seed_22020281/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_017_scen_0800_0849_seed_22020281__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_018_scen_0850_0899_seed_22120284/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_018_scen_0850_0899_seed_22120284__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_019_scen_0900_0949_seed_22220287/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_019_scen_0900_0949_seed_22220287__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_020_scen_0950_0999_seed_22320290/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_020_scen_0950_0999_seed_22320290__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_021_scen_1000_1049_seed_22420293/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_021_scen_1000_1049_seed_22420293__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_022_scen_1050_1099_seed_22520296/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_022_scen_1050_1099_seed_22520296__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_023_scen_1100_1149_seed_22620299/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_023_scen_1100_1149_seed_22620299__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_024_scen_1150_1199_seed_22720302/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_024_scen_1150_1199_seed_22720302__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_025_scen_1200_1249_seed_22820305/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_025_scen_1200_1249_seed_22820305__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_026_scen_1250_1299_seed_22920308/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_026_scen_1250_1299_seed_22920308__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_027_scen_1300_1349_seed_23020311/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_027_scen_1300_1349_seed_23020311__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_028_scen_1350_1399_seed_23120314/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_028_scen_1350_1399_seed_23120314__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_029_scen_1400_1449_seed_23220317/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_029_scen_1400_1449_seed_23220317__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_030_scen_1450_1499_seed_23320320/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_030_scen_1450_1499_seed_23320320__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_031_scen_1500_1549_seed_23420323/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_031_scen_1500_1549_seed_23420323__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_032_scen_1550_1599_seed_23520326/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_032_scen_1550_1599_seed_23520326__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_033_scen_1600_1649_seed_23620329/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_033_scen_1600_1649_seed_23620329__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_034_scen_1650_1699_seed_23720332/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_034_scen_1650_1699_seed_23720332__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_035_scen_1700_1749_seed_23820335/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_035_scen_1700_1749_seed_23820335__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_036_scen_1750_1799_seed_23920338/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_036_scen_1750_1799_seed_23920338__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_037_scen_1800_1849_seed_24020341/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_037_scen_1800_1849_seed_24020341__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_038_scen_1850_1899_seed_24120344/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_038_scen_1850_1899_seed_24120344__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_039_scen_1900_1949_seed_24220347/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_039_scen_1900_1949_seed_24220347__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
Loading nodes: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_040_scen_1950_1999_seed_24320350/gnn_node_features_and_targets_mvagg.csv
Using PE from /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_with_bess_aug_2000_40/run_001_scen_0000_0049_seed_20520233/gnn_node_index_master.csv with columns: ['pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8']
Wrote chunk tensor cache: /content/drive/MyDrive/datasets_gnn2/cache/da_gps_finetune_nobess_withder_gine/run_040_scen_1950_1999_seed_24320350__full__nobess__regce__rcbad51b871f__mauxb7bd1d58.pt
norm stats: loaded from init_run_dir /content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
Wrote run manifest (for daily compare / mid-train snapshots): /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/da_gps_run_manifest.json
torch.compile: enabled
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:6902: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = _GradScaler()
AMP (autocast + GradScaler): enabled
Loaded init_checkpoint=/content/drive/MyDrive/datasets_gnn2/checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt strict=False missing=0 unexpected=0
[init eval] train_pool_eval (training chunks)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5463: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (eval_before_train) ===
Val  |V| MAE=0.017585  angle MAE=3.189312  Re/Im MSE(nrm)=0.503004  r2_mean=0.5391  r2_min=-0.2126  worst_mae=0.071793  tot=1.2081  volt=0.5030
Test  |V| MAE=0.017406  angle MAE=3.051563  Re/Im MSE(nrm)=0.496211  r2_mean=0.5532  r2_min=-0.1969  worst_mae=0.071131  tot=1.1707  volt=0.4962
[init eval] holdout (nobess_holdout)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5463: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_holdout (eval_before_train) ===
Val  |V| MAE=0.003825  angle MAE=0.193789  Re/Im MSE(nrm)=0.010318  r2_mean=0.9303  r2_min=0.1156  worst_mae=0.013232  tot=0.0994  volt=0.0103
Test  |V| MAE=0.003861  angle MAE=0.194140  Re/Im MSE(nrm)=0.010475  r2_mean=0.9266  r2_min=0.0981  worst_mae=0.013365  tot=0.0994  volt=0.0105
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7355: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch    1/60 | train_tot=0.9670 train_volt=0.4738 train_cap=0.6181 train_reg=3.9513 train_meta_aux=0.3629 val_meta_aux=0.3594 | val_tot=0.8645 val_volt=0.4608 val_cap=0.4446 val_reg=3.2328 | val_r2_mean=0.5904 val_r2_min=-0.2811 val_r2_min_node=_hvmv_sub_lsb.1 val_worst_mae=0.0697 | best=0.8645
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.4310  cap_capbank0b_n_steps_on=0.5134  cap_capbank0c_n_steps_on=0.7439  cap_capbank1a_n_steps_on=0.9195  cap_capbank1b_n_steps_on=0.6821  cap_capbank1c_n_steps_on=0.9167
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.6283  cap_capbank2b_n_steps_on=0.6046  cap_capbank2c_n_steps_on=0.7420  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.3642  cap_capbank0b_n_steps_on=0.4017  cap_capbank0c_n_steps_on=0.6116  cap_capbank1a_n_steps_on=0.6455  cap_capbank1b_n_steps_on=0.4528  cap_capbank1c_n_steps_on=0.6260
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.4419  cap_capbank2b_n_steps_on=0.3996  cap_capbank2c_n_steps_on=0.5021  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=3.3657  reg_feeder_regb_tap_pu=3.3479  reg_feeder_regc_tap_pu=2.8018  reg_vreg2_a_tap_pu=3.6676  reg_vreg2_b_tap_pu=3.4801  reg_vreg2_c_tap_pu=3.3935
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=4.3588  reg_vreg3_b_tap_pu=4.7268  reg_vreg3_c_tap_pu=4.5415  reg_vreg4_a_tap_pu=4.8771  reg_vreg4_b_tap_pu=4.3468  reg_vreg4_c_tap_pu=4.5075
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=2.8561  reg_feeder_regb_tap_pu=2.5966  reg_feeder_regc_tap_pu=2.1895  reg_vreg2_a_tap_pu=3.4281  reg_vreg2_b_tap_pu=2.9495  reg_vreg2_c_tap_pu=3.0856
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=3.4826  reg_vreg3_b_tap_pu=3.3978  reg_vreg3_c_tap_pu=3.8912  reg_vreg4_a_tap_pu=4.0590  reg_vreg4_b_tap_pu=3.1511  reg_vreg4_c_tap_pu=3.7044
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0231  pv_pv2_q_post_kvar=1.0804  p_loss_total_post_kw=0.1427  q_loss_total_post_kvar=0.2053
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0234  pv_pv2_q_post_kvar=1.0897  p_loss_total_post_kw=0.1327  q_loss_total_post_kvar=0.1919

=== train_pool_eval (epoch 1) ===
Val  |V| MAE=0.017867  angle MAE=3.066364  Re/Im MSE(nrm)=0.460820  r2_mean=0.5736  r2_min=-0.3007  worst_mae=0.069728  tot=0.8645  volt=0.4608
Test  |V| MAE=0.018106  angle MAE=2.932192  Re/Im MSE(nrm)=0.456620  r2_mean=0.5778  r2_min=-0.2018  worst_mae=0.069093  tot=0.8545  volt=0.4566
[vs baseline] epoch 1 val |V| MAE: 0.017585 -> 0.017867 (Δ +0.00028, worse)
[vs baseline] epoch 1 val angle MAE: 3.189312 -> 3.066364 (Δ -0.12295, improved)
[vs baseline] epoch 1 val Re/Im MSE(nrm): 0.503004 -> 0.460820 (Δ -0.04218, improved)
[vs baseline] epoch 1 val r2_mean: 0.5391 -> 0.5736 (Δ +0.0345, improved)
[vs baseline] epoch 1 val r2_min: -0.2126 -> -0.3007 (Δ -0.0881, worse)
[vs baseline] epoch 1 val worst_mae: 0.071793 -> 0.069728 (Δ -0.00206, improved)
[vs baseline] epoch 1 val tot: 1.2081 -> 0.8645 (Δ -0.3437, improved)
[vs baseline] epoch 1 val volt: 0.5030 -> 0.4608 (Δ -0.0422, improved)
[vs baseline] epoch 1 test |V| MAE: 0.017406 -> 0.018106 (Δ +0.00070, worse)
[vs baseline] epoch 1 test angle MAE: 3.051563 -> 2.932192 (Δ -0.11937, improved)
[vs baseline] epoch 1 test Re/Im MSE(nrm): 0.496211 -> 0.456620 (Δ -0.03959, improved)
[vs baseline] epoch 1 test r2_mean: 0.5532 -> 0.5778 (Δ +0.0246, improved)
[vs baseline] epoch 1 test r2_min: -0.1969 -> -0.2018 (Δ -0.0049, worse)
[vs baseline] epoch 1 test worst_mae: 0.071131 -> 0.069093 (Δ -0.00204, improved)
[vs baseline] epoch 1 test tot: 1.1707 -> 0.8545 (Δ -0.3163, improved)
[vs baseline] epoch 1 test volt: 0.4962 -> 0.4566 (Δ -0.0396, improved)

=== nobess_holdout (epoch 1) ===
Val  |V| MAE=0.007971  angle MAE=0.639546  Re/Im MSE(nrm)=0.043900  r2_mean=0.8810  r2_min=-0.1765  worst_mae=0.029629  tot=0.2027  volt=0.0439
Test  |V| MAE=0.008138  angle MAE=0.650158  Re/Im MSE(nrm)=0.045039  r2_mean=0.8743  r2_min=-0.1615  worst_mae=0.029812  tot=0.2024  volt=0.0450
[da_gps chunk_parent] epoch   10/60 | train_tot=0.4813 train_volt=0.2209 train_cap=0.3205 train_reg=2.0433 train_meta_aux=0.2406 val_meta_aux=0.2423 | val_tot=0.4790 val_volt=0.2198 val_cap=0.3090 val_reg=2.0405 | val_r2_mean=0.7132 val_r2_min=0.0006 val_r2_min_node=_hvmv_sub_lsb.2 val_worst_mae=0.0661 | best=0.4790
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.2148  cap_capbank0b_n_steps_on=0.2553  cap_capbank0c_n_steps_on=0.3728  cap_capbank1a_n_steps_on=0.3598  cap_capbank1b_n_steps_on=0.3203  cap_capbank1c_n_steps_on=0.4932
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3675  cap_capbank2b_n_steps_on=0.3534  cap_capbank2c_n_steps_on=0.4680  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.2067  cap_capbank0b_n_steps_on=0.2518  cap_capbank0c_n_steps_on=0.3650  cap_capbank1a_n_steps_on=0.3301  cap_capbank1b_n_steps_on=0.2933  cap_capbank1c_n_steps_on=0.4621
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.3891  cap_capbank2b_n_steps_on=0.3555  cap_capbank2c_n_steps_on=0.4366  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.7435  reg_feeder_regb_tap_pu=1.6959  reg_feeder_regc_tap_pu=1.4892  reg_vreg2_a_tap_pu=1.8971  reg_vreg2_b_tap_pu=2.0599  reg_vreg2_c_tap_pu=2.2920
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.9309  reg_vreg3_b_tap_pu=2.1383  reg_vreg3_c_tap_pu=2.5356  reg_vreg4_a_tap_pu=2.2276  reg_vreg4_b_tap_pu=2.0230  reg_vreg4_c_tap_pu=2.4871
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.7054  reg_feeder_regb_tap_pu=1.6695  reg_feeder_regc_tap_pu=1.4916  reg_vreg2_a_tap_pu=1.9228  reg_vreg2_b_tap_pu=2.1863  reg_vreg2_c_tap_pu=2.2255
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.8473  reg_vreg3_b_tap_pu=2.2966  reg_vreg3_c_tap_pu=2.4911  reg_vreg4_a_tap_pu=2.1690  reg_vreg4_b_tap_pu=1.9955  reg_vreg4_c_tap_pu=2.4849
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0301  pv_pv2_q_post_kvar=0.7589  p_loss_total_post_kw=0.0797  q_loss_total_post_kvar=0.0936
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0238  pv_pv2_q_post_kvar=0.7135  p_loss_total_post_kw=0.1087  q_loss_total_post_kvar=0.1231

=== train_pool_eval (epoch 10) ===
Val  |V| MAE=0.014274  angle MAE=1.784704  Re/Im MSE(nrm)=0.219828  r2_mean=0.6988  r2_min=-0.2345  worst_mae=0.066095  tot=0.4790  volt=0.2198
Test  |V| MAE=0.014310  angle MAE=1.718762  Re/Im MSE(nrm)=0.218512  r2_mean=0.6981  r2_min=-0.0836  worst_mae=0.064776  tot=0.4768  volt=0.2185
[vs baseline] epoch 10 val |V| MAE: 0.017585 -> 0.014274 (Δ -0.00331, improved)
[vs baseline] epoch 10 val angle MAE: 3.189312 -> 1.784704 (Δ -1.40461, improved)
[vs baseline] epoch 10 val Re/Im MSE(nrm): 0.503004 -> 0.219828 (Δ -0.28318, improved)
[vs baseline] epoch 10 val r2_mean: 0.5391 -> 0.6988 (Δ +0.1597, improved)
[vs baseline] epoch 10 val r2_min: -0.2126 -> -0.2345 (Δ -0.0219, worse)
[vs baseline] epoch 10 val worst_mae: 0.071793 -> 0.066095 (Δ -0.00570, improved)
[vs baseline] epoch 10 val tot: 1.2081 -> 0.4790 (Δ -0.7291, improved)
[vs baseline] epoch 10 val volt: 0.5030 -> 0.2198 (Δ -0.2832, improved)
[vs baseline] epoch 10 test |V| MAE: 0.017406 -> 0.014310 (Δ -0.00310, improved)
[vs baseline] epoch 10 test angle MAE: 3.051563 -> 1.718762 (Δ -1.33280, improved)
[vs baseline] epoch 10 test Re/Im MSE(nrm): 0.496211 -> 0.218512 (Δ -0.27770, improved)
[vs baseline] epoch 10 test r2_mean: 0.5532 -> 0.6981 (Δ +0.1449, improved)
[vs baseline] epoch 10 test r2_min: -0.1969 -> -0.0836 (Δ +0.1133, improved)
[vs baseline] epoch 10 test worst_mae: 0.071131 -> 0.064776 (Δ -0.00636, improved)
[vs baseline] epoch 10 test tot: 1.1707 -> 0.4768 (Δ -0.6939, improved)
[vs baseline] epoch 10 test volt: 0.4962 -> 0.2185 (Δ -0.2777, improved)

=== nobess_holdout (epoch 10) ===
Val  |V| MAE=0.017252  angle MAE=3.469810  Re/Im MSE(nrm)=0.421420  r2_mean=0.3382  r2_min=-6.5356  worst_mae=0.055422  tot=0.6380  volt=0.4214
Test  |V| MAE=0.017469  angle MAE=3.451343  Re/Im MSE(nrm)=0.419208  r2_mean=0.3474  r2_min=-6.4137  worst_mae=0.056103  tot=0.6341  volt=0.4192
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
[da_gps chunk_parent] epoch   20/60 | train_tot=0.3539 train_volt=0.1232 train_cap=0.2982 train_reg=1.8757 train_meta_aux=0.1324 val_meta_aux=0.1334 | val_tot=0.3392 val_volt=0.1105 val_cap=0.2872 val_reg=1.8658 | val_r2_mean=0.7801 val_r2_min=0.0424 val_r2_min_node=d5710794-3_int.2 val_worst_mae=0.0602 | best=0.3392
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.1752  cap_capbank0b_n_steps_on=0.2235  cap_capbank0c_n_steps_on=0.3632  cap_capbank1a_n_steps_on=0.3185  cap_capbank1b_n_steps_on=0.3070  cap_capbank1c_n_steps_on=0.4658
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3416  cap_capbank2b_n_steps_on=0.3424  cap_capbank2c_n_steps_on=0.4450  cap_capbank3_n_steps_on=0.0001
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.1493  cap_capbank0b_n_steps_on=0.1954  cap_capbank0c_n_steps_on=0.3541  cap_capbank1a_n_steps_on=0.3122  cap_capbank1b_n_steps_on=0.2941  cap_capbank1c_n_steps_on=0.4393
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.3531  cap_capbank2b_n_steps_on=0.3476  cap_capbank2c_n_steps_on=0.4266  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.5393  reg_feeder_regb_tap_pu=1.5447  reg_feeder_regc_tap_pu=1.3783  reg_vreg2_a_tap_pu=1.7353  reg_vreg2_b_tap_pu=1.9357  reg_vreg2_c_tap_pu=2.2073
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.6825  reg_vreg3_b_tap_pu=1.9329  reg_vreg3_c_tap_pu=2.3982  reg_vreg4_a_tap_pu=1.9877  reg_vreg4_b_tap_pu=1.8468  reg_vreg4_c_tap_pu=2.3194
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.5024  reg_feeder_regb_tap_pu=1.4964  reg_feeder_regc_tap_pu=1.3305  reg_vreg2_a_tap_pu=1.7493  reg_vreg2_b_tap_pu=2.0645  reg_vreg2_c_tap_pu=2.1720
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.5623  reg_vreg3_b_tap_pu=2.0526  reg_vreg3_c_tap_pu=2.3807  reg_vreg4_a_tap_pu=1.9380  reg_vreg4_b_tap_pu=1.8181  reg_vreg4_c_tap_pu=2.3235
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0254  pv_pv2_q_post_kvar=0.3939  p_loss_total_post_kw=0.0533  q_loss_total_post_kvar=0.0571
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0193  pv_pv2_q_post_kvar=0.4172  p_loss_total_post_kw=0.0482  q_loss_total_post_kvar=0.0490

=== train_pool_eval (epoch 20) ===
Val  |V| MAE=0.011787  angle MAE=0.975983  Re/Im MSE(nrm)=0.110521  r2_mean=0.7670  r2_min=-0.1630  worst_mae=0.060158  tot=0.3392  volt=0.1105
Test  |V| MAE=0.011794  angle MAE=0.980144  Re/Im MSE(nrm)=0.114133  r2_mean=0.7684  r2_min=-0.0643  worst_mae=0.059769  tot=0.3442  volt=0.1141
[vs baseline] epoch 20 val |V| MAE: 0.017585 -> 0.011787 (Δ -0.00580, improved)
[vs baseline] epoch 20 val angle MAE: 3.189312 -> 0.975983 (Δ -2.21333, improved)
[vs baseline] epoch 20 val Re/Im MSE(nrm): 0.503004 -> 0.110521 (Δ -0.39248, improved)
[vs baseline] epoch 20 val r2_mean: 0.5391 -> 0.7670 (Δ +0.2279, improved)
[vs baseline] epoch 20 val r2_min: -0.2126 -> -0.1630 (Δ +0.0496, improved)
[vs baseline] epoch 20 val worst_mae: 0.071793 -> 0.060158 (Δ -0.01164, improved)
[vs baseline] epoch 20 val tot: 1.2081 -> 0.3392 (Δ -0.8690, improved)
[vs baseline] epoch 20 val volt: 0.5030 -> 0.1105 (Δ -0.3925, improved)
[vs baseline] epoch 20 test |V| MAE: 0.017406 -> 0.011794 (Δ -0.00561, improved)
[vs baseline] epoch 20 test angle MAE: 3.051563 -> 0.980144 (Δ -2.07142, improved)
[vs baseline] epoch 20 test Re/Im MSE(nrm): 0.496211 -> 0.114133 (Δ -0.38208, improved)
[vs baseline] epoch 20 test r2_mean: 0.5532 -> 0.7684 (Δ +0.2152, improved)
[vs baseline] epoch 20 test r2_min: -0.1969 -> -0.0643 (Δ +0.1326, improved)
[vs baseline] epoch 20 test worst_mae: 0.071131 -> 0.059769 (Δ -0.01136, improved)
[vs baseline] epoch 20 test tot: 1.1707 -> 0.3442 (Δ -0.8266, improved)
[vs baseline] epoch 20 test volt: 0.4962 -> 0.1141 (Δ -0.3821, improved)

=== nobess_holdout (epoch 20) ===
Val  |V| MAE=0.011912  angle MAE=2.335002  Re/Im MSE(nrm)=0.260060  r2_mean=0.3895  r2_min=-8.8779  worst_mae=0.042433  tot=0.4616  volt=0.2601
Test  |V| MAE=0.012015  angle MAE=2.307518  Re/Im MSE(nrm)=0.256766  r2_mean=0.4095  r2_min=-8.4425  worst_mae=0.042895  tot=0.4566  volt=0.2568
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
[da_gps chunk_parent] epoch   30/60 | train_tot=0.3078 train_volt=0.0956 train_cap=0.2824 train_reg=1.7425 train_meta_aux=0.0968 val_meta_aux=0.1176 | val_tot=0.3030 val_volt=0.0932 val_cap=0.2681 val_reg=1.7122 | val_r2_mean=0.7988 val_r2_min=0.0648 val_r2_min_node=_hvmv_sub_lsb.1 val_worst_mae=0.0543 | best=0.3030
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.1511  cap_capbank0b_n_steps_on=0.2054  cap_capbank0c_n_steps_on=0.3494  cap_capbank1a_n_steps_on=0.2961  cap_capbank1b_n_steps_on=0.2864  cap_capbank1c_n_steps_on=0.4566
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3252  cap_capbank2b_n_steps_on=0.3257  cap_capbank2c_n_steps_on=0.4283  cap_capbank3_n_steps_on=0.0001
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.1320  cap_capbank0b_n_steps_on=0.1735  cap_capbank0c_n_steps_on=0.3514  cap_capbank1a_n_steps_on=0.2728  cap_capbank1b_n_steps_on=0.2603  cap_capbank1c_n_steps_on=0.4226
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.3309  cap_capbank2b_n_steps_on=0.3232  cap_capbank2c_n_steps_on=0.4148  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.3768  reg_feeder_regb_tap_pu=1.3846  reg_feeder_regc_tap_pu=1.2906  reg_vreg2_a_tap_pu=1.5951  reg_vreg2_b_tap_pu=1.8388  reg_vreg2_c_tap_pu=2.1463
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.4954  reg_vreg3_b_tap_pu=1.7929  reg_vreg3_c_tap_pu=2.2986  reg_vreg4_a_tap_pu=1.8248  reg_vreg4_b_tap_pu=1.6918  reg_vreg4_c_tap_pu=2.1746
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.3277  reg_feeder_regb_tap_pu=1.3210  reg_feeder_regc_tap_pu=1.1899  reg_vreg2_a_tap_pu=1.6074  reg_vreg2_b_tap_pu=1.8670  reg_vreg2_c_tap_pu=2.1200
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.4382  reg_vreg3_b_tap_pu=1.8035  reg_vreg3_c_tap_pu=2.2522  reg_vreg4_a_tap_pu=1.7978  reg_vreg4_b_tap_pu=1.6644  reg_vreg4_c_tap_pu=2.1567
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0209  pv_pv2_q_post_kvar=0.2991  p_loss_total_post_kw=0.0339  q_loss_total_post_kvar=0.0334
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0152  pv_pv2_q_post_kvar=0.3918  p_loss_total_post_kw=0.0345  q_loss_total_post_kvar=0.0288

=== train_pool_eval (epoch 30) ===
Val  |V| MAE=0.010855  angle MAE=0.822372  Re/Im MSE(nrm)=0.093197  r2_mean=0.7868  r2_min=-0.0003  worst_mae=0.054290  tot=0.3030  volt=0.0932
Test  |V| MAE=0.010984  angle MAE=0.811873  Re/Im MSE(nrm)=0.097593  r2_mean=0.7838  r2_min=-0.0378  worst_mae=0.054470  tot=0.3108  volt=0.0976
[vs baseline] epoch 30 val |V| MAE: 0.017585 -> 0.010855 (Δ -0.00673, improved)
[vs baseline] epoch 30 val angle MAE: 3.189312 -> 0.822372 (Δ -2.36694, improved)
[vs baseline] epoch 30 val Re/Im MSE(nrm): 0.503004 -> 0.093197 (Δ -0.40981, improved)
[vs baseline] epoch 30 val r2_mean: 0.5391 -> 0.7868 (Δ +0.2476, improved)
[vs baseline] epoch 30 val r2_min: -0.2126 -> -0.0003 (Δ +0.2123, improved)
[vs baseline] epoch 30 val worst_mae: 0.071793 -> 0.054290 (Δ -0.01750, improved)
[vs baseline] epoch 30 val tot: 1.2081 -> 0.3030 (Δ -0.9052, improved)
[vs baseline] epoch 30 val volt: 0.5030 -> 0.0932 (Δ -0.4098, improved)
[vs baseline] epoch 30 test |V| MAE: 0.017406 -> 0.010984 (Δ -0.00642, improved)
[vs baseline] epoch 30 test angle MAE: 3.051563 -> 0.811873 (Δ -2.23969, improved)
[vs baseline] epoch 30 test Re/Im MSE(nrm): 0.496211 -> 0.097593 (Δ -0.39862, improved)
[vs baseline] epoch 30 test r2_mean: 0.5532 -> 0.7838 (Δ +0.2306, improved)
[vs baseline] epoch 30 test r2_min: -0.1969 -> -0.0378 (Δ +0.1591, improved)
[vs baseline] epoch 30 test worst_mae: 0.071131 -> 0.054470 (Δ -0.01666, improved)
[vs baseline] epoch 30 test tot: 1.1707 -> 0.3108 (Δ -0.8599, improved)
[vs baseline] epoch 30 test volt: 0.4962 -> 0.0976 (Δ -0.3986, improved)

=== nobess_holdout (epoch 30) ===
Val  |V| MAE=0.009176  angle MAE=1.893459  Re/Im MSE(nrm)=0.196903  r2_mean=0.5249  r2_min=-6.3435  worst_mae=0.034324  tot=0.3892  volt=0.1969
Test  |V| MAE=0.009271  angle MAE=1.871399  Re/Im MSE(nrm)=0.194550  r2_mean=0.5396  r2_min=-6.3783  worst_mae=0.034754  tot=0.3854  volt=0.1945
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
[da_gps chunk_parent] epoch   40/60 | train_tot=0.2900 train_volt=0.0870 train_cap=0.2706 train_reg=1.6733 train_meta_aux=0.0862 val_meta_aux=0.0878 | val_tot=0.2823 val_volt=0.0823 val_cap=0.2575 val_reg=1.6548 | val_r2_mean=0.8161 val_r2_min=0.0958 val_r2_min_node=_hvmv_sub_lsb.1 val_worst_mae=0.0494 | best=0.2823
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.1364  cap_capbank0b_n_steps_on=0.1906  cap_capbank0c_n_steps_on=0.3433  cap_capbank1a_n_steps_on=0.2770  cap_capbank1b_n_steps_on=0.2766  cap_capbank1c_n_steps_on=0.4386
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3148  cap_capbank2b_n_steps_on=0.3155  cap_capbank2c_n_steps_on=0.4136  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.1191  cap_capbank0b_n_steps_on=0.1598  cap_capbank0c_n_steps_on=0.3400  cap_capbank1a_n_steps_on=0.2604  cap_capbank1b_n_steps_on=0.2559  cap_capbank1c_n_steps_on=0.4082
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.3080  cap_capbank2b_n_steps_on=0.3191  cap_capbank2c_n_steps_on=0.4048  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.2796  reg_feeder_regb_tap_pu=1.3224  reg_feeder_regc_tap_pu=1.2384  reg_vreg2_a_tap_pu=1.5468  reg_vreg2_b_tap_pu=1.8056  reg_vreg2_c_tap_pu=2.0957
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.4312  reg_vreg3_b_tap_pu=1.7359  reg_vreg3_c_tap_pu=2.2202  reg_vreg4_a_tap_pu=1.7036  reg_vreg4_b_tap_pu=1.6155  reg_vreg4_c_tap_pu=2.0845
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.2695  reg_feeder_regb_tap_pu=1.2758  reg_feeder_regc_tap_pu=1.1364  reg_vreg2_a_tap_pu=1.5744  reg_vreg2_b_tap_pu=1.8251  reg_vreg2_c_tap_pu=2.0696
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.3933  reg_vreg3_b_tap_pu=1.7710  reg_vreg3_c_tap_pu=2.1807  reg_vreg4_a_tap_pu=1.7114  reg_vreg4_b_tap_pu=1.5737  reg_vreg4_c_tap_pu=2.0764
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0190  pv_pv2_q_post_kvar=0.2694  p_loss_total_post_kw=0.0281  q_loss_total_post_kvar=0.0282
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0124  pv_pv2_q_post_kvar=0.3004  p_loss_total_post_kw=0.0213  q_loss_total_post_kvar=0.0170

=== train_pool_eval (epoch 40) ===
Val  |V| MAE=0.010107  angle MAE=0.743253  Re/Im MSE(nrm)=0.082278  r2_mean=0.8035  r2_min=0.0176  worst_mae=0.049417  tot=0.2823  volt=0.0823
Test  |V| MAE=0.010328  angle MAE=0.745117  Re/Im MSE(nrm)=0.085561  r2_mean=0.8004  r2_min=0.0206  worst_mae=0.050345  tot=0.2895  volt=0.0856
[vs baseline] epoch 40 val |V| MAE: 0.017585 -> 0.010107 (Δ -0.00748, improved)
[vs baseline] epoch 40 val angle MAE: 3.189312 -> 0.743253 (Δ -2.44606, improved)
[vs baseline] epoch 40 val Re/Im MSE(nrm): 0.503004 -> 0.082278 (Δ -0.42073, improved)
[vs baseline] epoch 40 val r2_mean: 0.5391 -> 0.8035 (Δ +0.2644, improved)
[vs baseline] epoch 40 val r2_min: -0.2126 -> 0.0176 (Δ +0.2303, improved)
[vs baseline] epoch 40 val worst_mae: 0.071793 -> 0.049417 (Δ -0.02238, improved)
[vs baseline] epoch 40 val tot: 1.2081 -> 0.2823 (Δ -0.9259, improved)
[vs baseline] epoch 40 val volt: 0.5030 -> 0.0823 (Δ -0.4207, improved)
[vs baseline] epoch 40 test |V| MAE: 0.017406 -> 0.010328 (Δ -0.00708, improved)
[vs baseline] epoch 40 test angle MAE: 3.051563 -> 0.745117 (Δ -2.30645, improved)
[vs baseline] epoch 40 test Re/Im MSE(nrm): 0.496211 -> 0.085561 (Δ -0.41065, improved)
[vs baseline] epoch 40 test r2_mean: 0.5532 -> 0.8004 (Δ +0.2472, improved)
[vs baseline] epoch 40 test r2_min: -0.1969 -> 0.0206 (Δ +0.2176, improved)
[vs baseline] epoch 40 test worst_mae: 0.071131 -> 0.050345 (Δ -0.02079, improved)
[vs baseline] epoch 40 test tot: 1.1707 -> 0.2895 (Δ -0.8812, improved)
[vs baseline] epoch 40 test volt: 0.4962 -> 0.0856 (Δ -0.4107, improved)

=== nobess_holdout (epoch 40) ===
Val  |V| MAE=0.008388  angle MAE=1.214580  Re/Im MSE(nrm)=0.106918  r2_mean=0.6692  r2_min=-3.9831  worst_mae=0.030753  tot=0.2808  volt=0.1069
Test  |V| MAE=0.008451  angle MAE=1.207848  Re/Im MSE(nrm)=0.106831  r2_mean=0.6757  r2_min=-4.0589  worst_mae=0.031058  tot=0.2796  volt=0.1068
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
[da_gps chunk_parent] epoch   50/60 | train_tot=0.2792 train_volt=0.0814 train_cap=0.2645 train_reg=1.6313 train_meta_aux=0.0816 val_meta_aux=0.1068 | val_tot=0.2830 val_volt=0.0830 val_cap=0.2516 val_reg=1.6420 | val_r2_mean=0.8107 val_r2_min=0.0857 val_r2_min_node=_hvmv_sub_lsb.1 val_worst_mae=0.0472 | best=0.2823
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.1322  cap_capbank0b_n_steps_on=0.1885  cap_capbank0c_n_steps_on=0.3318  cap_capbank1a_n_steps_on=0.2633  cap_capbank1b_n_steps_on=0.2677  cap_capbank1c_n_steps_on=0.4351
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3070  cap_capbank2b_n_steps_on=0.3103  cap_capbank2c_n_steps_on=0.4091  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.1145  cap_capbank0b_n_steps_on=0.1569  cap_capbank0c_n_steps_on=0.3313  cap_capbank1a_n_steps_on=0.2543  cap_capbank1b_n_steps_on=0.2535  cap_capbank1c_n_steps_on=0.3935
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.2935  cap_capbank2b_n_steps_on=0.3162  cap_capbank2c_n_steps_on=0.4027  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.2486  reg_feeder_regb_tap_pu=1.2662  reg_feeder_regc_tap_pu=1.1928  reg_vreg2_a_tap_pu=1.5110  reg_vreg2_b_tap_pu=1.7699  reg_vreg2_c_tap_pu=2.0791
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.3809  reg_vreg3_b_tap_pu=1.6978  reg_vreg3_c_tap_pu=2.1843  reg_vreg4_a_tap_pu=1.6369  reg_vreg4_b_tap_pu=1.5587  reg_vreg4_c_tap_pu=2.0496
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.2502  reg_feeder_regb_tap_pu=1.2708  reg_feeder_regc_tap_pu=1.1127  reg_vreg2_a_tap_pu=1.5431  reg_vreg2_b_tap_pu=1.8223  reg_vreg2_c_tap_pu=2.0411
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.3783  reg_vreg3_b_tap_pu=1.7731  reg_vreg3_c_tap_pu=2.1534  reg_vreg4_a_tap_pu=1.6617  reg_vreg4_b_tap_pu=1.6160  reg_vreg4_c_tap_pu=2.0814
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0178  pv_pv2_q_post_kvar=0.2577  p_loss_total_post_kw=0.0262  q_loss_total_post_kvar=0.0247
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0129  pv_pv2_q_post_kvar=0.3689  p_loss_total_post_kw=0.0254  q_loss_total_post_kvar=0.0199

=== train_pool_eval (epoch 50) ===
Val  |V| MAE=0.010213  angle MAE=0.731212  Re/Im MSE(nrm)=0.082999  r2_mean=0.7981  r2_min=0.0049  worst_mae=0.047216  tot=0.2830  volt=0.0830
Test  |V| MAE=0.010427  angle MAE=0.728885  Re/Im MSE(nrm)=0.086421  r2_mean=0.7970  r2_min=0.0179  worst_mae=0.048095  tot=0.2907  volt=0.0864
[vs baseline] epoch 50 val |V| MAE: 0.017585 -> 0.010213 (Δ -0.00737, improved)
[vs baseline] epoch 50 val angle MAE: 3.189312 -> 0.731212 (Δ -2.45810, improved)
[vs baseline] epoch 50 val Re/Im MSE(nrm): 0.503004 -> 0.082999 (Δ -0.42000, improved)
[vs baseline] epoch 50 val r2_mean: 0.5391 -> 0.7981 (Δ +0.2590, improved)
[vs baseline] epoch 50 val r2_min: -0.2126 -> 0.0049 (Δ +0.2175, improved)
[vs baseline] epoch 50 val worst_mae: 0.071793 -> 0.047216 (Δ -0.02458, improved)
[vs baseline] epoch 50 val tot: 1.2081 -> 0.2830 (Δ -0.9251, improved)
[vs baseline] epoch 50 val volt: 0.5030 -> 0.0830 (Δ -0.4200, improved)
[vs baseline] epoch 50 test |V| MAE: 0.017406 -> 0.010427 (Δ -0.00698, improved)
[vs baseline] epoch 50 test angle MAE: 3.051563 -> 0.728885 (Δ -2.32268, improved)
[vs baseline] epoch 50 test Re/Im MSE(nrm): 0.496211 -> 0.086421 (Δ -0.40979, improved)
[vs baseline] epoch 50 test r2_mean: 0.5532 -> 0.7970 (Δ +0.2438, improved)
[vs baseline] epoch 50 test r2_min: -0.1969 -> 0.0179 (Δ +0.2148, improved)
[vs baseline] epoch 50 test worst_mae: 0.071131 -> 0.048095 (Δ -0.02304, improved)
[vs baseline] epoch 50 test tot: 1.1707 -> 0.2907 (Δ -0.8800, improved)
[vs baseline] epoch 50 test volt: 0.4962 -> 0.0864 (Δ -0.4098, improved)

=== nobess_holdout (epoch 50) ===
Val  |V| MAE=0.008619  angle MAE=1.056002  Re/Im MSE(nrm)=0.115467  r2_mean=0.5138  r2_min=-6.8018  worst_mae=0.031505  tot=0.2820  volt=0.1155
Test  |V| MAE=0.008678  angle MAE=1.050506  Re/Im MSE(nrm)=0.116120  r2_mean=0.5235  r2_min=-6.8183  worst_mae=0.031827  tot=0.2818  volt=0.1161
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
[da_gps chunk_parent] epoch   60/60 | train_tot=0.2711 train_volt=0.0772 train_cap=0.2585 train_reg=1.6040 train_meta_aux=0.0761 val_meta_aux=0.1037 | val_tot=0.2781 val_volt=0.0795 val_cap=0.2494 val_reg=1.6324 | val_r2_mean=0.8163 val_r2_min=0.1016 val_r2_min_node=_hvmv_sub_lsb.1 val_worst_mae=0.0458 | best=0.2781
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.1266  cap_capbank0b_n_steps_on=0.1787  cap_capbank0c_n_steps_on=0.3220  cap_capbank1a_n_steps_on=0.2518  cap_capbank1b_n_steps_on=0.2562  cap_capbank1c_n_steps_on=0.4233
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.3067  cap_capbank2b_n_steps_on=0.3120  cap_capbank2c_n_steps_on=0.4074  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.1139  cap_capbank0b_n_steps_on=0.1535  cap_capbank0c_n_steps_on=0.3243  cap_capbank1a_n_steps_on=0.2546  cap_capbank1b_n_steps_on=0.2554  cap_capbank1c_n_steps_on=0.3827
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.2919  cap_capbank2b_n_steps_on=0.3158  cap_capbank2c_n_steps_on=0.4015  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=1.2150  reg_feeder_regb_tap_pu=1.2450  reg_feeder_regc_tap_pu=1.1844  reg_vreg2_a_tap_pu=1.4849  reg_vreg2_b_tap_pu=1.7594  reg_vreg2_c_tap_pu=2.0575
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=1.3440  reg_vreg3_b_tap_pu=1.6701  reg_vreg3_c_tap_pu=2.1473  reg_vreg4_a_tap_pu=1.5918  reg_vreg4_b_tap_pu=1.5311  reg_vreg4_c_tap_pu=2.0172
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=1.2334  reg_feeder_regb_tap_pu=1.2630  reg_feeder_regc_tap_pu=1.1008  reg_vreg2_a_tap_pu=1.5272  reg_vreg2_b_tap_pu=1.8263  reg_vreg2_c_tap_pu=2.0284
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=1.3592  reg_vreg3_b_tap_pu=1.7833  reg_vreg3_c_tap_pu=2.1378  reg_vreg4_a_tap_pu=1.6550  reg_vreg4_b_tap_pu=1.6091  reg_vreg4_c_tap_pu=2.0648
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0169  pv_pv2_q_post_kvar=0.2412  p_loss_total_post_kw=0.0243  q_loss_total_post_kvar=0.0220
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0117  pv_pv2_q_post_kvar=0.3644  p_loss_total_post_kw=0.0219  q_loss_total_post_kvar=0.0167

=== train_pool_eval (epoch 60) ===
Val  |V| MAE=0.009965  angle MAE=0.692624  Re/Im MSE(nrm)=0.079540  r2_mean=0.8033  r2_min=0.0144  worst_mae=0.045783  tot=0.2781  volt=0.0795
Test  |V| MAE=0.010226  angle MAE=0.690146  Re/Im MSE(nrm)=0.083512  r2_mean=0.8013  r2_min=0.0346  worst_mae=0.046801  tot=0.2855  volt=0.0835
[vs baseline] epoch 60 val |V| MAE: 0.017585 -> 0.009965 (Δ -0.00762, improved)
[vs baseline] epoch 60 val angle MAE: 3.189312 -> 0.692624 (Δ -2.49669, improved)
[vs baseline] epoch 60 val Re/Im MSE(nrm): 0.503004 -> 0.079540 (Δ -0.42346, improved)
[vs baseline] epoch 60 val r2_mean: 0.5391 -> 0.8033 (Δ +0.2642, improved)
[vs baseline] epoch 60 val r2_min: -0.2126 -> 0.0144 (Δ +0.2270, improved)
[vs baseline] epoch 60 val worst_mae: 0.071793 -> 0.045783 (Δ -0.02601, improved)
[vs baseline] epoch 60 val tot: 1.2081 -> 0.2781 (Δ -0.9301, improved)
[vs baseline] epoch 60 val volt: 0.5030 -> 0.0795 (Δ -0.4235, improved)
[vs baseline] epoch 60 test |V| MAE: 0.017406 -> 0.010226 (Δ -0.00718, improved)
[vs baseline] epoch 60 test angle MAE: 3.051563 -> 0.690146 (Δ -2.36142, improved)
[vs baseline] epoch 60 test Re/Im MSE(nrm): 0.496211 -> 0.083512 (Δ -0.41270, improved)
[vs baseline] epoch 60 test r2_mean: 0.5532 -> 0.8013 (Δ +0.2481, improved)
[vs baseline] epoch 60 test r2_min: -0.1969 -> 0.0346 (Δ +0.2315, improved)
[vs baseline] epoch 60 test worst_mae: 0.071131 -> 0.046801 (Δ -0.02433, improved)
[vs baseline] epoch 60 test tot: 1.1707 -> 0.2855 (Δ -0.8853, improved)
[vs baseline] epoch 60 test volt: 0.4962 -> 0.0835 (Δ -0.4127, improved)

=== nobess_holdout (epoch 60) ===
Val  |V| MAE=0.007910  angle MAE=0.932922  Re/Im MSE(nrm)=0.093611  r2_mean=0.5956  r2_min=-5.7985  worst_mae=0.029368  tot=0.2568  volt=0.0936
Test  |V| MAE=0.007977  angle MAE=0.928738  Re/Im MSE(nrm)=0.094321  r2_mean=0.6013  r2_min=-5.8100  worst_mae=0.029589  tot=0.2567  volt=0.0943
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt

=== Final evaluation (best checkpoint) ===

=== train_pool_eval (best epoch 60) ===
Val  |V| MAE=0.009965  angle MAE=0.692623  Re/Im MSE(nrm)=0.079540  r2_mean=0.8033  r2_min=0.0144  worst_mae=0.045784  tot=0.2781  volt=0.0795
Test  |V| MAE=0.010226  angle MAE=0.690146  Re/Im MSE(nrm)=0.083512  r2_mean=0.8013  r2_min=0.0347  worst_mae=0.046801  tot=0.2855  volt=0.0835

=== nobess_holdout (best epoch 60) ===
Val  |V| MAE=0.007910  angle MAE=0.932921  Re/Im MSE(nrm)=0.093611  r2_mean=0.5956  r2_min=-5.7983  worst_mae=0.029368  tot=0.2568  volt=0.0936
Test  |V| MAE=0.007977  angle MAE=0.928738  Re/Im MSE(nrm)=0.094321  r2_mean=0.6013  r2_min=-5.8097  worst_mae=0.029589  tot=0.2567  volt=0.0943

=== Before / after (init checkpoint vs fine-tuned best) ===
split  metric                     baseline        after        delta     status
val    |V| MAE                    0.017585     0.009965    -0.007620   improved
val    angle MAE                  3.189312     0.692623    -2.496689   improved
val    Re/Im MSE(nrm)             0.503004     0.079540    -0.423464   improved
val    r2_mean                      0.5391       0.8033      +0.2642   improved
val    r2_min                      -0.2126       0.0144      +0.2271   improved
val    worst_mae                  0.071793     0.045784    -0.026010   improved
val    tot                          1.2081       0.2781      -0.9301   improved
val    volt                         0.5030       0.0795      -0.4235   improved
test   |V| MAE                    0.017406     0.010226    -0.007180   improved
test   angle MAE                  3.051563     0.690146    -2.361417   improved
test   Re/Im MSE(nrm)             0.496211     0.083512    -0.412698   improved
test   r2_mean                      0.5532       0.8013      +0.2481   improved
test   r2_min                      -0.1969       0.0347      +0.2316   improved
test   worst_mae                  0.071131     0.046801    -0.024331   improved
test   tot                          1.1707       0.2855      -0.8853   improved
test   volt                         0.4962       0.0835      -0.4127   improved
Val |V| MAE=0.009965  angle MAE=0.692623  Re/Im MSE(nrm)=0.079540
Test |V| MAE=0.010226  angle MAE=0.690146  Re/Im MSE(nrm)=0.083512  cap_BCE=0.262227  reg_CE=1.660800  reg_acc=0.3780  meta_aux_MSE(nrm)=0.102506  meta_aux_MSE(raw)=69511.112938  time=2322.3s
[da_gps chunk_parent] Test per-head cap_BCE:
  cap_capbank0a_n_steps_on=0.105292
  cap_capbank0b_n_steps_on=0.143496
  cap_capbank0c_n_steps_on=0.329375
  cap_capbank1a_n_steps_on=0.261438
  cap_capbank1b_n_steps_on=0.268953
  cap_capbank1c_n_steps_on=0.431919
  cap_capbank2a_n_steps_on=0.326521
  cap_capbank2b_n_steps_on=0.333882
  cap_capbank2c_n_steps_on=0.421396
  cap_capbank3_n_steps_on=0.000000
[da_gps chunk_parent] Test per-head reg_MSE / reg_MAE (nrm / tap pu):
  reg_feeder_rega_tap_pu: MSE nrm=1.024000 pu=1.024000  MAE nrm=0.668800 pu=0.668800
  reg_feeder_regb_tap_pu: MSE nrm=1.096000 pu=1.096000  MAE nrm=0.673600 pu=0.673600
  reg_feeder_regc_tap_pu: MSE nrm=0.942400 pu=0.942400  MAE nrm=0.609600 pu=0.609600
  reg_vreg2_a_tap_pu: MSE nrm=4.708800 pu=4.708800  MAE nrm=1.284800 pu=1.284800
  reg_vreg2_b_tap_pu: MSE nrm=6.580800 pu=6.580800  MAE nrm=1.569600 pu=1.569600
  reg_vreg2_c_tap_pu: MSE nrm=6.249600 pu=6.249600  MAE nrm=1.641600 pu=1.641600
  reg_vreg3_a_tap_pu: MSE nrm=3.520000 pu=3.520000  MAE nrm=1.152000 pu=1.152000
  reg_vreg3_b_tap_pu: MSE nrm=4.179200 pu=4.179200  MAE nrm=1.366400 pu=1.366400
  reg_vreg3_c_tap_pu: MSE nrm=6.433600 pu=6.433600  MAE nrm=1.816000 pu=1.816000
  reg_vreg4_a_tap_pu: MSE nrm=6.462400 pu=6.462400  MAE nrm=1.473600 pu=1.473600
  reg_vreg4_b_tap_pu: MSE nrm=4.756800 pu=4.756800  MAE nrm=1.361600 pu=1.361600
  reg_vreg4_c_tap_pu: MSE nrm=6.721600 pu=6.721600  MAE nrm=1.752000 pu=1.752000
[da_gps chunk_parent] Test per-head meta_aux_MSE (nrm / raw):
  pv_pv2_p_post_kw: nrm=0.022160  raw=3645.824775
  pv_pv2_q_post_kvar: nrm=0.350197  raw=7124.912023
  p_loss_total_post_kw: nrm=0.022191  raw=55780.742456
  q_loss_total_post_kvar: nrm=0.015475  raw=211492.961125
Saved /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/da_gps_multitask_best.pt

Fine-tune completed.
Run dir: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739
Checkpoint (best): /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/da_gps_multitask_best.pt
Checkpoint (last): /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
Regulator classes: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/reg_class_tables.json
Report: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/da_gps_report.json

=== Inference / warm-start ===
RUN_DIR     = /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739
CHECKPOINT  = /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260709_221739/training_last.pt
Point cell 9 bootstrap or daily-compare at the paths above after training.

train on 44 chunks

Streaming output truncated to the last 5000 lines.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   10/60 | train_tot=0.1723 train_volt=0.0489 train_cap=0.0985 train_reg=1.0993 train_meta_aux=0.0359 val_meta_aux=0.0363 | val_tot=0.1633 val_volt=0.0521 val_cap=0.0840 val_reg=0.9924 | val_r2_mean=0.8872 val_r2_min=0.1332 val_r2_min_node=190-8593.3 val_worst_mae=0.0187 | best=0.1633
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0295  cap_capbank0b_n_steps_on=0.0399  cap_capbank0c_n_steps_on=0.0745  cap_capbank1a_n_steps_on=0.0791  cap_capbank1b_n_steps_on=0.0868  cap_capbank1c_n_steps_on=0.1518
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1442  cap_capbank2b_n_steps_on=0.1534  cap_capbank2c_n_steps_on=0.2257  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0258  cap_capbank0b_n_steps_on=0.0328  cap_capbank0c_n_steps_on=0.0737  cap_capbank1a_n_steps_on=0.0625  cap_capbank1b_n_steps_on=0.0723  cap_capbank1c_n_steps_on=0.1359
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1252  cap_capbank2b_n_steps_on=0.1295  cap_capbank2c_n_steps_on=0.1821  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.8057  reg_feeder_regb_tap_pu=0.8361  reg_feeder_regc_tap_pu=0.7662  reg_vreg2_a_tap_pu=1.0465  reg_vreg2_b_tap_pu=1.2328  reg_vreg2_c_tap_pu=1.4198
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.8526  reg_vreg3_b_tap_pu=1.2163  reg_vreg3_c_tap_pu=1.4452  reg_vreg4_a_tap_pu=1.0133  reg_vreg4_b_tap_pu=1.1467  reg_vreg4_c_tap_pu=1.4107
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.7268  reg_feeder_regb_tap_pu=0.7422  reg_feeder_regc_tap_pu=0.6839  reg_vreg2_a_tap_pu=0.9941  reg_vreg2_b_tap_pu=1.1393  reg_vreg2_c_tap_pu=1.3326
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.7793  reg_vreg3_b_tap_pu=1.0812  reg_vreg3_c_tap_pu=1.2941  reg_vreg4_a_tap_pu=0.9199  reg_vreg4_b_tap_pu=0.9891  reg_vreg4_c_tap_pu=1.2263
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0044  pv_pv2_q_post_kvar=0.0994  p_loss_total_post_kw=0.0184  q_loss_total_post_kvar=0.0213
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0039  pv_pv2_q_post_kvar=0.0947  p_loss_total_post_kw=0.0220  q_loss_total_post_kvar=0.0244
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 10) ===
Val  |V| MAE=0.005159  angle MAE=0.450402  Re/Im MSE(nrm)=0.052055  r2_mean=0.9000  r2_min=0.0775  worst_mae=0.018666  tot=0.1633  volt=0.0521
Test  |V| MAE=0.005268  angle MAE=0.452630  Re/Im MSE(nrm)=0.053061  r2_mean=0.8935  r2_min=0.0587  worst_mae=0.018944  tot=0.1657  volt=0.0531
[vs baseline] epoch 10 val |V| MAE: 0.004859 -> 0.005159 (Δ +0.00030, worse)
[vs baseline] epoch 10 val angle MAE: 0.414008 -> 0.450402 (Δ +0.03639, worse)
[vs baseline] epoch 10 val Re/Im MSE(nrm): 0.047581 -> 0.052055 (Δ +0.00447, worse)
[vs baseline] epoch 10 val r2_mean: 0.9053 -> 0.9000 (Δ -0.0053, worse)
[vs baseline] epoch 10 val r2_min: 0.1192 -> 0.0775 (Δ -0.0417, worse)
[vs baseline] epoch 10 val worst_mae: 0.017596 -> 0.018666 (Δ +0.00107, worse)
[vs baseline] epoch 10 val tot: 0.1790 -> 0.1633 (Δ -0.0156, improved)
[vs baseline] epoch 10 val volt: 0.0476 -> 0.0521 (Δ +0.0045, worse)
[vs baseline] epoch 10 test |V| MAE: 0.004957 -> 0.005268 (Δ +0.00031, worse)
[vs baseline] epoch 10 test angle MAE: 0.421952 -> 0.452630 (Δ +0.03068, worse)
[vs baseline] epoch 10 test Re/Im MSE(nrm): 0.049633 -> 0.053061 (Δ +0.00343, worse)
[vs baseline] epoch 10 test r2_mean: 0.8994 -> 0.8935 (Δ -0.0059, worse)
[vs baseline] epoch 10 test r2_min: 0.0932 -> 0.0587 (Δ -0.0344, worse)
[vs baseline] epoch 10 test worst_mae: 0.017805 -> 0.018944 (Δ +0.00114, worse)
[vs baseline] epoch 10 test tot: 0.1823 -> 0.1657 (Δ -0.0167, improved)
[vs baseline] epoch 10 test volt: 0.0496 -> 0.0531 (Δ +0.0034, worse)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 10) ===
Val  |V| MAE=0.004012  angle MAE=0.197932  Re/Im MSE(nrm)=0.010520  r2_mean=0.9278  r2_min=0.0874  worst_mae=0.013735  tot=0.0995  volt=0.0105
Test  |V| MAE=0.004063  angle MAE=0.199328  Re/Im MSE(nrm)=0.010882  r2_mean=0.9214  r2_min=0.0657  worst_mae=0.013901  tot=0.1006  volt=0.0109
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 10) ===
Val  |V| MAE=0.019094  angle MAE=3.516754  Re/Im MSE(nrm)=0.556521  r2_mean=0.5621  r2_min=-0.0418  worst_mae=0.078554  tot=0.9385  volt=0.5565
Test  |V| MAE=0.019870  angle MAE=3.522249  Re/Im MSE(nrm)=0.564197  r2_mean=0.5551  r2_min=-0.0258  worst_mae=0.080051  tot=0.9536  volt=0.5642
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   20/60 | train_tot=0.1578 train_volt=0.0411 train_cap=0.0892 train_reg=1.0443 train_meta_aux=0.0332 val_meta_aux=0.0246 | val_tot=0.1442 val_volt=0.0384 val_cap=0.0771 val_reg=0.9569 | val_r2_mean=0.8944 val_r2_min=0.1429 val_r2_min_node=q14733.3 val_worst_mae=0.0183 | best=0.1442
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0254  cap_capbank0b_n_steps_on=0.0337  cap_capbank0c_n_steps_on=0.0683  cap_capbank1a_n_steps_on=0.0691  cap_capbank1b_n_steps_on=0.0795  cap_capbank1c_n_steps_on=0.1369
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1321  cap_capbank2b_n_steps_on=0.1391  cap_capbank2c_n_steps_on=0.2078  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0244  cap_capbank0b_n_steps_on=0.0297  cap_capbank0c_n_steps_on=0.0637  cap_capbank1a_n_steps_on=0.0541  cap_capbank1b_n_steps_on=0.0647  cap_capbank1c_n_steps_on=0.1226
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1134  cap_capbank2b_n_steps_on=0.1212  cap_capbank2c_n_steps_on=0.1770  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.7563  reg_feeder_regb_tap_pu=0.7915  reg_feeder_regc_tap_pu=0.7479  reg_vreg2_a_tap_pu=0.9949  reg_vreg2_b_tap_pu=1.1878  reg_vreg2_c_tap_pu=1.3877
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.7846  reg_vreg3_b_tap_pu=1.1285  reg_vreg3_c_tap_pu=1.4068  reg_vreg4_a_tap_pu=0.9466  reg_vreg4_b_tap_pu=1.0595  reg_vreg4_c_tap_pu=1.3398
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.6761  reg_feeder_regb_tap_pu=0.6814  reg_feeder_regc_tap_pu=0.6461  reg_vreg2_a_tap_pu=0.9643  reg_vreg2_b_tap_pu=1.1103  reg_vreg2_c_tap_pu=1.3215
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.7384  reg_vreg3_b_tap_pu=1.0191  reg_vreg3_c_tap_pu=1.2814  reg_vreg4_a_tap_pu=0.8999  reg_vreg4_b_tap_pu=0.9468  reg_vreg4_c_tap_pu=1.1979
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0042  pv_pv2_q_post_kvar=0.0942  p_loss_total_post_kw=0.0158  q_loss_total_post_kvar=0.0185
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0047  pv_pv2_q_post_kvar=0.0706  p_loss_total_post_kw=0.0106  q_loss_total_post_kvar=0.0124
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 20) ===
Val  |V| MAE=0.005073  angle MAE=0.416060  Re/Im MSE(nrm)=0.038353  r2_mean=0.9074  r2_min=0.1034  worst_mae=0.018290  tot=0.1442  volt=0.0384
Test  |V| MAE=0.005179  angle MAE=0.420071  Re/Im MSE(nrm)=0.040177  r2_mean=0.9010  r2_min=0.0826  worst_mae=0.018729  tot=0.1472  volt=0.0402
[vs baseline] epoch 20 val |V| MAE: 0.004859 -> 0.005073 (Δ +0.00021, worse)
[vs baseline] epoch 20 val angle MAE: 0.414008 -> 0.416060 (Δ +0.00205, worse)
[vs baseline] epoch 20 val Re/Im MSE(nrm): 0.047581 -> 0.038353 (Δ -0.00923, improved)
[vs baseline] epoch 20 val r2_mean: 0.9053 -> 0.9074 (Δ +0.0021, improved)
[vs baseline] epoch 20 val r2_min: 0.1192 -> 0.1034 (Δ -0.0158, worse)
[vs baseline] epoch 20 val worst_mae: 0.017596 -> 0.018290 (Δ +0.00069, worse)
[vs baseline] epoch 20 val tot: 0.1790 -> 0.1442 (Δ -0.0347, improved)
[vs baseline] epoch 20 val volt: 0.0476 -> 0.0384 (Δ -0.0092, improved)
[vs baseline] epoch 20 test |V| MAE: 0.004957 -> 0.005179 (Δ +0.00022, worse)
[vs baseline] epoch 20 test angle MAE: 0.421952 -> 0.420071 (Δ -0.00188, improved)
[vs baseline] epoch 20 test Re/Im MSE(nrm): 0.049633 -> 0.040177 (Δ -0.00946, improved)
[vs baseline] epoch 20 test r2_mean: 0.8994 -> 0.9010 (Δ +0.0016, improved)
[vs baseline] epoch 20 test r2_min: 0.0932 -> 0.0826 (Δ -0.0105, worse)
[vs baseline] epoch 20 test worst_mae: 0.017805 -> 0.018729 (Δ +0.00092, worse)
[vs baseline] epoch 20 test tot: 0.1823 -> 0.1472 (Δ -0.0352, improved)
[vs baseline] epoch 20 test volt: 0.0496 -> 0.0402 (Δ -0.0095, improved)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 20) ===
Val  |V| MAE=0.004200  angle MAE=0.245295  Re/Im MSE(nrm)=0.011824  r2_mean=0.9283  r2_min=0.1182  worst_mae=0.014334  tot=0.1033  volt=0.0118
Test  |V| MAE=0.004247  angle MAE=0.246745  Re/Im MSE(nrm)=0.012110  r2_mean=0.9224  r2_min=0.0951  worst_mae=0.014461  tot=0.1044  volt=0.0121
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 20) ===
Val  |V| MAE=0.015679  angle MAE=2.490066  Re/Im MSE(nrm)=0.360559  r2_mean=0.6535  r2_min=-0.0756  worst_mae=0.066334  tot=0.6409  volt=0.3606
Test  |V| MAE=0.016473  angle MAE=2.520502  Re/Im MSE(nrm)=0.380309  r2_mean=0.6412  r2_min=-0.0687  worst_mae=0.070447  tot=0.6657  volt=0.3803
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   30/60 | train_tot=0.1221 train_volt=0.0189 train_cap=0.0786 train_reg=0.9380 train_meta_aux=0.0148 val_meta_aux=0.0115 | val_tot=0.1149 val_volt=0.0178 val_cap=0.0702 val_reg=0.8896 | val_r2_mean=0.9183 val_r2_min=0.2188 val_r2_min_node=190-8593.3 val_worst_mae=0.0163 | best=0.1149
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0223  cap_capbank0b_n_steps_on=0.0311  cap_capbank0c_n_steps_on=0.0606  cap_capbank1a_n_steps_on=0.0537  cap_capbank1b_n_steps_on=0.0658  cap_capbank1c_n_steps_on=0.1209
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1156  cap_capbank2b_n_steps_on=0.1255  cap_capbank2c_n_steps_on=0.1906  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0206  cap_capbank0b_n_steps_on=0.0253  cap_capbank0c_n_steps_on=0.0620  cap_capbank1a_n_steps_on=0.0449  cap_capbank1b_n_steps_on=0.0584  cap_capbank1c_n_steps_on=0.1105
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1029  cap_capbank2b_n_steps_on=0.1130  cap_capbank2c_n_steps_on=0.1640  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.6380  reg_feeder_regb_tap_pu=0.6588  reg_feeder_regc_tap_pu=0.6143  reg_vreg2_a_tap_pu=0.9397  reg_vreg2_b_tap_pu=1.1067  reg_vreg2_c_tap_pu=1.3328
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.6995  reg_vreg3_b_tap_pu=0.9940  reg_vreg3_c_tap_pu=1.2794  reg_vreg4_a_tap_pu=0.8532  reg_vreg4_b_tap_pu=0.9319  reg_vreg4_c_tap_pu=1.2081
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.6082  reg_feeder_regb_tap_pu=0.6156  reg_feeder_regc_tap_pu=0.5816  reg_vreg2_a_tap_pu=0.9006  reg_vreg2_b_tap_pu=1.0645  reg_vreg2_c_tap_pu=1.2894
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.6572  reg_vreg3_b_tap_pu=0.9299  reg_vreg3_c_tap_pu=1.2140  reg_vreg4_a_tap_pu=0.8003  reg_vreg4_b_tap_pu=0.8731  reg_vreg4_c_tap_pu=1.1409
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0038  pv_pv2_q_post_kvar=0.0458  p_loss_total_post_kw=0.0046  q_loss_total_post_kvar=0.0051
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0034  pv_pv2_q_post_kvar=0.0366  p_loss_total_post_kw=0.0028  q_loss_total_post_kvar=0.0031
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 30) ===
Val  |V| MAE=0.004440  angle MAE=0.255876  Re/Im MSE(nrm)=0.017789  r2_mean=0.9218  r2_min=0.1087  worst_mae=0.016258  tot=0.1149  volt=0.0178
Test  |V| MAE=0.004521  angle MAE=0.259660  Re/Im MSE(nrm)=0.018712  r2_mean=0.9152  r2_min=0.0870  worst_mae=0.016529  tot=0.1166  volt=0.0187
[vs baseline] epoch 30 val |V| MAE: 0.004859 -> 0.004440 (Δ -0.00042, improved)
[vs baseline] epoch 30 val angle MAE: 0.414008 -> 0.255876 (Δ -0.15813, improved)
[vs baseline] epoch 30 val Re/Im MSE(nrm): 0.047581 -> 0.017789 (Δ -0.02979, improved)
[vs baseline] epoch 30 val r2_mean: 0.9053 -> 0.9218 (Δ +0.0165, improved)
[vs baseline] epoch 30 val r2_min: 0.1192 -> 0.1087 (Δ -0.0106, worse)
[vs baseline] epoch 30 val worst_mae: 0.017596 -> 0.016258 (Δ -0.00134, improved)
[vs baseline] epoch 30 val tot: 0.1790 -> 0.1149 (Δ -0.0640, improved)
[vs baseline] epoch 30 val volt: 0.0476 -> 0.0178 (Δ -0.0298, improved)
[vs baseline] epoch 30 test |V| MAE: 0.004957 -> 0.004521 (Δ -0.00044, improved)
[vs baseline] epoch 30 test angle MAE: 0.421952 -> 0.259660 (Δ -0.16229, improved)
[vs baseline] epoch 30 test Re/Im MSE(nrm): 0.049633 -> 0.018712 (Δ -0.03092, improved)
[vs baseline] epoch 30 test r2_mean: 0.8994 -> 0.9152 (Δ +0.0158, improved)
[vs baseline] epoch 30 test r2_min: 0.0932 -> 0.0870 (Δ -0.0061, worse)
[vs baseline] epoch 30 test worst_mae: 0.017805 -> 0.016529 (Δ -0.00128, improved)
[vs baseline] epoch 30 test tot: 0.1823 -> 0.1166 (Δ -0.0657, improved)
[vs baseline] epoch 30 test volt: 0.0496 -> 0.0187 (Δ -0.0309, improved)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 30) ===
Val  |V| MAE=0.003908  angle MAE=0.194970  Re/Im MSE(nrm)=0.010059  r2_mean=0.9309  r2_min=0.1100  worst_mae=0.013408  tot=0.0985  volt=0.0101
Test  |V| MAE=0.003965  angle MAE=0.196844  Re/Im MSE(nrm)=0.010395  r2_mean=0.9251  r2_min=0.0892  worst_mae=0.013542  tot=0.0996  volt=0.0104
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 30) ===
Val  |V| MAE=0.010911  angle MAE=0.995608  Re/Im MSE(nrm)=0.111680  r2_mean=0.8113  r2_min=0.0930  worst_mae=0.050865  tot=0.3139  volt=0.1117
Test  |V| MAE=0.011257  angle MAE=1.020881  Re/Im MSE(nrm)=0.119500  r2_mean=0.7954  r2_min=0.0610  worst_mae=0.052729  tot=0.3233  volt=0.1195
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   40/60 | train_tot=0.1200 train_volt=0.0179 train_cap=0.0771 train_reg=0.9305 train_meta_aux=0.0136 val_meta_aux=0.0132 | val_tot=0.1154 val_volt=0.0188 val_cap=0.0695 val_reg=0.8836 | val_r2_mean=0.9177 val_r2_min=0.2102 val_r2_min_node=190-8593.3 val_worst_mae=0.0166 | best=0.1149
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0210  cap_capbank0b_n_steps_on=0.0306  cap_capbank0c_n_steps_on=0.0599  cap_capbank1a_n_steps_on=0.0518  cap_capbank1b_n_steps_on=0.0639  cap_capbank1c_n_steps_on=0.1183
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1140  cap_capbank2b_n_steps_on=0.1239  cap_capbank2c_n_steps_on=0.1873  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0199  cap_capbank0b_n_steps_on=0.0249  cap_capbank0c_n_steps_on=0.0609  cap_capbank1a_n_steps_on=0.0441  cap_capbank1b_n_steps_on=0.0586  cap_capbank1c_n_steps_on=0.1088
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1019  cap_capbank2b_n_steps_on=0.1120  cap_capbank2c_n_steps_on=0.1642  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.6302  reg_feeder_regb_tap_pu=0.6558  reg_feeder_regc_tap_pu=0.6065  reg_vreg2_a_tap_pu=0.9358  reg_vreg2_b_tap_pu=1.1055  reg_vreg2_c_tap_pu=1.3226
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.6911  reg_vreg3_b_tap_pu=0.9891  reg_vreg3_c_tap_pu=1.2624  reg_vreg4_a_tap_pu=0.8456  reg_vreg4_b_tap_pu=0.9270  reg_vreg4_c_tap_pu=1.1950
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.5980  reg_feeder_regb_tap_pu=0.6148  reg_feeder_regc_tap_pu=0.5781  reg_vreg2_a_tap_pu=0.8820  reg_vreg2_b_tap_pu=1.0640  reg_vreg2_c_tap_pu=1.2830
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.6462  reg_vreg3_b_tap_pu=0.9356  reg_vreg3_c_tap_pu=1.2080  reg_vreg4_a_tap_pu=0.7890  reg_vreg4_b_tap_pu=0.8722  reg_vreg4_c_tap_pu=1.1324
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0038  pv_pv2_q_post_kvar=0.0426  p_loss_total_post_kw=0.0039  q_loss_total_post_kvar=0.0042
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0032  pv_pv2_q_post_kvar=0.0442  p_loss_total_post_kw=0.0024  q_loss_total_post_kvar=0.0028
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 40) ===
Val  |V| MAE=0.004525  angle MAE=0.265830  Re/Im MSE(nrm)=0.018766  r2_mean=0.9211  r2_min=0.0991  worst_mae=0.016589  tot=0.1154  volt=0.0188
Test  |V| MAE=0.004601  angle MAE=0.269221  Re/Im MSE(nrm)=0.019107  r2_mean=0.9151  r2_min=0.0730  worst_mae=0.016830  tot=0.1167  volt=0.0191
[vs baseline] epoch 40 val |V| MAE: 0.004859 -> 0.004525 (Δ -0.00033, improved)
[vs baseline] epoch 40 val angle MAE: 0.414008 -> 0.265830 (Δ -0.14818, improved)
[vs baseline] epoch 40 val Re/Im MSE(nrm): 0.047581 -> 0.018766 (Δ -0.02882, improved)
[vs baseline] epoch 40 val r2_mean: 0.9053 -> 0.9211 (Δ +0.0158, improved)
[vs baseline] epoch 40 val r2_min: 0.1192 -> 0.0991 (Δ -0.0202, worse)
[vs baseline] epoch 40 val worst_mae: 0.017596 -> 0.016589 (Δ -0.00101, improved)
[vs baseline] epoch 40 val tot: 0.1790 -> 0.1154 (Δ -0.0636, improved)
[vs baseline] epoch 40 val volt: 0.0476 -> 0.0188 (Δ -0.0288, improved)
[vs baseline] epoch 40 test |V| MAE: 0.004957 -> 0.004601 (Δ -0.00036, improved)
[vs baseline] epoch 40 test angle MAE: 0.421952 -> 0.269221 (Δ -0.15273, improved)
[vs baseline] epoch 40 test Re/Im MSE(nrm): 0.049633 -> 0.019107 (Δ -0.03053, improved)
[vs baseline] epoch 40 test r2_mean: 0.8994 -> 0.9151 (Δ +0.0157, improved)
[vs baseline] epoch 40 test r2_min: 0.0932 -> 0.0730 (Δ -0.0202, worse)
[vs baseline] epoch 40 test worst_mae: 0.017805 -> 0.016830 (Δ -0.00098, improved)
[vs baseline] epoch 40 test tot: 0.1823 -> 0.1167 (Δ -0.0656, improved)
[vs baseline] epoch 40 test volt: 0.0496 -> 0.0191 (Δ -0.0305, improved)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 40) ===
Val  |V| MAE=0.004024  angle MAE=0.204651  Re/Im MSE(nrm)=0.010439  r2_mean=0.9300  r2_min=0.1022  worst_mae=0.013590  tot=0.0989  volt=0.0104
Test  |V| MAE=0.004085  angle MAE=0.206704  Re/Im MSE(nrm)=0.010796  r2_mean=0.9240  r2_min=0.0778  worst_mae=0.013772  tot=0.1000  volt=0.0108
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 40) ===
Val  |V| MAE=0.010616  angle MAE=1.008865  Re/Im MSE(nrm)=0.119905  r2_mean=0.8133  r2_min=0.0600  worst_mae=0.053016  tot=0.3162  volt=0.1199
Test  |V| MAE=0.010853  angle MAE=1.026826  Re/Im MSE(nrm)=0.119825  r2_mean=0.8075  r2_min=0.0151  worst_mae=0.053886  tot=0.3190  volt=0.1198
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   50/60 | train_tot=0.1181 train_volt=0.0169 train_cap=0.0762 train_reg=0.9230 train_meta_aux=0.0128 val_meta_aux=0.0135 | val_tot=0.1161 val_volt=0.0192 val_cap=0.0700 val_reg=0.8862 | val_r2_mean=0.9175 val_r2_min=0.2159 val_r2_min_node=190-8593.3 val_worst_mae=0.0166 | best=0.1149
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0206  cap_capbank0b_n_steps_on=0.0290  cap_capbank0c_n_steps_on=0.0585  cap_capbank1a_n_steps_on=0.0510  cap_capbank1b_n_steps_on=0.0627  cap_capbank1c_n_steps_on=0.1174
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1129  cap_capbank2b_n_steps_on=0.1219  cap_capbank2c_n_steps_on=0.1882  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0186  cap_capbank0b_n_steps_on=0.0236  cap_capbank0c_n_steps_on=0.0600  cap_capbank1a_n_steps_on=0.0436  cap_capbank1b_n_steps_on=0.0603  cap_capbank1c_n_steps_on=0.1082
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1033  cap_capbank2b_n_steps_on=0.1124  cap_capbank2c_n_steps_on=0.1702  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.6247  reg_feeder_regb_tap_pu=0.6471  reg_feeder_regc_tap_pu=0.6011  reg_vreg2_a_tap_pu=0.9296  reg_vreg2_b_tap_pu=1.0979  reg_vreg2_c_tap_pu=1.3175
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.6831  reg_vreg3_b_tap_pu=0.9802  reg_vreg3_c_tap_pu=1.2559  reg_vreg4_a_tap_pu=0.8370  reg_vreg4_b_tap_pu=0.9167  reg_vreg4_c_tap_pu=1.1855
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.6005  reg_feeder_regb_tap_pu=0.6119  reg_feeder_regc_tap_pu=0.5735  reg_vreg2_a_tap_pu=0.8937  reg_vreg2_b_tap_pu=1.0619  reg_vreg2_c_tap_pu=1.2844
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.6516  reg_vreg3_b_tap_pu=0.9353  reg_vreg3_c_tap_pu=1.2083  reg_vreg4_a_tap_pu=0.8039  reg_vreg4_b_tap_pu=0.8735  reg_vreg4_c_tap_pu=1.1355
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0036  pv_pv2_q_post_kvar=0.0401  p_loss_total_post_kw=0.0036  q_loss_total_post_kvar=0.0039
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0031  pv_pv2_q_post_kvar=0.0434  p_loss_total_post_kw=0.0036  q_loss_total_post_kvar=0.0039
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 50) ===
Val  |V| MAE=0.004684  angle MAE=0.310389  Re/Im MSE(nrm)=0.019175  r2_mean=0.9201  r2_min=0.1057  worst_mae=0.016649  tot=0.1161  volt=0.0192
Test  |V| MAE=0.004761  angle MAE=0.308815  Re/Im MSE(nrm)=0.019718  r2_mean=0.9135  r2_min=0.0877  worst_mae=0.016931  tot=0.1176  volt=0.0197
[vs baseline] epoch 50 val |V| MAE: 0.004859 -> 0.004684 (Δ -0.00017, improved)
[vs baseline] epoch 50 val angle MAE: 0.414008 -> 0.310389 (Δ -0.10362, improved)
[vs baseline] epoch 50 val Re/Im MSE(nrm): 0.047581 -> 0.019175 (Δ -0.02841, improved)
[vs baseline] epoch 50 val r2_mean: 0.9053 -> 0.9201 (Δ +0.0148, improved)
[vs baseline] epoch 50 val r2_min: 0.1192 -> 0.1057 (Δ -0.0136, worse)
[vs baseline] epoch 50 val worst_mae: 0.017596 -> 0.016649 (Δ -0.00095, improved)
[vs baseline] epoch 50 val tot: 0.1790 -> 0.1161 (Δ -0.0628, improved)
[vs baseline] epoch 50 val volt: 0.0476 -> 0.0192 (Δ -0.0284, improved)
[vs baseline] epoch 50 test |V| MAE: 0.004957 -> 0.004761 (Δ -0.00020, improved)
[vs baseline] epoch 50 test angle MAE: 0.421952 -> 0.308815 (Δ -0.11314, improved)
[vs baseline] epoch 50 test Re/Im MSE(nrm): 0.049633 -> 0.019718 (Δ -0.02991, improved)
[vs baseline] epoch 50 test r2_mean: 0.8994 -> 0.9135 (Δ +0.0141, improved)
[vs baseline] epoch 50 test r2_min: 0.0932 -> 0.0877 (Δ -0.0055, worse)
[vs baseline] epoch 50 test worst_mae: 0.017805 -> 0.016931 (Δ -0.00087, improved)
[vs baseline] epoch 50 test tot: 0.1823 -> 0.1176 (Δ -0.0647, improved)
[vs baseline] epoch 50 test volt: 0.0496 -> 0.0197 (Δ -0.0299, improved)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 50) ===
Val  |V| MAE=0.004073  angle MAE=0.236479  Re/Im MSE(nrm)=0.010700  r2_mean=0.9299  r2_min=0.1124  worst_mae=0.013564  tot=0.0993  volt=0.0107
Test  |V| MAE=0.004133  angle MAE=0.236690  Re/Im MSE(nrm)=0.011061  r2_mean=0.9238  r2_min=0.0931  worst_mae=0.013727  tot=0.1005  volt=0.0111
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 50) ===
Val  |V| MAE=0.012109  angle MAE=1.208054  Re/Im MSE(nrm)=0.122110  r2_mean=0.8014  r2_min=0.0249  worst_mae=0.054111  tot=0.3202  volt=0.1221
Test  |V| MAE=0.012381  angle MAE=1.182856  Re/Im MSE(nrm)=0.124633  r2_mean=0.7892  r2_min=0.0220  worst_mae=0.055757  tot=0.3249  volt=0.1246
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7433: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:7646: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
[da_gps chunk_parent] epoch   60/60 | train_tot=0.1188 train_volt=0.0176 train_cap=0.0757 train_reg=0.9230 train_meta_aux=0.0128 val_meta_aux=0.0118 | val_tot=0.1131 val_volt=0.0166 val_cap=0.0709 val_reg=0.8825 | val_r2_mean=0.9178 val_r2_min=0.2088 val_r2_min_node=190-8593.3 val_worst_mae=0.0157 | best=0.1131
[da_gps chunk_parent]  cap_BCE train: cap_capbank0a_n_steps_on=0.0196  cap_capbank0b_n_steps_on=0.0285  cap_capbank0c_n_steps_on=0.0577  cap_capbank1a_n_steps_on=0.0509  cap_capbank1b_n_steps_on=0.0629  cap_capbank1c_n_steps_on=0.1168
[da_gps chunk_parent]  cap_BCE train: cap_capbank2a_n_steps_on=0.1134  cap_capbank2b_n_steps_on=0.1221  cap_capbank2c_n_steps_on=0.1852  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  cap_BCE val: cap_capbank0a_n_steps_on=0.0193  cap_capbank0b_n_steps_on=0.0255  cap_capbank0c_n_steps_on=0.0599  cap_capbank1a_n_steps_on=0.0435  cap_capbank1b_n_steps_on=0.0585  cap_capbank1c_n_steps_on=0.1104
[da_gps chunk_parent]  cap_BCE val: cap_capbank2a_n_steps_on=0.1104  cap_capbank2b_n_steps_on=0.1160  cap_capbank2c_n_steps_on=0.1657  cap_capbank3_n_steps_on=0.0000
[da_gps chunk_parent]  reg_CE train: reg_feeder_rega_tap_pu=0.6257  reg_feeder_regb_tap_pu=0.6450  reg_feeder_regc_tap_pu=0.5995  reg_vreg2_a_tap_pu=0.9323  reg_vreg2_b_tap_pu=1.0959  reg_vreg2_c_tap_pu=1.3149
[da_gps chunk_parent]  reg_CE train: reg_vreg3_a_tap_pu=0.6878  reg_vreg3_b_tap_pu=0.9786  reg_vreg3_c_tap_pu=1.2541  reg_vreg4_a_tap_pu=0.8432  reg_vreg4_b_tap_pu=0.9176  reg_vreg4_c_tap_pu=1.1807
[da_gps chunk_parent]  reg_CE val: reg_feeder_rega_tap_pu=0.5949  reg_feeder_regb_tap_pu=0.6142  reg_feeder_regc_tap_pu=0.5751  reg_vreg2_a_tap_pu=0.8816  reg_vreg2_b_tap_pu=1.0603  reg_vreg2_c_tap_pu=1.2819
[da_gps chunk_parent]  reg_CE val: reg_vreg3_a_tap_pu=0.6507  reg_vreg3_b_tap_pu=0.9286  reg_vreg3_c_tap_pu=1.2062  reg_vreg4_a_tap_pu=0.7869  reg_vreg4_b_tap_pu=0.8788  reg_vreg4_c_tap_pu=1.1307
[da_gps chunk_parent]  meta_aux_MSE_nrm train: pv_pv2_p_post_kw=0.0035  pv_pv2_q_post_kvar=0.0395  p_loss_total_post_kw=0.0040  q_loss_total_post_kvar=0.0041
[da_gps chunk_parent]  meta_aux_MSE_nrm val: pv_pv2_p_post_kw=0.0034  pv_pv2_q_post_kvar=0.0391  p_loss_total_post_kw=0.0022  q_loss_total_post_kvar=0.0024
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== train_pool_eval (epoch 60) ===
Val  |V| MAE=0.004283  angle MAE=0.223505  Re/Im MSE(nrm)=0.016583  r2_mean=0.9227  r2_min=0.1353  worst_mae=0.015679  tot=0.1131  volt=0.0166
Test  |V| MAE=0.004365  angle MAE=0.226651  Re/Im MSE(nrm)=0.017455  r2_mean=0.9166  r2_min=0.1056  worst_mae=0.016050  tot=0.1148  volt=0.0175
[vs baseline] epoch 60 val |V| MAE: 0.004859 -> 0.004283 (Δ -0.00058, improved)
[vs baseline] epoch 60 val angle MAE: 0.414008 -> 0.223505 (Δ -0.19050, improved)
[vs baseline] epoch 60 val Re/Im MSE(nrm): 0.047581 -> 0.016583 (Δ -0.03100, improved)
[vs baseline] epoch 60 val r2_mean: 0.9053 -> 0.9227 (Δ +0.0174, improved)
[vs baseline] epoch 60 val r2_min: 0.1192 -> 0.1353 (Δ +0.0161, improved)
[vs baseline] epoch 60 val worst_mae: 0.017596 -> 0.015679 (Δ -0.00192, improved)
[vs baseline] epoch 60 val tot: 0.1790 -> 0.1131 (Δ -0.0659, improved)
[vs baseline] epoch 60 val volt: 0.0476 -> 0.0166 (Δ -0.0310, improved)
[vs baseline] epoch 60 test |V| MAE: 0.004957 -> 0.004365 (Δ -0.00059, improved)
[vs baseline] epoch 60 test angle MAE: 0.421952 -> 0.226651 (Δ -0.19530, improved)
[vs baseline] epoch 60 test Re/Im MSE(nrm): 0.049633 -> 0.017455 (Δ -0.03218, improved)
[vs baseline] epoch 60 test r2_mean: 0.8994 -> 0.9166 (Δ +0.0172, improved)
[vs baseline] epoch 60 test r2_min: 0.0932 -> 0.1056 (Δ +0.0124, improved)
[vs baseline] epoch 60 test worst_mae: 0.017805 -> 0.016050 (Δ -0.00176, improved)
[vs baseline] epoch 60 test tot: 0.1823 -> 0.1148 (Δ -0.0675, improved)
[vs baseline] epoch 60 test volt: 0.0496 -> 0.0175 (Δ -0.0322, improved)
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== nobess_40 (epoch 60) ===
Val  |V| MAE=0.003785  angle MAE=0.170949  Re/Im MSE(nrm)=0.009658  r2_mean=0.9319  r2_min=0.1398  worst_mae=0.013052  tot=0.0984  volt=0.0097
Test  |V| MAE=0.003835  angle MAE=0.172160  Re/Im MSE(nrm)=0.009963  r2_mean=0.9263  r2_min=0.1126  worst_mae=0.013196  tot=0.0993  volt=0.0100
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== withder_4 (epoch 60) ===
Val  |V| MAE=0.010332  angle MAE=0.861810  Re/Im MSE(nrm)=0.100682  r2_mean=0.8105  r2_min=0.0803  worst_mae=0.047581  tot=0.2914  volt=0.1007
Test  |V| MAE=0.010781  angle MAE=0.886990  Re/Im MSE(nrm)=0.108248  r2_mean=0.7998  r2_min=0.0195  worst_mae=0.050632  tot=0.3026  volt=0.1082
  periodic checkpoint -> /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:4835: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):
/content/GNN-Sandia/train_da_gps_multitask_complex_voltage_gine.py:5583: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if use_amp else contextlib.nullcontext()):

=== Final evaluation (best checkpoint) ===

=== train_pool_eval (best epoch 60) ===
Val  |V| MAE=0.004283  angle MAE=0.223505  Re/Im MSE(nrm)=0.016583  r2_mean=0.9227  r2_min=0.1353  worst_mae=0.015678  tot=0.1131  volt=0.0166
Test  |V| MAE=0.004365  angle MAE=0.226651  Re/Im MSE(nrm)=0.017455  r2_mean=0.9166  r2_min=0.1056  worst_mae=0.016050  tot=0.1148  volt=0.0175

=== nobess_40 (best epoch 60) ===
Val  |V| MAE=0.003785  angle MAE=0.170949  Re/Im MSE(nrm)=0.009658  r2_mean=0.9319  r2_min=0.1398  worst_mae=0.013052  tot=0.0984  volt=0.0097
Test  |V| MAE=0.003835  angle MAE=0.172160  Re/Im MSE(nrm)=0.009963  r2_mean=0.9263  r2_min=0.1127  worst_mae=0.013196  tot=0.0993  volt=0.0100

=== withder_4 (best epoch 60) ===
Val  |V| MAE=0.010332  angle MAE=0.861810  Re/Im MSE(nrm)=0.100682  r2_mean=0.8105  r2_min=0.0804  worst_mae=0.047581  tot=0.2914  volt=0.1007
Test  |V| MAE=0.010781  angle MAE=0.886990  Re/Im MSE(nrm)=0.108248  r2_mean=0.7998  r2_min=0.0195  worst_mae=0.050632  tot=0.3026  volt=0.1082

=== Before / after (init checkpoint vs fine-tuned best) ===
split  metric                     baseline        after        delta     status
val    |V| MAE                    0.004859     0.004283    -0.000576   improved
val    angle MAE                  0.414008     0.223505    -0.190503   improved
val    Re/Im MSE(nrm)             0.047581     0.016583    -0.030999   improved
val    r2_mean                      0.9053       0.9227      +0.0174   improved
val    r2_min                       0.1192       0.1353      +0.0160   improved
val    worst_mae                  0.017596     0.015678    -0.001918   improved
val    tot                          0.1790       0.1131      -0.0659   improved
val    volt                         0.0476       0.0166      -0.0310   improved
test   |V| MAE                    0.004957     0.004365    -0.000592   improved
test   angle MAE                  0.421952     0.226651    -0.195301   improved
test   Re/Im MSE(nrm)             0.049633     0.017455    -0.032178   improved
test   r2_mean                      0.8994       0.9166      +0.0172   improved
test   r2_min                       0.0932       0.1056      +0.0124   improved
test   worst_mae                  0.017805     0.016050    -0.001755   improved
test   tot                          0.1823       0.1148      -0.0675   improved
test   volt                         0.0496       0.0175      -0.0322   improved
Val |V| MAE=0.004283  angle MAE=0.223505  Re/Im MSE(nrm)=0.016583
Test |V| MAE=0.004365  angle MAE=0.226651  Re/Im MSE(nrm)=0.017455  cap_BCE=0.073533  reg_CE=0.893350  reg_acc=0.6252  meta_aux_MSE(nrm)=0.011891  meta_aux_MSE(raw)=10251.187495  time=14366.0s
[da_gps chunk_parent] Test per-head cap_BCE:
  cap_capbank0a_n_steps_on=0.022417
  cap_capbank0b_n_steps_on=0.028647
  cap_capbank0c_n_steps_on=0.056118
  cap_capbank1a_n_steps_on=0.043836
  cap_capbank1b_n_steps_on=0.056490
  cap_capbank1c_n_steps_on=0.111553
  cap_capbank2a_n_steps_on=0.113609
  cap_capbank2b_n_steps_on=0.119056
  cap_capbank2c_n_steps_on=0.183602
  cap_capbank3_n_steps_on=0.000000
[da_gps chunk_parent] Test per-head reg_MSE / reg_MAE (nrm / tap pu):
  reg_feeder_rega_tap_pu: MSE nrm=0.311623 pu=0.311623  MAE nrm=0.271618 pu=0.271618
  reg_feeder_regb_tap_pu: MSE nrm=0.325649 pu=0.325649  MAE nrm=0.281742 pu=0.281742
  reg_feeder_regc_tap_pu: MSE nrm=0.330284 pu=0.330284  MAE nrm=0.261495 pu=0.261495
  reg_vreg2_a_tap_pu: MSE nrm=0.994633 pu=0.994633  MAE nrm=0.511648 pu=0.511648
  reg_vreg2_b_tap_pu: MSE nrm=1.357605 pu=1.357605  MAE nrm=0.638249 pu=0.638249
  reg_vreg2_c_tap_pu: MSE nrm=1.714843 pu=1.714843  MAE nrm=0.757653 pu=0.757653
  reg_vreg3_a_tap_pu: MSE nrm=0.617758 pu=0.617758  MAE nrm=0.354555 pu=0.354555
  reg_vreg3_b_tap_pu: MSE nrm=0.953775 pu=0.953775  MAE nrm=0.517380 pu=0.517380
  reg_vreg3_c_tap_pu: MSE nrm=1.484083 pu=1.484083  MAE nrm=0.685206 pu=0.685206
  reg_vreg4_a_tap_pu: MSE nrm=0.826320 pu=0.826320  MAE nrm=0.433589 pu=0.433589
  reg_vreg4_b_tap_pu: MSE nrm=0.861690 pu=0.861690  MAE nrm=0.472619 pu=0.472619
  reg_vreg4_c_tap_pu: MSE nrm=1.130260 pu=1.130260  MAE nrm=0.620198 pu=0.620198
[da_gps chunk_parent] Test per-head meta_aux_MSE (nrm / raw):
  pv_pv2_p_post_kw: nrm=0.002961  raw=487.102014
  pv_pv2_q_post_kvar: nrm=0.039684  raw=807.397038
  p_loss_total_post_kw: nrm=0.002465  raw=6195.808585
  q_loss_total_post_kvar: nrm=0.002452  raw=33514.441674
Saved /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/da_gps_multitask_best.pt

Fine-tune completed.
Run dir: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052
Checkpoint (best): /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/da_gps_multitask_best.pt
Checkpoint (last): /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
Regulator classes: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/reg_class_tables.json
Report: /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/da_gps_report.json

=== Inference / warm-start ===
RUN_DIR     = /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052
CHECKPOINT  = /content/drive/MyDrive/datasets_gnn2/runs/da_gps_finetune_withder_l2_h96_regce_20260710_011052/training_last.pt
Point cell 9 bootstrap or daily-compare at the paths above after training.

In [7]:
# Fine-tuned checkpoint warmstart band (self-contained)
%matplotlib inline
import os
import sys
from pathlib import Path

# ── find repo (Colab clone, GNN2_REPO_ROOT, Windows path, or cwd) ─────────────
def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "nonunique_notebook_bootstrap.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone to /content/GNN2 first. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )

REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Reload after `git pull` on Colab so you get p_pv_kw fix, wide warm-starts, discrete aux metrics
for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_warmstart_band_daily",
    "nonunique_opendss_daily",
    "nonunique_da_gps_daily_compare",
    "run_da_gps_daily_opendss_compare",
    "compare_gnn_inference_utils",
    "nonunique_notebook_bootstrap",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook
from nonunique_opendss_daily import DailySimConfig
from nonunique_da_gps_warmstart_band_daily import run_da_gps_warmstart_band_daily

# ── checkpoint source ─────────────────────────────────────────────────────────
# Default: shipped CCE baseline (h=96, L=2, reg CE, 4 meta-aux heads)
USE_FINETUNED_CHECKPOINT = True
# Local fine-tune run (also try Drive path on Colab). Toggle True to use it here,
# or run the appended fine-tune warmstart cell at the end of this notebook.
FINETUNE_RUN_DIR = Path(
    r"C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints"
    r"\da_gps_finetune_withder_l2_h96_regce_20260710_011052"
)
if not FINETUNE_RUN_DIR.is_dir():
    _drive_ft = Path(
        "/content/drive/MyDrive/datasets_gnn2/runs/"
        "da_gps_finetune_withder_l2_h96_regce_20260710_011052"
    )
    if _drive_ft.is_dir():
        FINETUNE_RUN_DIR = _drive_ft

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="auto",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
)

if USE_FINETUNED_CHECKPOINT:
    finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
    ckpt = finetune_run / "da_gps_multitask_best.pt"
    if not ckpt.is_file():
        ckpt = finetune_run / "training_last.pt"
    if not ckpt.is_file():
        raise FileNotFoundError(f"No checkpoint in {finetune_run}")
    boot = boot.__class__(
        **{
            **boot.__dict__,
            "run_dir": finetune_run,
            "checkpoint": ckpt,
        }
    )
    print(f"[checkpoint] fine-tuned: {ckpt}")

# ── knobs (same on Colab and local) ───────────────────────────────────────────
INCLUDE_DER      = True
DER_MAX_KW       = 500.0
DER_MAX_KVAR     = 50.0
# Comma-separated: total P/Q split equally across buses, then equally across 3 phases (GNN).
# OpenDSS: one 3ph Generator per bus with that bus's P/Q share. Q/P = DER_MAX_KVAR/DER_MAX_KW.
DER_BUS = "m1142818,r42246"


N_WARM_STARTS    = 5                 # more starts → wider cloud (slower)
WARM_START_MODE  = "wide"             # "uniform" | "corners" | "wide"
WARM_START_RANDOMIZE_STATIC_CAPS = False  # True = wilder cap bands
WARMSTART_SEED   = 42

STEP_MIN         = 5
DAILY_STRESS     = 0.0
SCENARIO_SCALE   = 1.0
REF_SAMPLE_INDEX = 0

PLOT_ALL_CACHE_NODES = True
PLOT_ALL_MAX_NODES   = 10              # 0 = all cache∩circuit nodes
PLOT_REG_CAP         = True
PLOT_META_AUX        = True
PLOT_WARMSTART_LINES = True
SHOW_INLINE          = False
VOLTAGE_PLOT_DPI     = 96
GNN_BATCH_STEPS      = None           # or 8; also env GNN_BATCH_STEPS

# Distinct OUT_DIR so fine-tune outputs do not overwrite the CCE warmstart run.
finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
OUT_DIR = finetune_run / "warmstart_band_daily_finetune"

# ── run ─────────────────────────────────────────────────────────────────────
cfg = DailySimConfig(
    step_min=STEP_MIN,
    include_der=INCLUDE_DER,
    der_nominal_kw=float(DER_MAX_KW if INCLUDE_DER else 0.0),
    der_nominal_kvar=float(DER_MAX_KVAR if INCLUDE_DER else 0.0),
    der_bus=str(DER_BUS),
    der_profile_csv=boot.der_profile,
    da_gps_run_dir=boot.run_dir,
    da_gps_cache_pt=boot.cache_pt,
    da_gps_checkpoint=boot.checkpoint,
)

print(
    f"env={'Colab' if boot.on_colab else 'local'}  device={boot.device}\n"
    f"checkpoint={boot.checkpoint}\n"
    f"DER={'ON' if INCLUDE_DER else 'OFF'}  "
    f"warm_starts={N_WARM_STARTS}  mode={WARM_START_MODE!r}  step_min={STEP_MIN}"
)

result = run_da_gps_warmstart_band_daily(
    cfg,
    n_warm_starts=N_WARM_STARTS,
    warm_start_mode=WARM_START_MODE,
    warm_start_randomize_static_caps=WARM_START_RANDOMIZE_STATIC_CAPS,
    seed=WARMSTART_SEED,
    load_profile_path=boot.load_profile,
    pv_profile_path=boot.irr_profile,
    ref_sample_index=REF_SAMPLE_INDEX,
    scenario_scale=SCENARIO_SCALE,
    daily_stress=DAILY_STRESS,
    plot_all_cache_nodes=PLOT_ALL_CACHE_NODES,
    plot_all_max_nodes=PLOT_ALL_MAX_NODES,
    out_dir=OUT_DIR,
    voltage_plot_dpi=VOLTAGE_PLOT_DPI,
    plot_reg_cap=PLOT_REG_CAP,
    plot_meta_aux=PLOT_META_AUX,
    plot_warmstart_lines=PLOT_WARMSTART_LINES,
    show=SHOW_INLINE,
    device=boot.device,
    gnn_batch_steps=GNN_BATCH_STEPS,
)

inside = result["da_gps_inside_band_frac"]
proximity = result["da_gps_cloud_proximity"]
set_dist = result["da_gps_set_distance"]
outside_dist = result["da_gps_mean_outside_distance"]
aggregated = result["da_gps_aggregated"]

_GROUP_UNITS = {
    "voltage": "pu",
    "regulator": "tap steps",
    "capacitor": "cap steps ON",
    "meta_aux": "pu/kW/kvar",
}

print("\nOutputs:", result["out_dir"])
print(f"Nodes: {len(result['collect_nodes'])}")

print("\n=== Aggregated warm-start band metrics (all devices) ===")
for group, stats in aggregated.items():
    n = int(stats.get("n_devices", 0))
    frac = float(stats.get("mean_inside_band_frac", float("nan")))
    prox = float(stats.get("mean_cloud_proximity", float("nan")))
    sdist = float(stats.get("mean_set_distance", float("nan")))
    odist = float(stats.get("mean_outside_distance", float("nan")))
    unit = _GROUP_UNITS.get(group, "")
    frac_s = f"{100.0 * frac:.1f}%" if frac == frac else "n/a"
    prox_s = f"{prox:.3f}" if prox == prox else "n/a"
    sdist_s = f"{sdist:.4g}" if sdist == sdist else "n/a"
    odist_s = f"{odist:.4g}" if odist == odist else "n/a"
    print(
        f"  [{group}] n={n}  inside={frac_s}  "
        f"cloud_proximity={prox_s}  set_distance={sdist_s}  "
        f"mean_outside_distance={odist_s} {unit}"
    )

print("\n=== Per-device breakdown ===")
for group, unit in (("regulator", "tap steps"), ("capacitor", "cap steps ON"), ("meta_aux", "pu/kW/kvar")):
    ib = inside.get(group) or {}
    pr = proximity.get(group) or {}
    sd = set_dist.get(group) or {}
    od = outside_dist.get(group) or {}
    if not ib:
        continue
    print(f"[{group}]")
    for name in sorted(ib):
        print(
            f"  {name}: inside={100.0 * ib[name]:.1f}%  "
            f"proximity={pr.get(name, float('nan')):.3f}  "
            f"set dist={sd.get(name, float('nan')):.4g}  "
            f"outside dist={od.get(name, float('nan')):.4g} {unit}"
        )

{
    "aggregated": aggregated,
    "inside_band_frac": inside,
    "cloud_proximity": proximity,
    "set_distance": set_dist,
    "mean_outside_distance": outside_dist,
}


[bootstrap] env=local  repo=C:\Users\alita\OneDrive\Desktop\GNN2
[bootstrap] device=cpu  cache=run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=C:\Users\alita\OneDrive\Desktop\GNN2\warmstart_band_runs\20260710_081821
[checkpoint] fine-tuned: C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_finetune_withder_l2_h96_regce_20260710_011052\da_gps_multitask_best.pt
env=local  device=cpu
checkpoint=C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_finetune_withder_l2_h96_regce_20260710_011052\da_gps_multitask_best.pt
DER=ON  warm_starts=5  mode='wide'  step_min=5
DA-GPS warm-start band daily (OpenDSS snapshot + N random controller inits/step)
  step_min=5 min, npts=288, n_warm_starts=5
  warm_start_mode='wide'  randomize_static_caps=False
  load/PV profiles: C:\Use

{'aggregated': {'voltage': {'n_devices': 3817,
   'mean_inside_band_frac': 0.6628796975518877,
   'mean_cloud_proximity': 0.7819965105328228,
   'mean_set_distance': 0.003160642935093779,
   'mean_outside_distance': 0.005479870333501089},
  'regulator': {'n_devices': 12,
   'mean_inside_band_frac': 0.5960648148148148,
   'mean_cloud_proximity': 0.6663997685980306,
   'mean_set_distance': 0.9262152777777778,
   'mean_outside_distance': 1.8761359924221956},
  'capacitor': {'n_devices': 10,
   'mean_inside_band_frac': 0.9961805555555555,
   'mean_cloud_proximity': 0.9975856506433631,
   'mean_set_distance': 0.0038194444444444448,
   'mean_outside_distance': 0.2},
  'meta_aux': {'n_devices': 4,
   'mean_inside_band_frac': 0.24479166666666666,
   'mean_cloud_proximity': 0.24861675558390853,
   'mean_set_distance': 487.35479819370164,
   'mean_outside_distance': 488.9867537929703}},
 'inside_band_frac': {'voltage': {'_hvmv_sub_lsb.1': 0.71875,
   '_hvmv_sub_lsb.2': 0.7326388888888888,
   '_h

## Multi-scenario warm-start band evaluation (metrics only)

Mix-and-match **battery capacity**, **DER bus location(s)**, and **load/PV profile pairs** to get more reliable **aggregated cloud metrics** than a single fixed DER + day-004 run. **No plotting** — this cell only prints / saves metrics.

### What is sampled (each of `N_SCENARIOS`)
| Knob | Source / default |
|------|------------------|
| `#` of DER buses | Uniform in `[N_BUSES_MIN, N_BUSES_MAX]` (default 1–3; same as with-BESS dataset gen) |
| Bus names | Sampled from `bess_candidate_buses_scattered_3ph_mv.csv` (72 three-phase MV buses shipped in the repo; same list used for with-BESS dataset gen) |
| Capacity | `CAPACITY_MODE="kw_uniform"`: peak `DER_MAX_KW` ∈ `[DER_KW_MIN, DER_KW_MAX]`; `Q = q_frac × |P|` with `q_frac` ∈ `[Q_FRAC_MIN, Q_FRAC_MAX]` (upper = dataset `bess_q_frac_max=0.44`). Set `CAPACITY_MODE="dataset_mva"` to sample total MVA ~ N(4.0, 0.1) like dataset gen, then `kW = MVA×1000`. |
| Load/PV pair | One of the 4 representative pairs `load_day_00i` / `irr_day_00i` (`i=1..4`) |
| DER injection schedule | Random among available `battery_arbitrage_der_injection*.csv` under `a representativ days/` (scaled by sampled capacity). Set `FORCE_DER_PROFILE` to pin one file. |

### Inputs you set
- **Model / checkpoint** — same bootstrap as the single-scenario cell (`USE_FINETUNED_CHECKPOINT`, `FINETUNE_RUN_DIR`, or shipped CCE).
- **`N_SCENARIOS`** — number of mix-and-match draws (start with 2–3 for a smoke test).
- **`N_WARM_STARTS`** — warm-starts per timestep (default **3**; raise toward 5 for wider clouds, slower).
- **`STEP_MIN`** — use `15` or `60` for faster smoke; `5` matches the full single-scenario day.

### Outputs
- Per scenario: `aggregated` voltage / regulator / capacitor / meta_aux metrics from `run_da_gps_warmstart_band_daily` (`plot_*=False`, `write_voltage_pngs=False`).
- Across scenarios: **mean ± std** and **median** of each aggregated metric.
- JSON: `{OUT_DIR}/multi_scenario_band_metrics.json` with per-scenario config + metrics and `summary_across_scenarios`.

### Runtime note
Each scenario runs a full OpenDSS warm-start band + GNN day. Expect on the order of **minutes per scenario** at `STEP_MIN=5` / `N_WARM_STARTS=3` (CPU slower than Colab GPU). Start with `N_SCENARIOS=2` smoke, then scale up.

The existing single-scenario warm-start cell above is **unchanged**.


In [ ]:
# Multi-scenario warm-start band eval (metrics only) — self-contained
%matplotlib inline
import os
import sys
from pathlib import Path

def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "multi_scenario_warmstart_band_eval.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found (need multi_scenario_warmstart_band_eval.py). "
        "On Colab: git pull in /content/GNN2. Locally: set GNN2_REPO_ROOT."
    )

REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_warmstart_band_daily",
    "nonunique_opendss_daily",
    "nonunique_notebook_bootstrap",
    "multi_scenario_warmstart_band_eval",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook
from multi_scenario_warmstart_band_eval import (
    MultiScenarioEvalConfig,
    run_multi_scenario_warmstart_band_eval,
)

# ── checkpoint (same pattern as single-scenario cell) ─────────────────────────
USE_FINETUNED_CHECKPOINT = True
FINETUNE_RUN_DIR = Path(
    r"C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints"
    r"\da_gps_finetune_withder_l2_h96_regce_20260710_011052"
)
if not FINETUNE_RUN_DIR.is_dir():
    _drive_ft = Path(
        "/content/drive/MyDrive/datasets_gnn2/runs/"
        "da_gps_finetune_withder_l2_h96_regce_20260710_011052"
    )
    if _drive_ft.is_dir():
        FINETUNE_RUN_DIR = _drive_ft

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="auto",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
)

if USE_FINETUNED_CHECKPOINT:
    finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
    ckpt = finetune_run / "da_gps_multitask_best.pt"
    if not ckpt.is_file():
        ckpt = finetune_run / "training_last.pt"
    if not ckpt.is_file():
        raise FileNotFoundError(f"No checkpoint in {finetune_run}")
    boot = boot.__class__(
        **{
            **boot.__dict__,
            "run_dir": finetune_run,
            "checkpoint": ckpt,
        }
    )
    print(f"[checkpoint] fine-tuned: {ckpt}")

# ── multi-scenario knobs ──────────────────────────────────────────────────────
N_SCENARIOS = 2              # smoke: 2–3; production cloud: 16–32+
N_WARM_STARTS = 3            # modest for speed; raise to 5 for wider bands
MULTI_SEED = 42
STEP_MIN = 5                 # 15 or 60 for faster smoke

N_BUSES_MIN, N_BUSES_MAX = 1, 3
CAPACITY_MODE = "kw_uniform"  # or "dataset_mva" (~N(4.0, 0.1) MVA → kW)
DER_KW_MIN, DER_KW_MAX = 250.0, 2000.0
Q_FRAC_MIN, Q_FRAC_MAX = 0.05, 0.44

# Optional: pin DER schedule; None = sample among available injection CSVs
FORCE_DER_PROFILE = None  # e.g. boot.day1 / "battery_arbitrage_der_injection.csv"

# Optional: override candidate CSV (default: repo/bess_candidate_buses_scattered_3ph_mv.csv)
CANDIDATE_CSV = None

OUT_DIR = Path(boot.checkpoint).resolve().parent / "multi_scenario_warmstart_band"

eval_cfg = MultiScenarioEvalConfig(
    n_scenarios=int(N_SCENARIOS),
    seed=int(MULTI_SEED),
    n_warm_starts=int(N_WARM_STARTS),
    step_min=int(STEP_MIN),
    n_buses_min=int(N_BUSES_MIN),
    n_buses_max=int(N_BUSES_MAX),
    capacity_mode=str(CAPACITY_MODE),
    der_kw_min=float(DER_KW_MIN),
    der_kw_max=float(DER_KW_MAX),
    q_frac_min=float(Q_FRAC_MIN),
    q_frac_max=float(Q_FRAC_MAX),
    candidate_csv=CANDIDATE_CSV,
    der_profile_csv=FORCE_DER_PROFILE,
    out_dir=OUT_DIR,
    device=boot.device,
)

print(
    f"env={'Colab' if boot.on_colab else 'local'}  device={boot.device}\n"
    f"checkpoint={boot.checkpoint}\n"
    f"N_SCENARIOS={N_SCENARIOS}  N_WARM_STARTS={N_WARM_STARTS}  "
    f"capacity={CAPACITY_MODE!r}  step_min={STEP_MIN}"
)

multi = run_multi_scenario_warmstart_band_eval(boot=boot, cfg=eval_cfg)

summary = multi["summary_across_scenarios"]
print("\n=== Across-scenario mean ± std (inside-band %) ===")
for group, metrics in summary.items():
    frac = metrics.get("mean_inside_band_frac") or {}
    if not frac:
        continue
    mean, std, med = frac.get("mean"), frac.get("std"), frac.get("median")

    def _pct(x):
        return f"{100.0 * x:.1f}%" if x == x else "n/a"

    print(f"  [{group}] {_pct(mean)} ± {_pct(std)}  (median {_pct(med)}, n={frac.get('n')})")

metrics_json = Path(OUT_DIR) / "multi_scenario_band_metrics.json"
print(f"\nWrote: {metrics_json}")
print(f"wall_s_total={multi['wall_s_total']:.1f}  mean/scenario={multi['wall_s_mean_per_scenario']:.1f}s")

multi["summary_across_scenarios"]


## Multi-scenario warm-start band evaluation — **CCE baseline** (metrics only)

This variant evaluates the **shipped CCE baseline** checkpoint (not the with-DER fine-tune).

Same mix-and-match protocol as the fine-tune multi-scenario cell above (battery capacity, DER bus location(s), load/PV pairs), but `USE_FINETUNED_CHECKPOINT = False` and the checkpoint is the architecture-search CCE run. Outputs go to a distinct `multi_scenario_warmstart_band_cce` folder so fine-tune results are not overwritten.


In [ ]:
# Multi-scenario warm-start band eval (metrics only) — CCE baseline
%matplotlib inline
import os
import sys
from pathlib import Path

def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "multi_scenario_warmstart_band_eval.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found (need multi_scenario_warmstart_band_eval.py). "
        "On Colab: git pull in /content/GNN2. Locally: set GNN2_REPO_ROOT."
    )

REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_warmstart_band_daily",
    "nonunique_opendss_daily",
    "nonunique_notebook_bootstrap",
    "multi_scenario_warmstart_band_eval",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook
from multi_scenario_warmstart_band_eval import (
    MultiScenarioEvalConfig,
    run_multi_scenario_warmstart_band_eval,
)

# ── checkpoint: shipped CCE baseline (NOT with-DER fine-tune) ─────────────────
USE_FINETUNED_CHECKPOINT = False  # use CCE baseline
# Absolute run dir (optional clarity; bootstrap uses DEFAULT_CHECKPOINT_SUBDIR):
# C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
FINETUNE_RUN_DIR = Path(
    r"C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints"
    r"\da_gps_finetune_withder_l2_h96_regce_20260710_011052"
)
if not FINETUNE_RUN_DIR.is_dir():
    _drive_ft = Path(
        "/content/drive/MyDrive/datasets_gnn2/runs/"
        "da_gps_finetune_withder_l2_h96_regce_20260710_011052"
    )
    if _drive_ft.is_dir():
        FINETUNE_RUN_DIR = _drive_ft

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="auto",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
)

if USE_FINETUNED_CHECKPOINT:
    finetune_run = Path(FINETUNE_RUN_DIR).expanduser().resolve()
    ckpt = finetune_run / "da_gps_multitask_best.pt"
    if not ckpt.is_file():
        ckpt = finetune_run / "training_last.pt"
    if not ckpt.is_file():
        raise FileNotFoundError(f"No checkpoint in {finetune_run}")
    boot = boot.__class__(
        **{
            **boot.__dict__,
            "run_dir": finetune_run,
            "checkpoint": ckpt,
        }
    )
    print(f"[checkpoint] fine-tuned: {ckpt}")
else:
    print(f"[checkpoint] CCE baseline: {boot.checkpoint}")

# ── multi-scenario knobs ──────────────────────────────────────────────────────
N_SCENARIOS = 2              # smoke: 2–3; production cloud: 16–32+
N_WARM_STARTS = 3            # modest for speed; raise to 5 for wider bands
MULTI_SEED = 42
STEP_MIN = 5                 # 15 or 60 for faster smoke

N_BUSES_MIN, N_BUSES_MAX = 1, 3
CAPACITY_MODE = "kw_uniform"  # or "dataset_mva" (~N(4.0, 0.1) MVA → kW)
DER_KW_MIN, DER_KW_MAX = 250.0, 2000.0
Q_FRAC_MIN, Q_FRAC_MAX = 0.05, 0.44

# Optional: pin DER schedule; None = sample among available injection CSVs
FORCE_DER_PROFILE = None  # e.g. boot.day1 / "battery_arbitrage_der_injection.csv"

# Optional: override candidate CSV (default: repo/bess_candidate_buses_scattered_3ph_mv.csv)
CANDIDATE_CSV = None

OUT_DIR = Path(boot.checkpoint).resolve().parent / "multi_scenario_warmstart_band_cce"

eval_cfg = MultiScenarioEvalConfig(
    n_scenarios=int(N_SCENARIOS),
    seed=int(MULTI_SEED),
    n_warm_starts=int(N_WARM_STARTS),
    step_min=int(STEP_MIN),
    n_buses_min=int(N_BUSES_MIN),
    n_buses_max=int(N_BUSES_MAX),
    capacity_mode=str(CAPACITY_MODE),
    der_kw_min=float(DER_KW_MIN),
    der_kw_max=float(DER_KW_MAX),
    q_frac_min=float(Q_FRAC_MIN),
    q_frac_max=float(Q_FRAC_MAX),
    candidate_csv=CANDIDATE_CSV,
    der_profile_csv=FORCE_DER_PROFILE,
    out_dir=OUT_DIR,
    device=boot.device,
)

print(
    f"env={'Colab' if boot.on_colab else 'local'}  device={boot.device}\n"
    f"checkpoint={boot.checkpoint}\n"
    f"USE_FINETUNED_CHECKPOINT={USE_FINETUNED_CHECKPOINT}\n"
    f"N_SCENARIOS={N_SCENARIOS}  N_WARM_STARTS={N_WARM_STARTS}  "
    f"capacity={CAPACITY_MODE!r}  step_min={STEP_MIN}"
)

multi = run_multi_scenario_warmstart_band_eval(boot=boot, cfg=eval_cfg)

summary = multi["summary_across_scenarios"]
print("\n=== Across-scenario mean ± std (inside-band %) ===")
for group, metrics in summary.items():
    frac = metrics.get("mean_inside_band_frac") or {}
    if not frac:
        continue
    mean, std, med = frac.get("mean"), frac.get("std"), frac.get("median")

    def _pct(x):
        return f"{100.0 * x:.1f}%" if x == x else "n/a"

    print(f"  [{group}] {_pct(mean)} ± {_pct(std)}  (median {_pct(med)}, n={frac.get('n')})")

metrics_json = Path(OUT_DIR) / "multi_scenario_band_metrics.json"
print(f"\nWrote: {metrics_json}")
print(f"wall_s_total={multi['wall_s_total']:.1f}  mean/scenario={multi['wall_s_mean_per_scenario']:.1f}s")

multi["summary_across_scenarios"]


[bootstrap] env=local  repo=C:\Users\alita\OneDrive\Desktop\GNN2
[bootstrap] device=cpu  cache=run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=C:\Users\alita\OneDrive\Desktop\GNN2\warmstart_band_runs\20260712_114743
[checkpoint] CCE baseline: C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE\training_last.pt
env=local  device=cpu
checkpoint=C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE\training_last.pt
USE_FINETUNED_CHECKPOINT=False
N_SCENARIOS=2  N_WARM_STARTS=3  capacity='kw_uniform'  step_min=5
Multi-scenario warm-start band eval (metrics only, no plots)
  n_scenarios=2  seed=42  n_warm_starts=3
  capacity_mode='kw_uniform'  buses=[1,3]
  ca

## Fair DA-GPS vs OpenDSS timing (Method B, metrics only)

**Purpose:** apples-to-apples wall-clock comparison of a full day on the **same** device (CPU and/or CUDA), using the Method B **separate-loop** path — not warm-start, not interleaved solve+infer.

**Recommended:** run this cell on **Colab CPU** for stable OpenDSS vs GNN-CPU numbers (less noisy than a local PC). On Colab, `DEVICES` defaults to `["cpu"]`. GPU timing is optional — set `DEVICES = ["cuda"]` or `["cpu", "cuda"]` to override.

### Method B loops (what this cell times)
1. **OpenDSS daily truth** — compile once, then `npts` sequential native `Solve()` (warm-start carry-forward). Times: loop wall, mean Solve/step, mean collect V/step.
2. **DA-GPS GNN-only** — separate loop: feature build + forward for each display step (setup once + per-step). Times: deployment wall (`setup + n × per_step`), wrapper wall (includes import/cache load).

Use `mode="da_gps_daily_compare"` / `run_opendss_daily_truth` + `run_da_gps_predictions`. Do **not** use `mode="warmstart"` for these numbers — that is a different experiment (DA-GPS warm-start vs cold OpenDSS).

### Included
- OpenDSS: sequential daily Solve (+ voltage collect on monitor nodes).
- GNN: one-time setup + feature + infer for each of `npts` steps (`STEP_MIN=5` → 288).
- Optional CUDA opts on GPU: TF32 matmul, deferred D2H (`GNN_DEFER_D2H`), CUDA graphs when enabled.

### Excluded / not comparable here
- Checkpoint training, tensor-cache build, heavy voltage PNG / plot suites (this cell skips `plot_all`).
- Method A interleaved warm-start solve path (different work).
- Comparing a CPU Method A number (~30s) to a GPU Method B number (~1.4s) without matching device.

### Knobs
- `DEVICES` — default: Colab → `["cpu"]`; local → `["cpu","cuda"]` if CUDA else `["cpu"]`. Override with `["cpu"]`, `["cuda"]`, or `["cpu","cuda"]`.
- `STEP_MIN=5` (288 steps), `INCLUDE_DER=False` by default.
- Checkpoint: shipped CCE `da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE`.

### Warm-start cell on CPU?
Yes — set `device="cpu"` (or `"auto"` when no GPU). Warm-start is still a **different** experiment from this timing summary.


In [ ]:
# Fair DA-GPS vs OpenDSS timing (Method B) — metrics only, no plots
# mode=da_gps_daily_compare path (NOT warmstart)
%matplotlib inline
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch


def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "nonunique_notebook_bootstrap.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone/pull /content/GNN2. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )


REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_daily_compare",
    "nonunique_opendss_daily",
    "nonunique_notebook_bootstrap",
    "run_da_gps_daily_opendss_compare",
    "compare_gnn_inference_utils",
    "compare_mv_daily_timing",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook, resolve_inference_device
from nonunique_opendss_daily import DailySimConfig, resolve_monitor_nodes
from nonunique_da_gps_daily_compare import (
    filter_cache_nodes_on_circuit,
    load_cache_node_order,
    print_timing_summary,
    run_da_gps_predictions,
    run_opendss_daily_truth,
)
from compare_opendss_snapshot_helpers import (
    compile_and_bind_parity_daily_opendss,
    prepare_parity_profiles,
)

# ── knobs ────────────────────────────────────────────────────────────────────
STEP_MIN = 5                      # 5 -> 288 steps/day; 60 -> 24
INCLUDE_DER = False               # timing default: no DER
DER_MAX_KW = 1000.0
DER_MAX_KVAR = 500.0
DER_BUS = "m1142818,r42246"
SCENARIO_SCALE = 1.0
DAILY_STRESS = 0.0
REF_SAMPLE_INDEX = 0
PLOT_ALL_CACHE_NODES = False      # monitor nodes only (faster collect)

# Device list (DEVICES=None → auto policy below).
# Override examples (explicit list always wins):
#   DEVICES = ["cpu"]
#   DEVICES = ["cuda"]
#   DEVICES = ["cpu", "cuda"]
#   DEVICE = "auto"  (expanded below if DEVICES is None)
DEVICE = "auto"
DEVICES = None  # if None: Colab→["cpu"]; local→["cpu","cuda"] if CUDA else ["cpu"]

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="cpu",  # resolve per-run below; bootstrap only needs paths
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
    out_parent=REPO / "da_gps_fair_timing_runs",
    run_tag=datetime.now().strftime("%Y%m%d_%H%M%S"),
)

_on_colab = bool(getattr(boot, "on_colab", False)) or Path("/content").is_dir()
_devices_explicit = DEVICES is not None

if DEVICES is None:
    if str(DEVICE).strip().lower() in ("", "auto", "default"):
        if _on_colab:
            # Colab CPU only — stable DSS vs GNN-CPU fair timing (user request)
            DEVICES = ["cpu"]
        else:
            DEVICES = ["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]
    else:
        DEVICES = [str(DEVICE).strip().lower()]
else:
    DEVICES = [str(d).strip().lower() for d in DEVICES]

if _on_colab and not _devices_explicit and DEVICES == ["cpu"]:
    print("env=Colab → using Colab CPU for fair timing")
elif _on_colab and _devices_explicit:
    print(f"env=Colab → DEVICES override respected: {DEVICES}")
else:
    print(f"env={'Colab' if _on_colab else 'local'} → DEVICES={DEVICES}")

OUT_ROOT = Path(boot.out_dir)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = DailySimConfig(
    step_min=STEP_MIN,
    include_der=INCLUDE_DER,
    der_nominal_kw=float(DER_MAX_KW if INCLUDE_DER else 0.0),
    der_nominal_kvar=float(DER_MAX_KVAR if INCLUDE_DER else 0.0),
    der_bus=str(DER_BUS),
    der_profile_csv=boot.der_profile,
    da_gps_run_dir=boot.run_dir,
    da_gps_cache_pt=boot.cache_pt,
    da_gps_checkpoint=boot.checkpoint,
)

print("=" * 72)
print("Fair DA-GPS vs OpenDSS timing (Method B, metrics only — no plot_all)")
print(f"  checkpoint={boot.checkpoint}")
print(f"  step_min={STEP_MIN}  npts={cfg.npts}  INCLUDE_DER={INCLUDE_DER}")
print(f"  DEVICES={DEVICES}")
print(f"  out_root={OUT_ROOT}")
print("=" * 72)
print(
    "NOTE: mode=da_gps_daily_compare (separate DSS daily + GNN-only loops). "
    "mode=warmstart is a different experiment."
)

# Probe circuit once so monitor/cache∩circuit node lists are valid for DSS + GNN.
_profiles_probe = prepare_parity_profiles(
    boot.load_profile,
    boot.irr_profile,
    npts=cfg.npts,
    step_min=float(cfg.step_min),
    daily_stress=DAILY_STRESS,
)
compile_and_bind_parity_daily_opendss(
    _profiles_probe, npts=cfg.npts, step_min=float(cfg.step_min)
)
monitor_nodes = resolve_monitor_nodes(cfg.monitor_candidates)
cache_nodes = load_cache_node_order(cfg.da_gps_cache_pt)
cache_on_circuit = filter_cache_nodes_on_circuit(cache_nodes)
collect_nodes = cache_on_circuit if PLOT_ALL_CACHE_NODES else monitor_nodes
print(f"[fair_timing] monitor nodes ({len(monitor_nodes)}): {monitor_nodes}")
print(
    f"[fair_timing] collect nodes: {len(collect_nodes)} "
    f"({'cache∩circuit' if PLOT_ALL_CACHE_NODES else 'monitors only'})"
)

# OpenDSS truth is device-independent — run once, reuse for each GNN device.
print("\n--- OpenDSS daily truth (once) ---")
dss = run_opendss_daily_truth(
    cfg,
    load_csv=boot.load_profile,
    irr_csv=boot.irr_profile,
    plot_nodes=collect_nodes,
    scenario_scale=SCENARIO_SCALE,
    daily_stress=DAILY_STRESS,
)

all_summaries: list[dict] = []

for dev_req in DEVICES:
    dev = resolve_inference_device(dev_req)
    if str(dev_req).lower() in ("cuda", "gpu") and dev != "cuda":
        print(f"[fair_timing] skip requested {dev_req!r}: CUDA not available")
        continue

    # CPU: disable CUDA graphs; CUDA: keep Colab-style defaults (graphs + defer D2H).
    if dev == "cpu":
        os.environ["GNN_CUDA_GRAPHS"] = "0"
    else:
        os.environ.setdefault("GNN_CUDA_GRAPHS", "1")
        os.environ.setdefault("GNN_DEFER_D2H", "1")

    print(f"\n--- DA-GPS GNN-only @ device={dev} (requested={dev_req}) ---")
    gnn = run_da_gps_predictions(
        cfg,
        collect_nodes,
        device=dev,
        ref_sample_index=REF_SAMPLE_INDEX,
        scenario_scale=SCENARIO_SCALE,
    )

    # Light MAE on overlapping monitor nodes (no figures).
    npts = int(cfg.npts)
    mae_vals = []
    for j, nk in enumerate(collect_nodes):
        if nk not in monitor_nodes and str(nk).lower() not in {m.lower() for m in monitor_nodes}:
            continue
        v_dss = np.asarray(dss["v_dss"][:npts, j], dtype=np.float64)
        v = gnn["voltages"].get(nk)
        if v is None:
            v = gnn["voltages"].get(str(nk).lower())
        if v is None:
            continue
        v_gnn = np.asarray(v, dtype=np.float64)[:npts]
        m = np.isfinite(v_dss) & np.isfinite(v_gnn)
        if m.any():
            mae_vals.append(float(np.mean(np.abs(v_dss[m] - v_gnn[m]))))
    mae = float(np.mean(mae_vals)) if mae_vals else float("nan")
    if np.isfinite(mae):
        print(f"[fair_timing] monitor |V| MAE vs DA-GPS: {mae:.6f} pu")

    print_timing_summary(
        n_ok=int(dss["converged"].sum()),
        npts=npts,
        step_min=int(cfg.step_min),
        dss_wall_s=float(dss["total_wall_s"]),
        dss_solve_s=dss["solve_s"],
        dss_collect_s=dss["collect_s"],
        gnn_wall_s=float(gnn["gnn_wall_s"]),
        gnn_setup_once_s=gnn.get("gnn_setup_once_s"),
        gnn_per_step_s=gnn.get("gnn_per_step_s"),
        gnn_total_wall_s=gnn.get("gnn_total_wall_s"),
        gnn_n_ok=gnn.get("n_ok"),
        gnn_grid=gnn.get("gnn_grid"),
    )

    gnn_deploy = gnn.get("gnn_total_wall_s")
    if gnn_deploy is None and gnn.get("gnn_setup_once_s") is not None and gnn.get("gnn_per_step_s") is not None:
        gnn_deploy = float(gnn["gnn_setup_once_s"]) + float(gnn.get("n_ok") or npts) * float(
            gnn["gnn_per_step_s"]
        )
    dss_wall = float(dss["total_wall_s"])
    speedup = (
        dss_wall / float(gnn_deploy)
        if gnn_deploy is not None and float(gnn_deploy) > 1e-9
        else float("nan")
    )

    summary = {
        "mode": "da_gps_daily_compare",
        "metrics_only": True,
        "step_min": int(cfg.step_min),
        "npts": npts,
        "include_der": bool(INCLUDE_DER),
        "device_requested": str(dev_req),
        "device": str(dev),
        "checkpoint": str(boot.checkpoint),
        "cache_pt": str(boot.cache_pt),
        "load_profile": str(boot.load_profile),
        "pv_profile": str(boot.irr_profile),
        "n_collect_nodes": len(collect_nodes),
        "monitor_nodes": list(monitor_nodes),
        "monitor_mae_pu": mae,
        "dss_wall_s": dss_wall,
        "dss_converged": int(dss["converged"].sum()),
        "gnn_wall_s": float(gnn["gnn_wall_s"]),
        "gnn_setup_once_s": gnn.get("gnn_setup_once_s"),
        "gnn_per_step_s": gnn.get("gnn_per_step_s"),
        "gnn_total_wall_s": gnn.get("gnn_total_wall_s"),
        "gnn_deployment_wall_s": gnn_deploy,
        "wall_speedup_dss_over_gnn_deploy": speedup,
        "env": {
            "GNN_CUDA_GRAPHS": os.environ.get("GNN_CUDA_GRAPHS"),
            "GNN_DEFER_D2H": os.environ.get("GNN_DEFER_D2H"),
            "GNN_TF32": os.environ.get("GNN_TF32", "(default on)"),
            "GNN_BATCH_STEPS": os.environ.get("GNN_BATCH_STEPS"),
        },
    }
    all_summaries.append(summary)

    out_dev = OUT_ROOT / f"timing_{dev}"
    out_dev.mkdir(parents=True, exist_ok=True)
    out_json = out_dev / "fair_timing_summary.json"
    out_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"[fair_timing] wrote {out_json}")

bundle = {
    "mode": "da_gps_daily_compare",
    "metrics_only": True,
    "step_min": int(cfg.step_min),
    "npts": int(cfg.npts),
    "devices": DEVICES,
    "dss_wall_s": float(dss["total_wall_s"]),
    "per_device": all_summaries,
}
bundle_path = OUT_ROOT / "fair_timing_all_devices.json"
bundle_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")

print("\n=== Fair timing rollup ===")
print(f"  OpenDSS daily loop wall: {dss['total_wall_s']:.2f} s  (shared)")
for s in all_summaries:
    gd = s.get("gnn_deployment_wall_s")
    gd_s = f"{float(gd):.2f} s" if gd is not None else "n/a"
    sp = s.get("wall_speedup_dss_over_gnn_deploy")
    sp_s = f"{float(sp):.2f}x" if sp is not None and np.isfinite(sp) else "n/a"
    print(f"  device={s['device']}: GNN deploy={gd_s}  speedup(DSS/GNN)={sp_s}")
print(f"Outputs: {OUT_ROOT}")
print(f"Bundle:  {bundle_path}")


[bootstrap] env=local  repo=C:\Users\alita\OneDrive\Desktop\GNN2
[bootstrap] device=cpu  cache=run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=C:\Users\alita\OneDrive\Desktop\GNN2\da_gps_fair_timing_runs\20260712_135104
Fair DA-GPS vs OpenDSS timing (Method B, metrics only — no plot_all)
  checkpoint=C:\Users\alita\OneDrive\Desktop\GNN2\gnn2_architecture_search\attention checkpoints\da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE\training_last.pt
  step_min=5  npts=288  INCLUDE_DER=False
  DEVICES=['cpu']
  out_root=C:\Users\alita\OneDrive\Desktop\GNN2\da_gps_fair_timing_runs\20260712_135104
NOTE: mode=da_gps_daily_compare (separate DSS daily + GNN-only loops). mode=warmstart is a different experiment.
[fair_timing] monitor nodes (4): ['190-8593.1', '190-8581.1', '190-7361.1', 'l2973163.2']
[fair_timing] collect nodes: 4 (m

on cpu

[bootstrap] env=Colab  repo=/content/GNN-Sandia
[bootstrap] device=cpu  cache=run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_200621
env=Colab → using Colab CPU for fair timing
========================================================================
Fair DA-GPS vs OpenDSS timing (Method B, metrics only — no plot_all)
  checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
  step_min=5  npts=288  INCLUDE_DER=False
  DEVICES=['cpu']
  out_root=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_200621
========================================================================
NOTE: mode=da_gps_daily_compare (separate DSS daily + GNN-only loops). mode=warmstart is a different experiment.
[fair_timing] monitor nodes (4): ['190-8593.1', '190-8581.1', '190-7361.1', 'l2973163.2']
[fair_timing] collect nodes: 4 (monitors only)

--- OpenDSS daily truth (once) ---
[da_gps_daily_compare] OpenDSS daily: compile once -> 288 sequential Solve() (step_min=5 min, mode=daily, warm-start carry-forward)
[da_gps_daily_compare] profiles: load=load_day_004.csv irr=irr_day_004.csv scenario_scale=1
[da_gps_daily_compare] OpenDSS daily finished: 288/288 converged, wall=3.39s, Solution.Mode() after last step='1'

--- DA-GPS GNN-only @ device=cpu (requested=cpu) ---
[DA-GPS] device=cpu (CUDA not available)
[da_gps_daily_compare] DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
[da_gps_daily] no da_gps_report.json or da_gps_run_manifest.json; synthesizing recipe from checkpoint training_last.pt
[da_gps_daily] report_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] norm_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] run_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[da_gps_daily] using edge CSV: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
[da_gps_daily] checkpoint=training_last.pt: loading **best_model_state_dict**
[da_gps_daily] regulator head: CE (12 per-device classifiers); tap plots use reg_class_values.pt, not reg_mean/std denorm.
[da_gps_daily] meta_aux denorm stats (mean | std) per head — GNN curves use pred*std+mean:
  [0] pv_pv2_p_post_kw: mean=342.225  std=405.614
  [1] pv_pv2_q_post_kvar: mean=-32.5123  std=142.637
  [2] p_loss_total_post_kw: mean=1721.15  std=1585.47
  [3] q_loss_total_post_kvar: mean=3806.96  std=3696.87
[da_gps_daily] checkpoint backbone: GINE (train_da_gps_multitask_complex_voltage_gine.py)
[DA-GPS] device=cpu (CUDA not available)
[da_gps_daily] torch.compile disabled (GNN_TORCH_COMPILE='0')
[da_gps_daily] mv↔sx mapping: 1177 rules from /content/GNN-Sandia/8500-node/mv_x_sx_node_mapping_8500.csv
[da_gps_daily] OpenDSS compile (static maps only): /content/GNN-Sandia/8500 nodes with solar unbalanced/Master-PV2MW-inv.dss (cwd /content/GNN-Sandia/8500 nodes with solar unbalanced); **no** per-step Solve()
[da_gps_daily] Feeder master is **fixed** to the solar-unbalanced tree above (``Master-PV2MW-inv.dss``). Load / irradiance / DER profile CLI args do not change which DSS master is redirected.
[da_gps_daily] daily load profile (override): /content/GNN-Sandia/8500 nodes with solar unbalanced/5minDayShape.csv
[da_gps_daily] PV irradiance mult m_irr (col 2, training ``m_pv_t``): /content/GNN-Sandia/8500 nodes with solar unbalanced/irr_day_001.csv  span=[0,1]  mean(m_irr)=0.4415 (``p_pv_kw`` = Pmpp0×m_irr[i]; DSS ``Pmpp`` = Pmpp0×m_irr[i] under snapshot)
[da_gps_daily] p_pv_kw (GNN x): ``pmpp_set×m_pv_t`` style = Pmpp0×m_irr[i] per PV, equal split over element phases (2 PVsystems, 6 bus-phase terms) — ``_apply_snapshot_with_pv`` / ``_collect_pv_maps``; DSS ``Pmpp`` = Pmpp0×m_irr[i] (unity IrradDay001 under snapshot mode).
[da_gps_daily] stress: daily_stress=0 scenario_scale=1 clip=[0.1,3]  m_raw∈[0.5681,1.0526] m_eff∈[0.5681,1.0526]
[da_gps_daily] GNN-only path: node index from tensor cache (3817 bus.phase rows); skipping OpenDSS AllNodeNames + daily Solve loop.
[da_gps_daily] multitask daily series: cap=10 reg=12 meta_aux=4 (OpenDSS: 10 caps, 12 regs, 2 PVsystems)
[da_gps_daily] DSS PVSystem names (for pv_*_p_post_kw meta match): ['pv1', 'pv2']
[da_gps_daily] GNN setup once: 0.4856s (model+norm tensors+static feature tables+cuda-graph/warmup=0.2020s)  defer_d2h=False cuda_graphs=False
[da_gps_daily] feature diag (first step): nodes with |P|+|Q|>1e-3: 1177/3817
[24/288] GNN-only: feat=0.00s gnn=0.65s
[48/288] GNN-only: feat=0.01s gnn=1.31s
[72/288] GNN-only: feat=0.01s gnn=1.96s
[96/288] GNN-only: feat=0.02s gnn=2.63s
[120/288] GNN-only: feat=0.02s gnn=3.43s
[144/288] GNN-only: feat=0.02s gnn=4.17s
[168/288] GNN-only: feat=0.03s gnn=4.93s
[192/288] GNN-only: feat=0.04s gnn=5.61s
[216/288] GNN-only: feat=0.04s gnn=6.28s
[240/288] GNN-only: feat=0.04s gnn=6.92s
[264/288] GNN-only: feat=0.05s gnn=7.60s
[288/288] GNN-only: feat=0.05s gnn=8.25s
[da_gps_daily] voltages_only: returning 4 node |V| series (4 with finite values), reg_tap_pu (288, 12) (3456 finite), cap_sigmoid (288, 10) (2880 finite); skipped plots/CSV exports.
[fair_timing] monitor |V| MAE vs DA-GPS: 0.003915 pu

[da_gps_daily_compare] === Timing summary ===
  Display grid: step_min=5 min, npts=288
  DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
  OpenDSS Solve() wall:         288 × 0.0114s = 3.2875s  (compile-once not timed; loop wall incl. collect = 3.39 s)
  OpenDSS mean Solve() / step:  11.42 ms  (converged steps only in denominator)
  OpenDSS mean collect V/step:  0.32 ms
  DA-GPS deployment wall:       0.4856s + 288 × 0.0288s = 8.7861s  (setup + feature + infer per displayed step)
  DA-GPS wrapper wall:          18.53 s  (outer cell: import + cache/ckpt load before timed setup)
  DA-GPS wrapper overhead:      9.74 s  (wrapper − deployment; mostly one-time import/load)
  DA-GPS mean wall / display step: 30.51 ms  (deployment wall / npts=288; compare to OpenDSS mean Solve above)
  Wall speedup (OpenDSS/GNN deploy): 0.39x  (dss loop wall / gnn_total_wall_s; excludes wrapper import)
[fair_timing] wrote /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_200621/timing_cpu/fair_timing_summary.json

=== Fair timing rollup ===
  OpenDSS daily loop wall: 3.39 s  (shared)
  device=cpu: GNN deploy=8.79 s  speedup(DSS/GNN)=0.39x
Outputs: /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_200621
Bundle:  /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_200621/fair_timing_all_devices.json

on gpu

[bootstrap] env=Colab  repo=/content/GNN-Sandia
[bootstrap] device=cpu  cache=run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254
========================================================================
Fair DA-GPS vs OpenDSS timing (Method B, metrics only — no plot_all)
  checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
  step_min=5  npts=288  INCLUDE_DER=False
  DEVICES=['cpu', 'cuda']
  out_root=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254
========================================================================
NOTE: mode=da_gps_daily_compare (separate DSS daily + GNN-only loops). mode=warmstart is a different experiment.
[fair_timing] monitor nodes (4): ['190-8593.1', '190-8581.1', '190-7361.1', 'l2973163.2']
[fair_timing] collect nodes: 4 (monitors only)

--- OpenDSS daily truth (once) ---
[da_gps_daily_compare] OpenDSS daily: compile once -> 288 sequential Solve() (step_min=5 min, mode=daily, warm-start carry-forward)
[da_gps_daily_compare] profiles: load=load_day_004.csv irr=irr_day_004.csv scenario_scale=1
[da_gps_daily_compare] OpenDSS daily finished: 288/288 converged, wall=4.49s, Solution.Mode() after last step='1'

--- DA-GPS GNN-only @ device=cpu (requested=cpu) ---
[DA-GPS] device=cpu
[da_gps_daily_compare] DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
[da_gps_daily] no da_gps_report.json or da_gps_run_manifest.json; synthesizing recipe from checkpoint training_last.pt
[da_gps_daily] report_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] norm_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] run_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[da_gps_daily] using edge CSV: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
[da_gps_daily] checkpoint=training_last.pt: loading **best_model_state_dict**
[da_gps_daily] regulator head: CE (12 per-device classifiers); tap plots use reg_class_values.pt, not reg_mean/std denorm.
[da_gps_daily] meta_aux denorm stats (mean | std) per head — GNN curves use pred*std+mean:
  [0] pv_pv2_p_post_kw: mean=342.225  std=405.614
  [1] pv_pv2_q_post_kvar: mean=-32.5123  std=142.637
  [2] p_loss_total_post_kw: mean=1721.15  std=1585.47
  [3] q_loss_total_post_kvar: mean=3806.96  std=3696.87
[da_gps_daily] checkpoint backbone: GINE (train_da_gps_multitask_complex_voltage_gine.py)
[DA-GPS] device=cpu
[da_gps_daily] torch.compile disabled (GNN_TORCH_COMPILE='0')
[da_gps_daily] mv↔sx mapping: 1177 rules from /content/GNN-Sandia/8500-node/mv_x_sx_node_mapping_8500.csv
[da_gps_daily] OpenDSS compile (static maps only): /content/GNN-Sandia/8500 nodes with solar unbalanced/Master-PV2MW-inv.dss (cwd /content/GNN-Sandia/8500 nodes with solar unbalanced); **no** per-step Solve()
[da_gps_daily] Feeder master is **fixed** to the solar-unbalanced tree above (``Master-PV2MW-inv.dss``). Load / irradiance / DER profile CLI args do not change which DSS master is redirected.
[da_gps_daily] daily load profile (override): /content/GNN-Sandia/8500 nodes with solar unbalanced/5minDayShape.csv
[da_gps_daily] PV irradiance mult m_irr (col 2, training ``m_pv_t``): /content/GNN-Sandia/8500 nodes with solar unbalanced/irr_day_001.csv  span=[0,1]  mean(m_irr)=0.4415 (``p_pv_kw`` = Pmpp0×m_irr[i]; DSS ``Pmpp`` = Pmpp0×m_irr[i] under snapshot)
[da_gps_daily] p_pv_kw (GNN x): ``pmpp_set×m_pv_t`` style = Pmpp0×m_irr[i] per PV, equal split over element phases (2 PVsystems, 6 bus-phase terms) — ``_apply_snapshot_with_pv`` / ``_collect_pv_maps``; DSS ``Pmpp`` = Pmpp0×m_irr[i] (unity IrradDay001 under snapshot mode).
[da_gps_daily] stress: daily_stress=0 scenario_scale=1 clip=[0.1,3]  m_raw∈[0.5681,1.0526] m_eff∈[0.5681,1.0526]
[da_gps_daily] GNN-only path: node index from tensor cache (3817 bus.phase rows); skipping OpenDSS AllNodeNames + daily Solve loop.
[da_gps_daily] multitask daily series: cap=10 reg=12 meta_aux=4 (OpenDSS: 10 caps, 12 regs, 2 PVsystems)
[da_gps_daily] DSS PVSystem names (for pv_*_p_post_kw meta match): ['pv1', 'pv2']
[da_gps_daily] GNN setup once: 0.6600s (model+norm tensors+static feature tables+cuda-graph/warmup=0.1507s)  defer_d2h=False cuda_graphs=False
[da_gps_daily] feature diag (first step): nodes with |P|+|Q|>1e-3: 1177/3817
[24/288] GNN-only: feat=0.01s gnn=0.44s
[48/288] GNN-only: feat=0.01s gnn=0.89s
[72/288] GNN-only: feat=0.02s gnn=1.32s
[96/288] GNN-only: feat=0.03s gnn=1.75s
[120/288] GNN-only: feat=0.03s gnn=2.18s
[144/288] GNN-only: feat=0.04s gnn=2.63s
[168/288] GNN-only: feat=0.04s gnn=3.07s
[192/288] GNN-only: feat=0.05s gnn=3.52s
[216/288] GNN-only: feat=0.05s gnn=3.95s
[240/288] GNN-only: feat=0.06s gnn=4.39s
[264/288] GNN-only: feat=0.06s gnn=4.84s
[288/288] GNN-only: feat=0.07s gnn=5.36s
[da_gps_daily] voltages_only: returning 4 node |V| series (4 with finite values), reg_tap_pu (288, 12) (3456 finite), cap_sigmoid (288, 10) (2880 finite); skipped plots/CSV exports.
[fair_timing] monitor |V| MAE vs DA-GPS: 0.003915 pu

[da_gps_daily_compare] === Timing summary ===
  Display grid: step_min=5 min, npts=288
  DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
  OpenDSS Solve() wall:         288 × 0.0151s = 4.3503s  (compile-once not timed; loop wall incl. collect = 4.49 s)
  OpenDSS mean Solve() / step:  15.11 ms  (converged steps only in denominator)
  OpenDSS mean collect V/step:  0.44 ms
  DA-GPS deployment wall:       0.6600s + 288 × 0.0189s = 6.0899s  (setup + feature + infer per displayed step)
  DA-GPS wrapper wall:          6.56 s  (outer cell: import + cache/ckpt load before timed setup)
  DA-GPS wrapper overhead:      0.47 s  (wrapper − deployment; mostly one-time import/load)
  DA-GPS mean wall / display step: 21.15 ms  (deployment wall / npts=288; compare to OpenDSS mean Solve above)
  Wall speedup (OpenDSS/GNN deploy): 0.74x  (dss loop wall / gnn_total_wall_s; excludes wrapper import)
[fair_timing] wrote /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254/timing_cpu/fair_timing_summary.json

--- DA-GPS GNN-only @ device=cuda (requested=cuda) ---
[da_gps_daily_compare] DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
[da_gps_daily] no da_gps_report.json or da_gps_run_manifest.json; synthesizing recipe from checkpoint training_last.pt
[da_gps_daily] report_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] norm_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] run_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[da_gps_daily] using edge CSV: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
[da_gps_daily] checkpoint=training_last.pt: loading **best_model_state_dict**
[da_gps_daily] regulator head: CE (12 per-device classifiers); tap plots use reg_class_values.pt, not reg_mean/std denorm.
[da_gps_daily] meta_aux denorm stats (mean | std) per head — GNN curves use pred*std+mean:
  [0] pv_pv2_p_post_kw: mean=342.225  std=405.614
  [1] pv_pv2_q_post_kvar: mean=-32.5123  std=142.637
  [2] p_loss_total_post_kw: mean=1721.15  std=1585.47
  [3] q_loss_total_post_kvar: mean=3806.96  std=3696.87
[da_gps_daily] checkpoint backbone: GINE (train_da_gps_multitask_complex_voltage_gine.py)
[GNN] CUDA inference opts: TF32 disabled (GNN_TF32=0, float32_matmul_precision=highest)
[da_gps_daily] torch.compile disabled (GNN_TORCH_COMPILE='0')
[da_gps_daily] mv↔sx mapping: 1177 rules from /content/GNN-Sandia/8500-node/mv_x_sx_node_mapping_8500.csv
[da_gps_daily] OpenDSS compile (static maps only): /content/GNN-Sandia/8500 nodes with solar unbalanced/Master-PV2MW-inv.dss (cwd /content/GNN-Sandia/8500 nodes with solar unbalanced); **no** per-step Solve()
[da_gps_daily] Feeder master is **fixed** to the solar-unbalanced tree above (``Master-PV2MW-inv.dss``). Load / irradiance / DER profile CLI args do not change which DSS master is redirected.
[da_gps_daily] daily load profile (override): /content/GNN-Sandia/8500 nodes with solar unbalanced/5minDayShape.csv
[da_gps_daily] PV irradiance mult m_irr (col 2, training ``m_pv_t``): /content/GNN-Sandia/8500 nodes with solar unbalanced/irr_day_001.csv  span=[0,1]  mean(m_irr)=0.4415 (``p_pv_kw`` = Pmpp0×m_irr[i]; DSS ``Pmpp`` = Pmpp0×m_irr[i] under snapshot)
[da_gps_daily] p_pv_kw (GNN x): ``pmpp_set×m_pv_t`` style = Pmpp0×m_irr[i] per PV, equal split over element phases (2 PVsystems, 6 bus-phase terms) — ``_apply_snapshot_with_pv`` / ``_collect_pv_maps``; DSS ``Pmpp`` = Pmpp0×m_irr[i] (unity IrradDay001 under snapshot mode).
[da_gps_daily] stress: daily_stress=0 scenario_scale=1 clip=[0.1,3]  m_raw∈[0.5681,1.0526] m_eff∈[0.5681,1.0526]
[da_gps_daily] GNN-only path: node index from tensor cache (3817 bus.phase rows); skipping OpenDSS AllNodeNames + daily Solve loop.
[da_gps_daily] multitask daily series: cap=10 reg=12 meta_aux=4 (OpenDSS: 10 caps, 12 regs, 2 PVsystems)
[da_gps_daily] DSS PVSystem names (for pv_*_p_post_kw meta match): ['pv1', 'pv2']
[da_gps_daily] GNN setup once: 0.3258s (model+norm tensors+static feature tables+cuda-graph/warmup=0.0073s)  defer_d2h=True cuda_graphs=False
[da_gps_daily] feature diag (first step): nodes with |P|+|Q|>1e-3: 1177/3817
[24/288] GNN-only: feat=0.00s gnn=0.15s
[48/288] GNN-only: feat=0.01s gnn=0.30s
[72/288] GNN-only: feat=0.01s gnn=0.44s
[96/288] GNN-only: feat=0.02s gnn=0.59s
[120/288] GNN-only: feat=0.02s gnn=0.74s
[144/288] GNN-only: feat=0.02s gnn=0.88s
[168/288] GNN-only: feat=0.03s gnn=1.03s
[192/288] GNN-only: feat=0.03s gnn=1.18s
[216/288] GNN-only: feat=0.04s gnn=1.34s
[240/288] GNN-only: feat=0.04s gnn=1.48s
[264/288] GNN-only: feat=0.04s gnn=1.63s
[288/288] GNN-only: feat=0.05s gnn=1.78s
[da_gps_daily] voltages_only: returning 4 node |V| series (4 with finite values), reg_tap_pu (288, 12) (3456 finite), cap_sigmoid (288, 10) (2880 finite); skipped plots/CSV exports.
[fair_timing] monitor |V| MAE vs DA-GPS: 0.003915 pu

[da_gps_daily_compare] === Timing summary ===
  Display grid: step_min=5 min, npts=288
  DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
  OpenDSS Solve() wall:         288 × 0.0151s = 4.3503s  (compile-once not timed; loop wall incl. collect = 4.49 s)
  OpenDSS mean Solve() / step:  15.11 ms  (converged steps only in denominator)
  OpenDSS mean collect V/step:  0.44 ms
  DA-GPS deployment wall:       0.3258s + 288 × 0.0064s = 2.1693s  (setup + feature + infer per displayed step)
  DA-GPS wrapper wall:          2.63 s  (outer cell: import + cache/ckpt load before timed setup)
  DA-GPS wrapper overhead:      0.46 s  (wrapper − deployment; mostly one-time import/load)
  DA-GPS mean wall / display step: 7.53 ms  (deployment wall / npts=288; compare to OpenDSS mean Solve above)
  Wall speedup (OpenDSS/GNN deploy): 2.07x  (dss loop wall / gnn_total_wall_s; excludes wrapper import)
[fair_timing] wrote /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254/timing_cuda/fair_timing_summary.json

=== Fair timing rollup ===
  OpenDSS daily loop wall: 4.49 s  (shared)
  device=cpu: GNN deploy=6.09 s  speedup(DSS/GNN)=0.74x
  device=cuda: GNN deploy=2.17 s  speedup(DSS/GNN)=2.07x
Outputs: /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254
Bundle:  /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_194254/fair_timing_all_devices.json

## TF32 vs FP32 GPU inference timing (Method B, GNN-only)

**Purpose:** wall-clock comparison of one full day of **DA-GPS GNN-only** inference on CUDA with TensorFloat-32 **on** vs true FP32 (TF32 **off**). Focus is GNN deployment wall — OpenDSS is optional (run once) only for shared monitor `|V|` MAE.

### Method B path
Uses the same separate-loop GNN path as the fair timing cell: `run_da_gps_predictions` / `mode=da_gps_daily_compare` (not warm-start). No OpenDSS required for the speed numbers.

### TF32 toggle (matches `configure_cuda_inference`)
- **TF32 ON:** `GNN_TF32` unset / not `0` → `allow_tf32=True`, `float32_matmul_precision="high"`
- **TF32 OFF / FP32:** `GNN_TF32=0` → `allow_tf32=False`, `float32_matmul_precision="highest"`

### Knobs
- Requires CUDA (raises / skips clearly if no GPU).
- `STEP_MIN=5` (288 steps), `INCLUDE_DER=False`.
- Checkpoint: same CCE bootstrap as fair timing.
- Output: `da_gps_fair_timing_runs/.../tf32_vs_fp32_gpu_timing.json`


In [ ]:
# TF32 vs FP32 GPU one-day GNN inference timing (Method B, GNN-only)
# Reuses fair-timing CCE bootstrap; focus = GNN wall on CUDA
%matplotlib inline
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch


def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "nonunique_notebook_bootstrap.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found. On Colab: clone/pull /content/GNN2. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )


REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

for _m in (
    "nonunique_daily_experiment",
    "nonunique_da_gps_daily_compare",
    "nonunique_opendss_daily",
    "nonunique_notebook_bootstrap",
    "run_da_gps_daily_opendss_compare",
    "compare_gnn_inference_utils",
    "compare_mv_daily_timing",
):
    sys.modules.pop(_m, None)

from compare_gnn_inference_utils import configure_cuda_inference
from nonunique_notebook_bootstrap import bootstrap_warmstart_notebook, resolve_inference_device
from nonunique_opendss_daily import DailySimConfig, resolve_monitor_nodes
from nonunique_da_gps_daily_compare import (
    filter_cache_nodes_on_circuit,
    load_cache_node_order,
    run_da_gps_predictions,
    run_opendss_daily_truth,
)
from compare_opendss_snapshot_helpers import (
    compile_and_bind_parity_daily_opendss,
    prepare_parity_profiles,
)

# ── knobs ────────────────────────────────────────────────────────────────────
STEP_MIN = 5
INCLUDE_DER = False
DER_MAX_KW = 1000.0
DER_MAX_KVAR = 500.0
DER_BUS = "m1142818,r42246"
SCENARIO_SCALE = 1.0
DAILY_STRESS = 0.0
REF_SAMPLE_INDEX = 0
RUN_DSS_FOR_MAE = True  # shared OpenDSS truth once; MAE for both modes

DEFAULT_CHECKPOINT_SUBDIR = (
    "gnn2_architecture_search/attention checkpoints/"
    "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "TF32 vs FP32 timing requires CUDA. No GPU available "
        f"(torch.cuda.is_available()={torch.cuda.is_available()}). "
        "Skip this cell on CPU-only machines."
    )

dev = resolve_inference_device("cuda")
if dev != "cuda":
    raise RuntimeError(
        f"Requested CUDA but resolve_inference_device returned {dev!r}. "
        "TF32 vs FP32 timing requires a CUDA device."
    )

boot = bootstrap_warmstart_notebook(
    repo=REPO,
    device="cuda",
    checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR,
    out_parent=REPO / "da_gps_fair_timing_runs",
    run_tag=datetime.now().strftime("%Y%m%d_%H%M%S") + "_tf32_vs_fp32",
)

OUT_ROOT = Path(boot.out_dir)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = DailySimConfig(
    step_min=STEP_MIN,
    include_der=INCLUDE_DER,
    der_nominal_kw=float(DER_MAX_KW if INCLUDE_DER else 0.0),
    der_nominal_kvar=float(DER_MAX_KVAR if INCLUDE_DER else 0.0),
    der_bus=str(DER_BUS),
    der_profile_csv=boot.der_profile,
    da_gps_run_dir=boot.run_dir,
    da_gps_cache_pt=boot.cache_pt,
    da_gps_checkpoint=boot.checkpoint,
)

print("=" * 72)
print("TF32 vs FP32 GPU GNN-only timing (Method B)")
print(f"  device={dev}  GPU={torch.cuda.get_device_name(0)}")
print(f"  checkpoint={boot.checkpoint}")
print(f"  step_min={STEP_MIN}  npts={cfg.npts}  INCLUDE_DER={INCLUDE_DER}")
print(f"  out_root={OUT_ROOT}")
print("=" * 72)

# Monitor / collect nodes (DSS compile only if MAE requested)
_profiles_probe = prepare_parity_profiles(
    boot.load_profile,
    boot.irr_profile,
    npts=cfg.npts,
    step_min=float(cfg.step_min),
    daily_stress=DAILY_STRESS,
)
compile_and_bind_parity_daily_opendss(
    _profiles_probe, npts=cfg.npts, step_min=float(cfg.step_min)
)
monitor_nodes = resolve_monitor_nodes(cfg.monitor_candidates)
cache_nodes = load_cache_node_order(cfg.da_gps_cache_pt)
cache_on_circuit = filter_cache_nodes_on_circuit(cache_nodes)
collect_nodes = monitor_nodes  # monitors only — no heavy collect
print(f"[tf32_timing] monitor nodes ({len(monitor_nodes)}): {monitor_nodes}")

dss = None
if RUN_DSS_FOR_MAE:
    print("\n--- OpenDSS daily truth (once, for MAE) ---")
    dss = run_opendss_daily_truth(
        cfg,
        load_csv=boot.load_profile,
        irr_csv=boot.irr_profile,
        plot_nodes=collect_nodes,
        scenario_scale=SCENARIO_SCALE,
        daily_stress=DAILY_STRESS,
    )


def _set_tf32_mode(*, enabled: bool) -> None:
    """Match compare_gnn_inference_utils.configure_cuda_inference via GNN_TF32 + backends."""
    if enabled:
        os.environ.pop("GNN_TF32", None)  # default ON in configure_cuda_inference
        # Explicit backends (also set again inside configure_cuda_inference)
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        if hasattr(torch.backends.cuda, "matmul") and hasattr(torch.backends.cuda.matmul, "allow_tf32"):
            torch.backends.cuda.matmul.allow_tf32 = True
        if hasattr(torch.backends.cudnn, "allow_tf32"):
            torch.backends.cudnn.allow_tf32 = True
    else:
        os.environ["GNN_TF32"] = "0"
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("highest")
        if hasattr(torch.backends.cuda, "matmul") and hasattr(torch.backends.cuda.matmul, "allow_tf32"):
            torch.backends.cuda.matmul.allow_tf32 = False
        if hasattr(torch.backends.cudnn, "allow_tf32"):
            torch.backends.cudnn.allow_tf32 = False
    configure_cuda_inference(torch.device("cuda"))
    print(
        f"[tf32_timing] TF32={'ON' if enabled else 'OFF'}  "
        f"GNN_TF32={os.environ.get('GNN_TF32', '(default on)')}  "
        f"matmul.allow_tf32={getattr(getattr(torch.backends.cuda, 'matmul', None), 'allow_tf32', None)}  "
        f"cudnn.allow_tf32={getattr(torch.backends.cudnn, 'allow_tf32', None)}",
        flush=True,
    )


def _monitor_mae(gnn: dict) -> float:
    if dss is None:
        return float("nan")
    npts = int(cfg.npts)
    mae_vals = []
    mon_l = {m.lower() for m in monitor_nodes}
    for j, nk in enumerate(collect_nodes):
        if nk not in monitor_nodes and str(nk).lower() not in mon_l:
            continue
        v_dss = np.asarray(dss["v_dss"][:npts, j], dtype=np.float64)
        v = gnn["voltages"].get(nk)
        if v is None:
            v = gnn["voltages"].get(str(nk).lower())
        if v is None:
            continue
        v_gnn = np.asarray(v, dtype=np.float64)[:npts]
        m = np.isfinite(v_dss) & np.isfinite(v_gnn)
        if m.any():
            mae_vals.append(float(np.mean(np.abs(v_dss[m] - v_gnn[m]))))
    return float(np.mean(mae_vals)) if mae_vals else float("nan")


def _deploy_wall(gnn: dict, npts: int) -> float | None:
    gnn_deploy = gnn.get("gnn_total_wall_s")
    if gnn_deploy is None and gnn.get("gnn_setup_once_s") is not None and gnn.get("gnn_per_step_s") is not None:
        gnn_deploy = float(gnn["gnn_setup_once_s"]) + float(gnn.get("n_ok") or npts) * float(
            gnn["gnn_per_step_s"]
        )
    return float(gnn_deploy) if gnn_deploy is not None else None


# Colab-style CUDA opts (graphs + defer D2H) for both modes
os.environ.setdefault("GNN_CUDA_GRAPHS", "1")
os.environ.setdefault("GNN_DEFER_D2H", "1")

mode_results: dict[str, dict] = {}

for label, tf32_on in (("tf32_on", True), ("fp32_tf32_off", False)):
    print(f"\n--- DA-GPS GNN-only @ cuda  mode={label} ---")
    _set_tf32_mode(enabled=tf32_on)
    gnn = run_da_gps_predictions(
        cfg,
        collect_nodes,
        device="cuda",
        ref_sample_index=REF_SAMPLE_INDEX,
        scenario_scale=SCENARIO_SCALE,
    )
    npts = int(cfg.npts)
    setup_s = gnn.get("gnn_setup_once_s")
    per_step_s = gnn.get("gnn_per_step_s")
    deploy_s = _deploy_wall(gnn, npts)
    mae = _monitor_mae(gnn)
    print(
        f"[tf32_timing] {label}: setup={setup_s} s  per_step={per_step_s} s  "
        f"deploy={deploy_s} s  wrapper={gnn.get('gnn_wall_s')} s"
    )
    if np.isfinite(mae):
        print(f"[tf32_timing] {label}: monitor |V| MAE vs OpenDSS = {mae:.6f} pu")
    mode_results[label] = {
        "tf32_enabled": bool(tf32_on),
        "GNN_TF32": os.environ.get("GNN_TF32", "(default on)"),
        "matmul_allow_tf32": bool(
            getattr(getattr(torch.backends.cuda, "matmul", None), "allow_tf32", False)
        ),
        "cudnn_allow_tf32": bool(getattr(torch.backends.cudnn, "allow_tf32", False)),
        "gnn_setup_once_s": setup_s,
        "gnn_per_step_s": per_step_s,
        "gnn_total_wall_s": gnn.get("gnn_total_wall_s"),
        "gnn_deployment_wall_s": deploy_s,
        "gnn_wall_s": float(gnn["gnn_wall_s"]),
        "n_ok": gnn.get("n_ok"),
        "monitor_mae_pu": mae,
    }

t_on = mode_results["tf32_on"].get("gnn_deployment_wall_s")
t_off = mode_results["fp32_tf32_off"].get("gnn_deployment_wall_s")
speedup_tf32_over_fp32 = (
    float(t_off) / float(t_on)
    if t_on is not None and t_off is not None and float(t_on) > 1e-9
    else float("nan")
)
ratio_fp32_over_tf32 = (
    float(t_on) / float(t_off)
    if t_on is not None and t_off is not None and float(t_off) > 1e-9
    else float("nan")
)

print("\n=== TF32 vs FP32 rollup ===")
for label, r in mode_results.items():
    print(
        f"  {label}: setup={r['gnn_setup_once_s']}  per_step={r['gnn_per_step_s']}  "
        f"deploy={r['gnn_deployment_wall_s']}  mae={r['monitor_mae_pu']}"
    )
print(f"  speedup TF32/FP32 (fp32_wall / tf32_wall): {speedup_tf32_over_fp32}")
print(f"  ratio   TF32/FP32 (tf32_wall / fp32_wall): {ratio_fp32_over_tf32}")

summary = {
    "mode": "da_gps_daily_compare",
    "experiment": "tf32_vs_fp32_gpu",
    "metrics_only": True,
    "device": "cuda",
    "gpu_name": torch.cuda.get_device_name(0),
    "step_min": int(cfg.step_min),
    "npts": int(cfg.npts),
    "include_der": bool(INCLUDE_DER),
    "checkpoint": str(boot.checkpoint),
    "cache_pt": str(boot.cache_pt),
    "monitor_nodes": list(monitor_nodes),
    "dss_wall_s": float(dss["total_wall_s"]) if dss is not None else None,
    "tf32_on": mode_results["tf32_on"],
    "fp32_tf32_off": mode_results["fp32_tf32_off"],
    "speedup_tf32_over_fp32": speedup_tf32_over_fp32,
    "ratio_tf32_wall_over_fp32_wall": ratio_fp32_over_tf32,
    "env": {
        "GNN_CUDA_GRAPHS": os.environ.get("GNN_CUDA_GRAPHS"),
        "GNN_DEFER_D2H": os.environ.get("GNN_DEFER_D2H"),
        "GNN_BATCH_STEPS": os.environ.get("GNN_BATCH_STEPS"),
    },
}

out_json = OUT_ROOT / "tf32_vs_fp32_gpu_timing.json"
out_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")
# Also write a stable sibling path under the fair-timing parent for easy discovery
stable = REPO / "da_gps_fair_timing_runs" / "tf32_vs_fp32_gpu_timing.json"
stable.parent.mkdir(parents=True, exist_ok=True)
stable.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\n[tf32_timing] wrote {out_json}")
print(f"[tf32_timing] wrote {stable}")


[bootstrap] env=Colab  repo=/content/GNN-Sandia
[bootstrap] device=cuda  cache=run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt
[bootstrap] checkpoint=da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[bootstrap] out_dir=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_192622_tf32_vs_fp32
========================================================================
TF32 vs FP32 GPU GNN-only timing (Method B)
  device=cuda  GPU=NVIDIA L4
  checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
  step_min=5  npts=288  INCLUDE_DER=False
  out_root=/content/GNN-Sandia/da_gps_fair_timing_runs/20260712_192622_tf32_vs_fp32
========================================================================
[tf32_timing] monitor nodes (4): ['190-8593.1', '190-8581.1', '190-7361.1', 'l2973163.2']

--- OpenDSS daily truth (once, for MAE) ---
[da_gps_daily_compare] OpenDSS daily: compile once -> 288 sequential Solve() (step_min=5 min, mode=daily, warm-start carry-forward)
[da_gps_daily_compare] profiles: load=load_day_004.csv irr=irr_day_004.csv scenario_scale=1
[da_gps_daily_compare] OpenDSS daily finished: 288/288 converged, wall=4.56s, Solution.Mode() after last step='1'

--- DA-GPS GNN-only @ cuda  mode=tf32_on ---
[GNN] CUDA inference opts: TF32 matmul (float32_matmul_precision=high), cudnn.benchmark=True
[tf32_timing] TF32=ON  GNN_TF32=(default on)  matmul.allow_tf32=True  cudnn.allow_tf32=True
[DA-GPS] device=cuda (GPU: NVIDIA L4)
[da_gps_daily_compare] DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
[da_gps_daily] no da_gps_report.json or da_gps_run_manifest.json; synthesizing recipe from checkpoint training_last.pt
[da_gps_daily] report_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] norm_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] run_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[da_gps_daily] using edge CSV: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
[da_gps_daily] checkpoint=training_last.pt: loading **best_model_state_dict**
[da_gps_daily] regulator head: CE (12 per-device classifiers); tap plots use reg_class_values.pt, not reg_mean/std denorm.
[da_gps_daily] meta_aux denorm stats (mean | std) per head — GNN curves use pred*std+mean:
  [0] pv_pv2_p_post_kw: mean=342.225  std=405.614
  [1] pv_pv2_q_post_kvar: mean=-32.5123  std=142.637
  [2] p_loss_total_post_kw: mean=1721.15  std=1585.47
  [3] q_loss_total_post_kvar: mean=3806.96  std=3696.87
[da_gps_daily] checkpoint backbone: GINE (train_da_gps_multitask_complex_voltage_gine.py)
[DA-GPS] device=cuda (GPU: NVIDIA L4)
[GNN] CUDA inference opts: TF32 matmul (float32_matmul_precision=high), cudnn.benchmark=True
[da_gps_daily] torch.compile disabled (GNN_TORCH_COMPILE='0')
[da_gps_daily] mv↔sx mapping: 1177 rules from /content/GNN-Sandia/8500-node/mv_x_sx_node_mapping_8500.csv
[da_gps_daily] OpenDSS compile (static maps only): /content/GNN-Sandia/8500 nodes with solar unbalanced/Master-PV2MW-inv.dss (cwd /content/GNN-Sandia/8500 nodes with solar unbalanced); **no** per-step Solve()
[da_gps_daily] Feeder master is **fixed** to the solar-unbalanced tree above (``Master-PV2MW-inv.dss``). Load / irradiance / DER profile CLI args do not change which DSS master is redirected.
[da_gps_daily] daily load profile (override): /content/GNN-Sandia/8500 nodes with solar unbalanced/5minDayShape.csv
[da_gps_daily] PV irradiance mult m_irr (col 2, training ``m_pv_t``): /content/GNN-Sandia/8500 nodes with solar unbalanced/irr_day_001.csv  span=[0,1]  mean(m_irr)=0.4415 (``p_pv_kw`` = Pmpp0×m_irr[i]; DSS ``Pmpp`` = Pmpp0×m_irr[i] under snapshot)
[da_gps_daily] p_pv_kw (GNN x): ``pmpp_set×m_pv_t`` style = Pmpp0×m_irr[i] per PV, equal split over element phases (2 PVsystems, 6 bus-phase terms) — ``_apply_snapshot_with_pv`` / ``_collect_pv_maps``; DSS ``Pmpp`` = Pmpp0×m_irr[i] (unity IrradDay001 under snapshot mode).
[da_gps_daily] stress: daily_stress=0 scenario_scale=1 clip=[0.1,3]  m_raw∈[0.5681,1.0526] m_eff∈[0.5681,1.0526]
[da_gps_daily] GNN-only path: node index from tensor cache (3817 bus.phase rows); skipping OpenDSS AllNodeNames + daily Solve loop.
[da_gps_daily] multitask daily series: cap=10 reg=12 meta_aux=4 (OpenDSS: 10 caps, 12 regs, 2 PVsystems)
[da_gps_daily] DSS PVSystem names (for pv_*_p_post_kw meta match): ['pv1', 'pv2']
[GNN] CUDA Graph capture OK (normalize→model→denorm); replay per step after x_t_dev update
[da_gps_daily] GNN setup once: 1.3841s (model+norm tensors+static feature tables+cuda-graph/warmup=0.8652s)  defer_d2h=True cuda_graphs=True
[da_gps_daily] feature diag (first step): nodes with |P|+|Q|>1e-3: 1177/3817
[24/288] GNN-only: feat=0.00s gnn=0.08s
[48/288] GNN-only: feat=0.01s gnn=0.16s
[72/288] GNN-only: feat=0.01s gnn=0.24s
[96/288] GNN-only: feat=0.02s gnn=0.32s
[120/288] GNN-only: feat=0.02s gnn=0.40s
[144/288] GNN-only: feat=0.02s gnn=0.48s
[168/288] GNN-only: feat=0.03s gnn=0.56s
[192/288] GNN-only: feat=0.03s gnn=0.64s
[216/288] GNN-only: feat=0.03s gnn=0.72s
[240/288] GNN-only: feat=0.04s gnn=0.81s
[264/288] GNN-only: feat=0.04s gnn=0.89s
[288/288] GNN-only: feat=0.04s gnn=0.97s
[da_gps_daily] voltages_only: returning 4 node |V| series (4 with finite values), reg_tap_pu (288, 12) (3456 finite), cap_sigmoid (288, 10) (2880 finite); skipped plots/CSV exports.
[tf32_timing] tf32_on: setup=1.3841312049999033 s  per_step=0.003567428701434968 s  deploy=2.411550671013174 s  wrapper=6.477264857999671 s
[tf32_timing] tf32_on: monitor |V| MAE vs OpenDSS = 0.003915 pu

--- DA-GPS GNN-only @ cuda  mode=fp32_tf32_off ---
[GNN] CUDA inference opts: TF32 disabled (GNN_TF32=0, float32_matmul_precision=highest)
[tf32_timing] TF32=OFF  GNN_TF32=0  matmul.allow_tf32=False  cudnn.allow_tf32=False
[da_gps_daily_compare] DA-GPS GNN: 288 forwards @ 5 min (display grid; matches OpenDSS)
[da_gps_daily] no da_gps_report.json or da_gps_run_manifest.json; synthesizing recipe from checkpoint training_last.pt
[da_gps_daily] report_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] norm_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] run_dir=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE
[da_gps_daily] checkpoint=/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints/da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE/training_last.pt
[da_gps_daily] using edge CSV: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
Loading compacted edges: /content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40/run_001_scen_0000_0049_seed_20420233/gnn_edges_phase_static.csv
  directed edges: 15256
[da_gps_daily] checkpoint=training_last.pt: loading **best_model_state_dict**
[da_gps_daily] regulator head: CE (12 per-device classifiers); tap plots use reg_class_values.pt, not reg_mean/std denorm.
[da_gps_daily] meta_aux denorm stats (mean | std) per head — GNN curves use pred*std+mean:
  [0] pv_pv2_p_post_kw: mean=342.225  std=405.614
  [1] pv_pv2_q_post_kvar: mean=-32.5123  std=142.637
  [2] p_loss_total_post_kw: mean=1721.15  std=1585.47
  [3] q_loss_total_post_kvar: mean=3806.96  std=3696.87
[da_gps_daily] checkpoint backbone: GINE (train_da_gps_multitask_complex_voltage_gine.py)
[GNN] CUDA inference opts: TF32 disabled (GNN_TF32=0, float32_matmul_precision=highest)
[da_gps_daily] torch.compile disabled (GNN_TORCH_COMPILE='0')
[da_gps_daily] mv↔sx mapping: 1177 rules from /content/GNN-Sandia/8500-node/mv_x_sx_node_mapping_8500.csv
[da_gps_daily] OpenDSS compile (static maps only): /content/GNN-Sandia/8500 nodes with solar unbalanced/Master-PV2MW-inv.dss (cwd /content/GNN-Sandia/8500 nodes with solar unbalanced); **no** per-step Solve()
[da_gps_daily] Feeder master is **fixed** to the solar-unbalanced tree above (``Master-PV2MW-inv.dss``). Load / irradiance / DER profile CLI args do not change which DSS master is redirected.
[da_gps_daily] daily load profile (override): /content/GNN-Sandia/8500 nodes with solar unbalanced/5minDayShape.csv
[da_gps_daily] PV irradiance mult m_irr (col 2, training ``m_pv_t``): /content/GNN-Sandia/8500 nodes with solar unbalanced/irr_day_001.csv  span=[0,1]  mean(m_irr)=0.4415 (``p_pv_kw`` = Pmpp0×m_irr[i]; DSS ``Pmpp`` = Pmpp0×m_irr[i] under snapshot)
[da_gps_daily] p_pv_kw (GNN x): ``pmpp_set×m_pv_t`` style = Pmpp0×m_irr[i] per PV, equal split over element phases (2 PVsystems, 6 bus-phase terms) — ``_apply_snapshot_with_pv`` / ``_collect_pv_maps``; DSS ``Pmpp`` = Pmpp0×m_irr[i] (unity IrradDay001 under snapshot mode).
[da_gps_daily] stress: daily_stress=0 scenario_scale=1 clip=[0.1,3]  m_raw∈[0.5681,1.0526] m_eff∈[0.5681,1.0526]
[da_gps_daily] GNN-only path: node index from tensor cache (3817 bus.phase rows); skipping OpenDSS AllNodeNames + daily Solve loop.
[da_gps_daily] multitask daily series: cap=10 reg=12 meta_aux=4 (OpenDSS: 10 caps, 12 regs, 2 PVsystems)
[da_gps_daily] DSS PVSystem names (for pv_*_p_post_kw meta match): ['pv1', 'pv2']
[GNN] CUDA Graph capture OK (normalize→model→denorm); replay per step after x_t_dev update
[da_gps_daily] GNN setup once: 0.3450s (model+norm tensors+static feature tables+cuda-graph/warmup=0.0366s)  defer_d2h=True cuda_graphs=True
[da_gps_daily] feature diag (first step): nodes with |P|+|Q|>1e-3: 1177/3817
[24/288] GNN-only: feat=0.00s gnn=0.09s
[48/288] GNN-only: feat=0.01s gnn=0.18s
[72/288] GNN-only: feat=0.01s gnn=0.26s
[96/288] GNN-only: feat=0.01s gnn=0.35s
[120/288] GNN-only: feat=0.02s gnn=0.44s
[144/288] GNN-only: feat=0.02s gnn=0.53s
[168/288] GNN-only: feat=0.03s gnn=0.61s
[192/288] GNN-only: feat=0.03s gnn=0.70s
[216/288] GNN-only: feat=0.03s gnn=0.79s
[240/288] GNN-only: feat=0.04s gnn=0.88s
[264/288] GNN-only: feat=0.04s gnn=0.96s
[288/288] GNN-only: feat=0.04s gnn=1.05s
[da_gps_daily] voltages_only: returning 4 node |V| series (4 with finite values), reg_tap_pu (288, 12) (3456 finite), cap_sigmoid (288, 10) (2880 finite); skipped plots/CSV exports.
[tf32_timing] fp32_tf32_off: setup=0.344998157000191 s  per_step=0.0038501699235951037 s  deploy=1.4538470949955808 s  wrapper=1.9124011879998761 s
[tf32_timing] fp32_tf32_off: monitor |V| MAE vs OpenDSS = 0.003915 pu

=== TF32 vs FP32 rollup ===
  tf32_on: setup=1.3841312049999033  per_step=0.003567428701434968  deploy=2.411550671013174  mae=0.003915155998670008
  fp32_tf32_off: setup=0.344998157000191  per_step=0.0038501699235951037  deploy=1.4538470949955808  mae=0.003915056369028792
  speedup TF32/FP32 (fp32_wall / tf32_wall): 0.6028681513811072
  ratio   TF32/FP32 (tf32_wall / fp32_wall): 1.6587374829954207

[tf32_timing] wrote /content/GNN-Sandia/da_gps_fair_timing_runs/20260712_192622_tf32_vs_fp32/tf32_vs_fp32_gpu_timing.json
[tf32_timing] wrote /content/GNN-Sandia/da_gps_fair_timing_runs/tf32_vs_fp32_gpu_timing.json

## Method A GPU snapshot timing (interleaved OpenDSS + GNN)

**Purpose:** fill the missing **GPU** Method A timing for the interleaved **snapshot** path in `run_da_gps_daily_opendss_compare.py` (past Method A numbers were often **CPU**-only for GNN).

### What Method A times
At each 5-min step: OpenDSS **snapshot** solve + DA-GPS feature build + GNN forward (interleaved in one loop). Look in stdout for **Wall-clock breakdown** and **Deploy speedup**.

### vs Method B (fair-timing cells above)
| | Method A (this cell) | Method B (fair timing) |
|--|--|--|
| Script / path | `run_da_gps_daily_opendss_compare.py` | separate `run_opendss_daily_truth` + `run_da_gps_predictions` |
| OpenDSS | **snapshot** solves per step | native **daily** QSTS march |
| Coupling | interleaved DSS + GNN | separate loops |
| Device note | DSS stays **CPU**; only GNN uses **CUDA** | fair same-device DSS daily vs GNN-only |

### Colab knobs (timing-focused defaults)
- `DEVICE = "cuda"` (required; raises if no GPU)
- `PLOT_ALL_CACHE_NODES = False` — a few `--plot-node` only (set `True` only if you want hours of PNGs)
- `META_DEBUG = False` — quieter logs for timing
- `DER_MAX_KW = 0`
- Checkpoint: CCE `da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE`


In [ ]:
# Method A GPU snapshot timing — run_da_gps_daily_opendss_compare.py (interleaved)
# Colab-ready launcher based on the local Method A cell; DEVICE=cuda required.
# Timing focus: plot-all OFF, META_DEBUG OFF, DER_MAX_KW=0.
%matplotlib inline
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import torch


def _find_gnn2_repo() -> Path:
    cands: list[Path] = []
    env = os.environ.get("GNN2_REPO_ROOT", "").strip()
    if env:
        cands.append(Path(env))
    if Path("/content").is_dir():
        cands.extend([Path("/content/GNN2"), Path("/content/GNN-Sandia")])
    cands.append(Path(r"C:\Users\alita\OneDrive\Desktop\GNN2"))
    cands.append(Path.cwd())
    seen: set[Path] = set()
    for raw in cands:
        p = raw.expanduser().resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "run_da_gps_daily_opendss_compare.py").is_file():
            return p
    raise FileNotFoundError(
        "GNN2 repo not found (need run_da_gps_daily_opendss_compare.py). "
        "On Colab: clone/pull /content/GNN2 or /content/GNN-Sandia. "
        "Locally: set GNN2_REPO_ROOT or run from the repo folder."
    )


REPO = _find_gnn2_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

for _m in (
    "nonunique_notebook_bootstrap",
    "run_da_gps_daily_opendss_compare",
):
    sys.modules.pop(_m, None)

from nonunique_notebook_bootstrap import (  # noqa: E402
    CACHE_CANDIDATES,
    bootstrap_warmstart_notebook,
    is_colab,
    resolve_cache_pt,
)

# ── require CUDA (Method A GPU timing) ───────────────────────────────────────
DEVICE = "cuda"
if not torch.cuda.is_available():
    raise RuntimeError(
        "Method A GPU timing requires CUDA, but torch.cuda.is_available() is False. "
        "On Colab: Runtime → Change runtime type → GPU. "
        "For CPU Method A, use the older launcher with DEVICE='' / omit --device."
    )

CCE_NAME = "da_gps_chunked_l4_mvagg_gine_metaaux_regce_20260516_225149_CCE"
DEFAULT_CHECKPOINT_SUBDIR_CCE = (
    "gnn2_architecture_search/attention checkpoints/" + CCE_NAME
)


def _resolve_cce_run_dir(repo: Path) -> Path:
    cands = [
        repo / DEFAULT_CHECKPOINT_SUBDIR_CCE,
        repo / "gnn2_architecture_search" / "attention checkpoints" / CCE_NAME,
        Path("/content/drive/MyDrive/datasets_gnn2/checkpoints") / CCE_NAME,
        Path("/content/drive/MyDrive/datasets_gnn2") / "attention checkpoints" / CCE_NAME,
    ]
    for p in cands:
        p = p.expanduser().resolve()
        if (p / "training_last.pt").is_file() or (p / "da_gps_multitask_best.pt").is_file():
            return p
    raise FileNotFoundError(
        f"CCE checkpoint folder not found. Tried:\n  " + "\n  ".join(str(c) for c in cands)
    )


def _resolve_cache_pt_colab_local(repo: Path) -> Path:
    # Prefer bootstrap local candidates, then Drive caches used on Colab.
    try:
        return resolve_cache_pt(repo)
    except FileNotFoundError:
        pass
    drive_cands = [
        Path("/content/drive/MyDrive/datasets_gnn2/cache/gnn_only_chunked_mvagg_full_gine")
        / "run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt",
        Path("/content/drive/MyDrive/datasets_gnn2")
        / "run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt",
    ]
    for rel in CACHE_CANDIDATES:
        drive_cands.append(Path("/content/drive/MyDrive/datasets_gnn2") / Path(rel).name)
    for p in drive_cands:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "No DA-GPS cache .pt found under repo or Drive. "
        f"Expected one of {list(CACHE_CANDIDATES)} or a Drive cache copy."
    )


# Bootstrap when CCE lives under the repo (local / cloned). Else resolve Drive paths.
_on_colab = is_colab() or Path("/content").is_dir()
RUN_DIR: Path
CACHE_PT: Path
CHECKPOINT: Path
LOAD_PROFILE_DIR_OR_FILE: Path
PV_IRRADIANCE_DIR_OR_FILE: Path
DER_PROFILE_DIR_OR_FILE: Path

try:
    boot = bootstrap_warmstart_notebook(
        repo=REPO,
        device="cuda",
        checkpoint_subdir=DEFAULT_CHECKPOINT_SUBDIR_CCE,
        out_parent=REPO / "da_gps_method_a_gpu_timing_runs",
        run_tag=datetime.now().strftime("%Y%m%d_%H%M%S"),
    )
    RUN_DIR = Path(boot.run_dir)
    CACHE_PT = Path(boot.cache_pt)
    CHECKPOINT = Path(boot.checkpoint)
    LOAD_PROFILE_DIR_OR_FILE = Path(boot.load_profile)
    PV_IRRADIANCE_DIR_OR_FILE = Path(boot.irr_profile)
    DER_PROFILE_DIR_OR_FILE = Path(boot.der_profile)
    OUT_DIR = Path(boot.out_dir)
except FileNotFoundError as _boot_err:
    print(f"[method_a_gpu] bootstrap fallback ({_boot_err})")
    RUN_DIR = _resolve_cce_run_dir(REPO)
    ck_best = RUN_DIR / "da_gps_multitask_best.pt"
    ck_last = RUN_DIR / "training_last.pt"
    CHECKPOINT = ck_best if ck_best.is_file() else ck_last
    CACHE_PT = _resolve_cache_pt_colab_local(REPO)
    day1 = REPO / "a representativ days"
    LOAD_PROFILE_DIR_OR_FILE = day1 / "load_day_004.csv"
    PV_IRRADIANCE_DIR_OR_FILE = day1 / "irr_day_004.csv"
    DER_PROFILE_DIR_OR_FILE = day1 / "battery_arbitrage_der_injection.csv"
    OUT_DIR = REPO / "da_gps_method_a_gpu_timing_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")

EDGE_CSV = None

LOAD_PROFILE_FILENAME = "load_day_004.csv"
PV_IRRADIANCE_FILENAME = "irr_day_004.csv"
DER_PROFILE_FILENAME = "battery_arbitrage_der_injection.csv"
DER_MAX_KW = 0
DER_BUSES = "l2673319,l2917359"
DER_Q_FRAC_P = 0.1

DAILY_PROFILE = ""
DAILY_STRESS = 0.0
SCENARIO_SCALE = 1.0
REF_SAMPLE_INDEX = 0
# False = cleaner timing logs; set True to debug meta-aux mismatches
META_DEBUG = False
V_YLIM_FIXED = False
YMIN, YMAX = 0.92, 1.08

# Timing default: OFF (plot-all of ~3k nodes takes hours of PNGs).
# Set True only when you need the full voltage gallery.
PLOT_ALL_CACHE_NODES = False
PLOT_ALL_MAX_NODES = 0  # only used when PLOT_ALL_CACHE_NODES=True; 0 = every node
PLOT_NODES = ["l2673319.1", "l2917359.1", "190-8593.1"]  # few nodes when plot-all off
VOLTAGE_PLOT_DPI = 72
VOLTAGE_FIG_W = 0.0
VOLTAGE_FIG_H = 0.0

os.environ.setdefault("GNN_TORCH_COMPILE", "0")
os.environ.setdefault("GNN_CUDA_GRAPHS", "1")
os.environ.setdefault("GNN_DEFER_D2H", "1")
if META_DEBUG:
    os.environ["GNN_DAILY_META_DEBUG"] = "1"
else:
    os.environ.pop("GNN_DAILY_META_DEBUG", None)


def _profile_must_exist(spec: Path, fname: str) -> None:
    if spec.is_dir():
        p = spec / fname
        if not p.is_file():
            raise FileNotFoundError(p)
    elif not spec.is_file():
        raise FileNotFoundError(spec)


_profile_must_exist(LOAD_PROFILE_DIR_OR_FILE, LOAD_PROFILE_FILENAME)
_profile_must_exist(PV_IRRADIANCE_DIR_OR_FILE, PV_IRRADIANCE_FILENAME)
if DER_MAX_KW > 0.0 and str(DER_BUSES).strip():
    _profile_must_exist(DER_PROFILE_DIR_OR_FILE, DER_PROFILE_FILENAME)

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint missing: {CHECKPOINT}")
if not CACHE_PT.is_file():
    raise FileNotFoundError(f"Cache missing: {CACHE_PT}")

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

script = REPO / "run_da_gps_daily_opendss_compare.py"
cmd = [
    sys.executable,
    "-u",
    str(script),
    "--run-dir",
    str(RUN_DIR),
    "--cache-pt",
    str(CACHE_PT),
    "--checkpoint",
    str(CHECKPOINT),
    *(["--edge-csv", str(EDGE_CSV.resolve())] if EDGE_CSV and str(EDGE_CSV).strip() else []),
    *(["--daily-profile", str(DAILY_PROFILE)] if str(DAILY_PROFILE).strip() else []),
    "--load-profile-path",
    str(LOAD_PROFILE_DIR_OR_FILE),
    "--load-profile-filename",
    LOAD_PROFILE_FILENAME,
    "--pv-irradiance-profile-path",
    str(PV_IRRADIANCE_DIR_OR_FILE),
    "--pv-irradiance-filename",
    PV_IRRADIANCE_FILENAME,
    "--daily-stress",
    str(DAILY_STRESS),
    "--scenario-scale",
    str(SCENARIO_SCALE),
    "--ref-sample-index",
    str(REF_SAMPLE_INDEX),
    "--device",
    DEVICE,
    "--out-dir",
    str(OUT_DIR),
]
if DER_MAX_KW > 0.0 and str(DER_BUSES).strip():
    cmd += [
        "--der-profile-path",
        str(DER_PROFILE_DIR_OR_FILE),
        "--der-profile-filename",
        DER_PROFILE_FILENAME,
        "--der-max-kw",
        str(DER_MAX_KW),
        "--der-buses",
        DER_BUSES.replace(" ", ""),
        "--der-q-frac-p",
        str(DER_Q_FRAC_P),
    ]
if META_DEBUG:
    cmd.append("--meta-debug")
if V_YLIM_FIXED:
    cmd += ["--v-ylim-fixed", "--ymin", str(YMIN), "--ymax", str(YMAX)]

if PLOT_ALL_CACHE_NODES:
    cmd.append("--plot-all-cache-nodes")
    if int(PLOT_ALL_MAX_NODES) > 0:
        cmd += ["--plot-all-max-nodes", str(int(PLOT_ALL_MAX_NODES))]
    if int(VOLTAGE_PLOT_DPI) > 0:
        cmd += ["--voltage-plot-dpi", str(int(VOLTAGE_PLOT_DPI))]
    if float(VOLTAGE_FIG_W) > 0:
        cmd += ["--voltage-plot-fig-w", str(float(VOLTAGE_FIG_W))]
    if float(VOLTAGE_FIG_H) > 0:
        cmd += ["--voltage-plot-fig-h", str(float(VOLTAGE_FIG_H))]
else:
    cmd += [x for n in PLOT_NODES for x in ("--plot-node", n)]

print("=" * 72)
print("Method A snapshot timing (interleaved OpenDSS + GNN) — GPU GNN")
print("  OpenDSS stays on CPU; only GNN uses CUDA.")
print(f"  env={'Colab' if _on_colab else 'local'}  device={DEVICE}")
print(f"  run_dir={RUN_DIR}")
print(f"  checkpoint={CHECKPOINT}")
print(f"  cache_pt={CACHE_PT}")
print(f"  load={LOAD_PROFILE_DIR_OR_FILE}")
print(f"  irr={PV_IRRADIANCE_DIR_OR_FILE}")
print(f"  plot_all={PLOT_ALL_CACHE_NODES}  meta_debug={META_DEBUG}  DER_MAX_KW={DER_MAX_KW}")
print(f"  out_dir={OUT_DIR}")
print("  Look for: Wall-clock breakdown / Deploy speedup in stdout below.")
print("=" * 72)
print("Running:\n", " ".join(cmd), "\n", flush=True)

# Stream stdout so Colab shows progress (capture_output would buffer for the whole day).
with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\n[method_a_gpu] done. Outputs under:", OUT_DIR)
print("[method_a_gpu] Re-check stdout for 'Wall-clock breakdown' and 'Deploy speedup'.")
